# Intraday Implied Volatility Surface Forecasting

This notebook studies **15-minute SPY implied total-variance surface dynamics**.

The key modeling idea is to separate deterministic maturity roll-down from the
surface innovation that must be forecast:

\[
Y_t = W_{t+15} - \mathcal{R}_{15}(W_t).
\]

A persistence-anchored CNN predicts the conditional mean innovation.  The
selected deterministic model is

\[
\hat W_{t+15}
=
\mathcal{R}_{15}(W_t)
+
0.765\,f_\theta(H_t),
\]

where \(H_t\) contains the recent masked volatility surfaces and market-state
features.

### Current out-of-sample result

Using the validation-selected constant shrinkage coefficient, the model reduces
test MSE by approximately **7.59%** relative to roll-down persistence, with a
day-level win rate of approximately **60.4%**.

### Data note

Raw option quote data are **not included** in this repository.  Set the
environment variable `SPY_OPTION_DATA_ROOT` to your local raw-data directory.
All paths in this notebook are repository-relative.


## 1. Configuration and raw-data validation

The first blocks define the fixed \((k,\tau)\) grid, quote-quality filters,
train/validation/test splits, and inspect a real option contract before any
surface construction.


In [ ]:
# ============================================================
# BLOCK 1
# Imports + Global Configuration
# ============================================================

from __future__ import annotations

import copy
import gc
import json
import math
import os
import random
import re

from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from IPython.display import display

# 使用普通 tqdm，不使用 Jupyter widget，
# 避免 "Error displaying widget: model not found"
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 1. Raw SPY data location
# ============================================================

# Set SPY_OPTION_DATA_ROOT in your environment to the folder that
# contains one subdirectory per trading date.  If the variable is
# not set, the notebook falls back to ./data/SPY.
PROJECT_ROOT = Path.cwd()

RAW_ROOT = Path(
    os.environ.get(
        "SPY_OPTION_DATA_ROOT",
        str(PROJECT_ROOT / "data" / "SPY"),
    )
).expanduser()

DEBUG_DATE = "2023-01-05"


# ============================================================
# 2. Output locations
# ============================================================

SURFACE_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "SPY_masked_surfaces"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "results"
)

SURFACE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 3. Date range
# ============================================================

BUILD_START_DATE = pd.Timestamp(
    "2023-01-01"
)

BUILD_END_DATE = pd.Timestamp(
    "2026-05-30"
)


# Train / Validation / Test
TRAIN_END = pd.Timestamp(
    "2024-12-31"
)

VALID_END = pd.Timestamp(
    "2025-06-30"
)

TEST_END = pd.Timestamp(
    "2026-05-30"
)


# Known dates without usable full-session raw data.
# These dates are excluded non-destructively rather than deleting files.
EXCLUDED_DATES = {
    "2024-05-01",
    "2024-05-02",
    "2024-05-03",
    "2024-05-06",
}


# ============================================================
# 4. Intraday settings
# ============================================================

INTERVAL_MINUTES = 15

# 原始 CSV 包含 09:30 ~ 16:15
RAW_SESSION_START = "09:30:00"
RAW_SESSION_END = "16:15:00"

# 第一版预测模型先避开：
# 09:30 stale quotes
# 以及收盘后的数据
MODEL_SESSION_START = "09:45:00"
MODEL_SESSION_END = "15:45:00"

# 过去4张：
# t-45, t-30, t-15, t
HISTORY_LENGTH = 4

# realized volatility 使用最近4个15分钟 return
RV_WINDOW_RETURNS = 4


# ============================================================
# 5. Fixed target surface grid
# ============================================================

# 41个 log-moneyness coordinates
K_GRID = np.linspace(
    -0.25,
    0.25,
    41,
    dtype=np.float64,
)

# 10个 target maturities
DTE_GRID = np.array(
    [
        7,
        14,
        21,
        30,
        45,
        60,
        90,
        120,
        180,
        270,
    ],
    dtype=np.float64,
)

TAU_GRID = (
    DTE_GRID / 365.0
)

N_K = len(
    K_GRID
)

N_TAU = len(
    DTE_GRID
)


# Surface value columns
GRID_COLUMNS = [
    f"w_k{k:02d}_t{t:02d}"
    for k in range(N_K)
    for t in range(N_TAU)
]


# Mask columns
MASK_COLUMNS = [
    f"m_k{k:02d}_t{t:02d}"
    for k in range(N_K)
    for t in range(N_TAU)
]


# ============================================================
# 6. Quote cleaning
# ============================================================

MIN_IV = 0.01
MAX_IV = 5.00

MIN_DTE = 2.0
MAX_DTE = 365.0

# 原始 contracts 最大使用范围
RAW_K_WINDOW = (
    -0.35,
    0.35,
)

MAX_REL_SPREAD = 1.0

# stale quote filter
APPLY_QUOTE_AGE_FILTER = True

MAX_QUOTE_AGE_SECONDS = (
    5 * 60
)

# 第一版使用 OTM options
USE_OTM_ONLY = True

MIN_POINTS_PER_SNAPSHOT = 60

MIN_EXPIRIES_PER_SNAPSHOT = 4


# ============================================================
# 7. Small maturity boundary extrapolation
# ============================================================

# 只允许非常小的 maturity 边界 extrapolation。
#
# 例如：
# target 7D vs actual 7.125D
# target 270D vs actual 267.125D
#
# 相对 gap <= 2% 才允许。
MAX_BOUNDARY_DTE_RELATIVE_GAP = 0.02


# ============================================================
# 8. Neural-network training settings
# ============================================================

SEED = 1234

BATCH_SIZE = 128

MAX_EPOCHS = 60

PATIENCE = 12

LEARNING_RATE = 1e-3

WEIGHT_DECAY = 1e-5

GRADIENT_CLIP = 1.0

NUM_WORKERS = 0

BOOTSTRAP_REPETITIONS = 2000


# ============================================================
# 9. Random seed
# ============================================================

def seed_everything(
    seed: int = SEED,
):

    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


seed_everything()


# ============================================================
# 10. Device
# ============================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# 11. Basic diagnostics
# ============================================================

print(
    "DEVICE:",
    DEVICE
)

print(
    "RAW_ROOT:",
    RAW_ROOT
)

print(
    "RAW_ROOT exists:",
    RAW_ROOT.exists()
)

print(
    "DEBUG directory exists:",
    (
        RAW_ROOT
        / DEBUG_DATE
    ).exists()
)

print(
    "Surface grid:",
    N_K,
    "x",
    N_TAU,
    "=",
    N_K * N_TAU
)

print(
    "Surface columns:",
    len(
        GRID_COLUMNS
    )
)

print(
    "Mask columns:",
    len(
        MASK_COLUMNS
    )
)

print(
    "SURFACE_ROOT:",
    SURFACE_ROOT
)

print(
    "OUTPUT_ROOT:",
    OUTPUT_ROOT
)

In [ ]:
# ============================================================
# BLOCK 2
# Inspect one real SPY contract CSV
# ============================================================

DEBUG_DIR = (
    RAW_ROOT
    / DEBUG_DATE
)

csv_files = sorted(
    DEBUG_DIR.glob("*.csv")
)


print(
    "DEBUG_DIR:",
    DEBUG_DIR
)

print(
    "CSV file count:",
    len(csv_files)
)


if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No CSV files found in:\n"
        f"{DEBUG_DIR}"
    )


# ------------------------------------------------------------
# Prefer the contract you previously showed:
#
# SPY___230112C00357000.csv
# ------------------------------------------------------------

preferred_name = (
    "SPY___230112C00357000.csv"
)

preferred_path = (
    DEBUG_DIR
    / preferred_name
)


if preferred_path.exists():

    sample_path = (
        preferred_path
    )

else:

    sample_path = (
        csv_files[0]
    )


print(
    "\nSample file:"
)

print(
    sample_path.name
)


# ============================================================
# 1. Read raw CSV
# ============================================================

sample_raw = pd.read_csv(
    sample_path
)


print(
    "\nShape:"
)

print(
    sample_raw.shape
)


print(
    "\nColumns:"
)

print(
    sample_raw.columns.tolist()
)


# ============================================================
# 2. Expected schema
# ============================================================

EXPECTED_COLUMNS = [
    "timestamp",
    "stockSymbol",
    "optionExpirationDate",
    "optionStrike",
    "optionType",
    "optionStyle",
    "optionSymbol",
    "optionBidPrice",
    "optionAskPrice",
    "optionBidDateTime",
    "optionAskDateTime",
    "optionVolume",
    "optionIv",
    "underlyingPrice",
    "optionDelta",
    "optionGamma",
    "optionTheta",
    "optionVega",
    "calcTimestamp",
]


missing_columns = [
    column
    for column
    in EXPECTED_COLUMNS
    if column
    not in sample_raw.columns
]


print(
    "\nMissing expected columns:"
)

print(
    missing_columns
)


# ============================================================
# 3. Parse datetime columns
# ============================================================

datetime_columns = [
    "timestamp",
    "optionExpirationDate",
    "optionBidDateTime",
    "optionAskDateTime",
    "calcTimestamp",
]


sample = (
    sample_raw.copy()
)


for column in datetime_columns:

    if column in sample.columns:

        sample[column] = (
            pd.to_datetime(
                sample[column],
                errors="coerce",
                format="mixed",
            )
        )


# ============================================================
# 4. Numeric columns
# ============================================================

numeric_columns = [
    "optionStrike",
    "optionBidPrice",
    "optionAskPrice",
    "optionVolume",
    "optionIv",
    "underlyingPrice",
    "optionDelta",
    "optionGamma",
    "optionTheta",
    "optionVega",
]


for column in numeric_columns:

    if column in sample.columns:

        sample[column] = (
            pd.to_numeric(
                sample[column],
                errors="coerce",
            )
        )


# ============================================================
# 5. Basic contract information
# ============================================================

print(
    "\nContract information:"
)


for column in [
    "stockSymbol",
    "optionSymbol",
    "optionExpirationDate",
    "optionStrike",
    "optionType",
    "optionStyle",
]:

    if column in sample.columns:

        values = (
            sample[column]
            .dropna()
            .unique()
        )

        print(
            column,
            "->",
            values[:10]
        )


# ============================================================
# 6. Timestamp diagnostics
# ============================================================

print(
    "\nTimestamp diagnostics:"
)


print(
    "row count:",
    len(sample)
)


print(
    "timestamp min:",
    sample[
        "timestamp"
    ].min()
)


print(
    "timestamp max:",
    sample[
        "timestamp"
    ].max()
)


timestamp_gaps = (
    sample[
        "timestamp"
    ]
    .sort_values()
    .diff()
    .dropna()
)


print(
    "\nTimestamp gap counts:"
)

display(
    timestamp_gaps
    .value_counts()
)


# ============================================================
# 7. IV diagnostics
# ============================================================

print(
    "\noptionIv diagnostics:"
)


display(
    sample[
        "optionIv"
    ].describe()
)


print(
    "\nFirst IV values:"
)

print(
    sample[
        "optionIv"
    ]
    .head(
        10
    )
    .tolist()
)


# ============================================================
# 8. Underlying diagnostics
# ============================================================

print(
    "\nUnderlying price diagnostics:"
)


display(
    sample[
        "underlyingPrice"
    ].describe()
)


# ============================================================
# 9. Quote age
# ============================================================

if (
    "optionBidDateTime"
    in sample.columns
    and
    "optionAskDateTime"
    in sample.columns
    and
    "calcTimestamp"
    in sample.columns
):

    latest_quote_time = (
        sample[
            [
                "optionBidDateTime",
                "optionAskDateTime",
            ]
        ]
        .max(
            axis=1
        )
    )


    sample[
        "quote_age_seconds"
    ] = (
        sample[
            "calcTimestamp"
        ]
        - latest_quote_time
    ).dt.total_seconds()


    print(
        "\nQuote age diagnostics:"
    )


    display(
        sample[
            [
                "timestamp",
                "optionBidDateTime",
                "optionAskDateTime",
                "calcTimestamp",
                "quote_age_seconds",
            ]
        ]
        .head(
            10
        )
    )


    print(
        "\nQuote age summary:"
    )


    display(
        sample[
            "quote_age_seconds"
        ].describe()
    )


# ============================================================
# 10. Show first rows
# ============================================================

important_columns = [
    "timestamp",
    "optionExpirationDate",
    "optionStrike",
    "optionType",
    "optionStyle",
    "optionSymbol",
    "optionBidPrice",
    "optionAskPrice",
    "optionIv",
    "underlyingPrice",
    "optionDelta",
    "optionGamma",
    "optionTheta",
    "optionVega",
    "calcTimestamp",
]


important_columns = [
    column
    for column
    in important_columns
    if column
    in sample.columns
]


print(
    "\nFirst rows:"
)


display(
    sample[
        important_columns
    ].head(
        10
    )
)


print(
    "\nBLOCK 2 finished successfully."
)

In [ ]:
# ============================================================
# BLOCK 3
# Read one contract CSV into standardized columns
# ============================================================

USE_COLUMNS = [
    "timestamp",
    "stockSymbol",
    "optionExpirationDate",
    "optionStrike",
    "optionType",
    "optionStyle",
    "optionSymbol",
    "optionBidPrice",
    "optionAskPrice",
    "optionBidDateTime",
    "optionAskDateTime",
    "optionVolume",
    "optionIv",
    "underlyingPrice",
    "optionDelta",
    "optionGamma",
    "optionTheta",
    "optionVega",
    "calcTimestamp",
]


def read_contract_csv(
    path: Path,
    trade_date: str,
) -> pd.DataFrame:
    """
    读取单个 SPY option contract 的全天 15 分钟 CSV。

    输出会增加：
        latest_quote_time
        quote_age_seconds
        mid
        spread
        relative_spread
        tau
        dte
        log_moneyness
        total_variance
        source_file
    """

    # --------------------------------------------------------
    # 1. 读取 header
    # --------------------------------------------------------

    try:

        header = (
            pd.read_csv(
                path,
                nrows=0,
            )
            .columns
            .tolist()
        )

    except Exception:

        return pd.DataFrame()


    required_columns = {
        "timestamp",
        "stockSymbol",
        "optionExpirationDate",
        "optionStrike",
        "optionType",
        "optionStyle",
        "optionSymbol",
        "optionBidPrice",
        "optionAskPrice",
        "optionIv",
        "underlyingPrice",
        "calcTimestamp",
    }


    if not required_columns.issubset(
        header
    ):

        return pd.DataFrame()


    available_columns = [
        column
        for column in USE_COLUMNS
        if column in header
    ]


    # --------------------------------------------------------
    # 2. 读取文件
    # --------------------------------------------------------

    try:

        frame = pd.read_csv(
            path,
            usecols=available_columns,
        )

    except Exception:

        return pd.DataFrame()


    if frame.empty:

        return pd.DataFrame()


    # --------------------------------------------------------
    # 3. Parse datetime columns
    # --------------------------------------------------------

    datetime_columns = [
        "timestamp",
        "optionExpirationDate",
        "optionBidDateTime",
        "optionAskDateTime",
        "calcTimestamp",
    ]


    for column in datetime_columns:

        if column in frame.columns:

            frame[column] = (
                pd.to_datetime(
                    frame[column],
                    errors="coerce",
                    format="mixed",
                )
            )


    # --------------------------------------------------------
    # 4. Numeric columns
    # --------------------------------------------------------

    numeric_columns = [
        "optionStrike",
        "optionBidPrice",
        "optionAskPrice",
        "optionVolume",
        "optionIv",
        "underlyingPrice",
        "optionDelta",
        "optionGamma",
        "optionTheta",
        "optionVega",
    ]


    for column in numeric_columns:

        if column in frame.columns:

            frame[column] = (
                pd.to_numeric(
                    frame[column],
                    errors="coerce",
                )
            )


    # --------------------------------------------------------
    # 5. Standardize text columns
    # --------------------------------------------------------

    frame["stockSymbol"] = (
        frame["stockSymbol"]
        .astype(str)
        .str.upper()
        .str.strip()
    )


    frame["optionType"] = (
        frame["optionType"]
        .astype(str)
        .str.upper()
        .str.strip()
        .str[0]
    )


    frame["optionStyle"] = (
        frame["optionStyle"]
        .astype(str)
        .str.upper()
        .str.strip()
        .str[0]
    )


    # --------------------------------------------------------
    # 6. 只保留当前 trade_date
    # --------------------------------------------------------

    expected_date = (
        pd.Timestamp(
            trade_date
        )
        .normalize()
    )


    frame = frame[
        frame[
            "timestamp"
        ]
        .dt.normalize()
        .eq(
            expected_date
        )
    ].copy()


    if frame.empty:

        return frame


    # --------------------------------------------------------
    # 7. 最新 bid/ask quote 时间
    # --------------------------------------------------------

    if (
        "optionBidDateTime"
        in frame.columns
    ):

        bid_time = (
            frame[
                "optionBidDateTime"
            ]
        )

    else:

        bid_time = pd.Series(
            pd.NaT,
            index=frame.index,
        )


    if (
        "optionAskDateTime"
        in frame.columns
    ):

        ask_time = (
            frame[
                "optionAskDateTime"
            ]
        )

    else:

        ask_time = pd.Series(
            pd.NaT,
            index=frame.index,
        )


    frame[
        "latest_quote_time"
    ] = (
        pd.concat(
            [
                bid_time,
                ask_time,
            ],
            axis=1,
        )
        .max(
            axis=1
        )
    )


    # --------------------------------------------------------
    # 8. Quote age
    # --------------------------------------------------------

    frame[
        "quote_age_seconds"
    ] = (
        frame[
            "calcTimestamp"
        ]
        - frame[
            "latest_quote_time"
        ]
    ).dt.total_seconds()


    # 极少数时间戳异常时避免负数
    frame.loc[
        frame[
            "quote_age_seconds"
        ]
        < 0,
        "quote_age_seconds",
    ] = np.nan


    # --------------------------------------------------------
    # 9. Bid/ask statistics
    # --------------------------------------------------------

    frame["mid"] = (
        0.5
        * (
            frame[
                "optionBidPrice"
            ]
            + frame[
                "optionAskPrice"
            ]
        )
    )


    frame["spread"] = (
        frame[
            "optionAskPrice"
        ]
        - frame[
            "optionBidPrice"
        ]
    )


    frame[
        "relative_spread"
    ] = (
        frame[
            "spread"
        ]
        / frame[
            "mid"
        ].clip(
            lower=0.01
        )
    )


    # --------------------------------------------------------
    # 10. 精确 maturity
    #
    # 第一版约定 expiration 当天 16:00。
    # --------------------------------------------------------

    expiration_timestamp = (
        frame[
            "optionExpirationDate"
        ]
        .dt.normalize()
        + pd.Timedelta(
            hours=16
        )
    )


    frame["tau"] = (
        expiration_timestamp
        - frame[
            "timestamp"
        ]
    ).dt.total_seconds() / (
        365.0
        * 24.0
        * 60.0
        * 60.0
    )


    frame["dte"] = (
        frame[
            "tau"
        ]
        * 365.0
    )


    # --------------------------------------------------------
    # 11. Spot log-moneyness
    #
    # k = log(K / S)
    #
    # 后续如果需要更严格，可换成 forward moneyness。
    # --------------------------------------------------------

    frame[
        "log_moneyness"
    ] = np.log(
        frame[
            "optionStrike"
        ]
        / frame[
            "underlyingPrice"
        ]
    )


    # --------------------------------------------------------
    # 12. Total variance
    #
    # w = tau * IV^2
    # --------------------------------------------------------

    frame[
        "total_variance"
    ] = (
        frame[
            "tau"
        ]
        * frame[
            "optionIv"
        ].pow(2)
    )


    # --------------------------------------------------------
    # 13. Source
    # --------------------------------------------------------

    frame[
        "source_file"
    ] = path.name


    # --------------------------------------------------------
    # 14. Sort
    # --------------------------------------------------------

    frame = (
        frame
        .sort_values(
            "timestamp"
        )
        .reset_index(
            drop=True
        )
    )


    return frame


print(
    "read_contract_csv defined:",
    "read_contract_csv"
    in globals()
)

In [ ]:
test_contract = (
    read_contract_csv(
        sample_path,
        DEBUG_DATE,
    )
)


print(
    "\nTest contract shape:",
    test_contract.shape
)


display(
    test_contract[
        [
            "timestamp",
            "optionSymbol",
            "optionExpirationDate",
            "optionStrike",
            "optionType",
            "optionIv",
            "underlyingPrice",
            "quote_age_seconds",
            "dte",
            "log_moneyness",
            "total_variance",
        ]
    ].head(
        10
    )
)

In [ ]:
# ============================================================
# BLOCK 4
# Load all contract CSVs for one trading day
# ============================================================


def discover_contract_files(
    trade_date: str,
    raw_root: Path = RAW_ROOT,
) -> list[Path]:
    """
    找到某一天目录下所有 option contract CSV。
    """

    day_directory = (
        raw_root
        / trade_date
    )


    if not day_directory.exists():

        raise FileNotFoundError(
            f"Date directory does not exist:\n"
            f"{day_directory}"
        )


    paths = sorted(
        day_directory.glob(
            "*.csv"
        )
    )


    # 排除可能存在的非 contract 文件
    ignored_names = {
        "failed_contracts.csv",
        "errors.csv",
        "surface_build_log.csv",
    }


    paths = [
        path
        for path in paths
        if path.name.lower()
        not in ignored_names
    ]


    return paths


def load_day_raw_quotes(
    trade_date: str,
    raw_root: Path = RAW_ROOT,
    limit_files: Optional[int] = None,
) -> pd.DataFrame:
    """
    读取一天所有 contract CSV，
    调用 Block 3 的 read_contract_csv()，
    合并成一个 DataFrame。

    此处暂时不做：
        - stale quote filter
        - OTM filter
        - IV filter
        - spread filter
        - surface interpolation
    """

    paths = (
        discover_contract_files(
            trade_date,
            raw_root=raw_root,
        )
    )


    if (
        limit_files
        is not None
    ):

        paths = (
            paths[
                :limit_files
            ]
        )


    if len(paths) == 0:

        raise FileNotFoundError(
            f"No contract CSV files "
            f"for {trade_date}"
        )


    print(
        f"{trade_date}: "
        f"{len(paths):,} contract files"
    )


    frames = []

    failures = []


    for file_index, path in enumerate(
        paths,
        start=1,
    ):

        try:

            contract = (
                read_contract_csv(
                    path,
                    trade_date,
                )
            )


            if not contract.empty:

                frames.append(
                    contract
                )


        except Exception as error:

            failures.append(
                {
                    "file": (
                        path.name
                    ),

                    "error": (
                        repr(
                            error
                        )
                    ),
                }
            )


        # 普通文字进度，
        # 不用 tqdm widget
        if (
            file_index == 1
            or
            file_index % 200 == 0
            or
            file_index == len(paths)
        ):

            print(
                f"  read "
                f"{file_index:,}"
                f"/"
                f"{len(paths):,}"
            )


    if len(frames) == 0:

        raise RuntimeError(
            f"No readable contract data "
            f"for {trade_date}"
        )


    day = pd.concat(
        frames,
        ignore_index=True,
    )


    # --------------------------------------------------------
    # timestamp 统一到分钟
    # --------------------------------------------------------

    day[
        "timestamp"
    ] = (
        day[
            "timestamp"
        ]
        .dt.floor(
            "min"
        )
    )


    # --------------------------------------------------------
    # 只保留预期日内时间范围
    # --------------------------------------------------------

    start_time = (
        pd.to_datetime(
            RAW_SESSION_START
        )
        .time()
    )

    end_time = (
        pd.to_datetime(
            RAW_SESSION_END
        )
        .time()
    )


    clock_time = (
        day[
            "timestamp"
        ]
        .dt.time
    )


    day = day[
        (
            clock_time
            >= start_time
        )
        &
        (
            clock_time
            <= end_time
        )
    ].copy()


    # --------------------------------------------------------
    # 必须是 15 分钟 grid
    # --------------------------------------------------------

    day = day[
        (
            day[
                "timestamp"
            ]
            .dt.minute
            % INTERVAL_MINUTES
        )
        == 0
    ].copy()


    # --------------------------------------------------------
    # 同一个 contract / timestamp 如果意外重复，
    # 保留：
    #
    # 1. quote age 更小
    # 2. spread 更小
    # --------------------------------------------------------

    day = (
        day
        .sort_values(
            [
                "timestamp",
                "optionSymbol",
                "quote_age_seconds",
                "relative_spread",
            ],
            na_position="last",
        )
        .drop_duplicates(
            [
                "timestamp",
                "optionSymbol",
            ],
            keep="first",
        )
        .sort_values(
            [
                "timestamp",
                "optionExpirationDate",
                "optionStrike",
                "optionType",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    print(
        "\nLoad summary:"
    )

    print(
        "rows:",
        f"{len(day):,}"
    )

    print(
        "timestamps:",
        day[
            "timestamp"
        ].nunique()
    )

    print(
        "contracts:",
        day[
            "optionSymbol"
        ].nunique()
    )

    print(
        "expiries:",
        day[
            "optionExpirationDate"
        ].nunique()
    )

    print(
        "failed files:",
        len(
            failures
        )
    )


    if failures:

        failure_frame = (
            pd.DataFrame(
                failures
            )
        )

        display(
            failure_frame.head(
                20
            )
        )


    return day


print(
    "load_day_raw_quotes defined:",
    "load_day_raw_quotes"
    in globals()
)

In [ ]:
# ============================================================
# Test Block 4 on 2023-01-05
# ============================================================

debug_raw_quotes = (
    load_day_raw_quotes(
        DEBUG_DATE
    )
)


print(
    "\nRows per timestamp:"
)

rows_per_timestamp = (
    debug_raw_quotes
    .groupby(
        "timestamp",
        observed=True,
    )
    .size()
)


display(
    rows_per_timestamp
)


print(
    "\nRows-per-timestamp summary:"
)

display(
    rows_per_timestamp.describe()
)


print(
    "\nExpiration count per timestamp:"
)

expiry_count_per_timestamp = (
    debug_raw_quotes
    .groupby(
        "timestamp",
        observed=True,
    )[
        "optionExpirationDate"
    ]
    .nunique()
)


display(
    expiry_count_per_timestamp
)


print(
    "\nAvailable clock times:"
)

print(
    debug_raw_quotes[
        "timestamp"
    ]
    .dt.strftime(
        "%H:%M:%S"
    )
    .drop_duplicates()
    .tolist()
)

In [ ]:
# ============================================================
# BLOCK 5
# Clean option quotes
#
# raw quotes
#     ↓
# remove stale / invalid quotes
#     ↓
# select OTM options
#     ↓
# one observation per timestamp / expiry / strike
# ============================================================


def clean_day_quotes(
    raw_quotes: pd.DataFrame,
) -> tuple[
    pd.DataFrame,
    pd.DataFrame,
]:
    """
    输入：
        Block 4 产生的一整天 raw option quotes

    输出：
        clean_quotes
        audit_table
    """

    quotes = (
        raw_quotes
        .copy()
    )


    # ========================================================
    # 1. Basic validity filters
    # ========================================================

    valid = (
        # SPY only
        quotes[
            "stockSymbol"
        ]
        .eq("SPY")

        # American-style option
        & quotes[
            "optionStyle"
        ]
        .eq("A")

        # C / P
        & quotes[
            "optionType"
        ]
        .isin(
            ["C", "P"]
        )

        # IV validity
        & quotes[
            "optionIv"
        ]
        .between(
            MIN_IV,
            MAX_IV,
        )

        # Positive underlying
        & (
            quotes[
                "underlyingPrice"
            ]
            > 0
        )

        # Positive strike
        & (
            quotes[
                "optionStrike"
            ]
            > 0
        )

        # DTE range
        & quotes[
            "dte"
        ]
        .between(
            MIN_DTE,
            MAX_DTE,
        )

        # Raw moneyness range
        & quotes[
            "log_moneyness"
        ]
        .between(
            *RAW_K_WINDOW
        )

        # Positive maturity
        & (
            quotes[
                "tau"
            ]
            > 0
        )

        # finite total variance
        & np.isfinite(
            quotes[
                "total_variance"
            ]
        )

        # valid bid
        & (
            quotes[
                "optionBidPrice"
            ]
            >= 0
        )

        # valid ask
        & (
            quotes[
                "optionAskPrice"
            ]
            >= quotes[
                "optionBidPrice"
            ]
        )

        & (
            quotes[
                "optionAskPrice"
            ]
            > 0
        )

        # relative spread
        & (
            quotes[
                "relative_spread"
            ]
            <= MAX_REL_SPREAD
        )
    )


    # ========================================================
    # 2. Remove stale quotes
    # ========================================================

    if APPLY_QUOTE_AGE_FILTER:

        valid &= (
            quotes[
                "quote_age_seconds"
            ]
            .between(
                0,
                MAX_QUOTE_AGE_SECONDS,
            )
        )


    # ========================================================
    # 3. Define OTM options
    #
    # Call:
    #     K >= S
    #
    # Put:
    #     K < S
    #
    # ATM 附近用 call。
    # ========================================================

    quotes[
        "is_otm"
    ] = (
        (
            quotes[
                "optionType"
            ]
            .eq("C")

            & (
                quotes[
                    "optionStrike"
                ]
                >= quotes[
                    "underlyingPrice"
                ]
            )
        )

        |

        (
            quotes[
                "optionType"
            ]
            .eq("P")

            & (
                quotes[
                    "optionStrike"
                ]
                < quotes[
                    "underlyingPrice"
                ]
            )
        )
    )


    if USE_OTM_ONLY:

        valid &= (
            quotes[
                "is_otm"
            ]
        )


    # ========================================================
    # 4. Apply filters
    # ========================================================

    clean = (
        quotes.loc[
            valid
        ]
        .copy()
    )


    # ========================================================
    # 5. If same timestamp / expiry / strike appears twice,
    # keep the better quote.
    #
    # priority:
    #   1. fresher quote
    #   2. narrower relative spread
    # ========================================================

    clean = (
        clean
        .sort_values(
            [
                "timestamp",
                "optionExpirationDate",
                "optionStrike",
                "quote_age_seconds",
                "relative_spread",
            ],
            na_position="last",
        )
        .drop_duplicates(
            [
                "timestamp",
                "optionExpirationDate",
                "optionStrike",
            ],
            keep="first",
        )
        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # 6. Interpolation weights
    #
    # tighter spread -> larger weight
    # ========================================================

    clean[
        "weight"
    ] = (
        1.0
        / clean[
            "relative_spread"
        ]
        .clip(
            lower=0.02
        )
        .pow(2)
    )


    # Vega only provides a mild adjustment.
    if (
        "optionVega"
        in clean.columns
    ):

        positive_vega = (
            clean.loc[
                clean[
                    "optionVega"
                ]
                > 0,
                "optionVega",
            ]
        )


        if (
            not positive_vega.empty
        ):

            vega_scale = float(
                positive_vega.median()
            )

        else:

            vega_scale = 1.0


        vega_multiplier = (
            np.sqrt(
                (
                    clean[
                        "optionVega"
                    ]
                    .abs()
                    / max(
                        vega_scale,
                        1e-8,
                    )
                )
                .clip(
                    0.25,
                    4.0,
                )
            )
            .fillna(
                1.0
            )
        )


        clean[
            "weight"
        ] *= (
            vega_multiplier
        )


    clean[
        "weight"
    ] = (
        clean[
            "weight"
        ]
        .clip(
            1.0,
            10_000.0,
        )
    )


    # ========================================================
    # 7. Build audit table
    # ========================================================

    raw_summary = (
        raw_quotes
        .groupby(
            "timestamp",
            observed=True,
        )
        .agg(
            raw_rows=(
                "optionSymbol",
                "size",
            ),

            raw_contracts=(
                "optionSymbol",
                "nunique",
            ),

            stale_fraction=(
                "quote_age_seconds",

                lambda x: float(
                    (
                        x
                        > MAX_QUOTE_AGE_SECONDS
                    ).mean()
                ),
            ),

            median_quote_age=(
                "quote_age_seconds",
                "median",
            ),
        )
    )


    clean_summary = (
        clean
        .groupby(
            "timestamp",
            observed=True,
        )
        .agg(
            clean_rows=(
                "optionSymbol",
                "size",
            ),

            clean_expiries=(
                "optionExpirationDate",
                "nunique",
            ),

            k_min=(
                "log_moneyness",
                "min",
            ),

            k_max=(
                "log_moneyness",
                "max",
            ),

            dte_min=(
                "dte",
                "min",
            ),

            dte_max=(
                "dte",
                "max",
            ),

            median_relative_spread=(
                "relative_spread",
                "median",
            ),

            median_quote_age=(
                "quote_age_seconds",
                "median",
            ),
        )
    )


    audit = (
        raw_summary
        .join(
            clean_summary,
            how="outer",
            lsuffix="_raw",
            rsuffix="_clean",
        )
        .reset_index()
    )


    audit[
        "clock_time"
    ] = (
        audit[
            "timestamp"
        ]
        .dt.strftime(
            "%H:%M:%S"
        )
    )


    return (
        clean,
        audit,
    )


print(
    "clean_day_quotes defined:",
    "clean_day_quotes"
    in globals()
)

In [ ]:
# ============================================================
# Test BLOCK 5 on 2023-01-05
# ============================================================

(
    debug_clean_quotes,
    debug_quote_audit,
) = clean_day_quotes(
    debug_raw_quotes
)


print(
    "\nClean summary:"
)

print(
    "rows:",
    len(
        debug_clean_quotes
    )
)

print(
    "timestamps:",
    debug_clean_quotes[
        "timestamp"
    ].nunique()
)

print(
    "expiries:",
    debug_clean_quotes[
        "optionExpirationDate"
    ].nunique()
)


print(
    "\nAudit table:"
)

display(
    debug_quote_audit
)


print(
    "\nClean rows per timestamp:"
)

display(
    debug_clean_quotes
    .groupby(
        "timestamp",
        observed=True,
    )
    .size()
)


print(
    "\nCall / Put counts:"
)

display(
    debug_clean_quotes[
        "optionType"
    ].value_counts()
)


print(
    "\nMoneyness summary:"
)

display(
    debug_clean_quotes[
        "log_moneyness"
    ].describe()
)


print(
    "\nDTE summary:"
)

display(
    debug_clean_quotes[
        "dte"
    ].describe()
)

In [ ]:
# ============================================================
# BLOCK 6
# Diagnose expiry-specific moneyness support
# ============================================================


# ------------------------------------------------------------
# 1. Per timestamp / expiry support
# ------------------------------------------------------------

expiry_support = (
    debug_clean_quotes
    .groupby(
        [
            "timestamp",
            "optionExpirationDate",
        ],
        observed=True,
    )
    .agg(

        dte=(
            "dte",
            "median",
        ),

        k_min=(
            "log_moneyness",
            "min",
        ),

        k_max=(
            "log_moneyness",
            "max",
        ),

        n_strikes=(
            "optionStrike",
            "nunique",
        ),

        n_quotes=(
            "optionSymbol",
            "size",
        ),
    )
    .reset_index()
)


print(
    "expiry_support shape:",
    expiry_support.shape
)


print(
    "\nFirst rows:"
)

display(
    expiry_support.head(
        30
    )
)


# ------------------------------------------------------------
# 2. Pick middle timestamp
# ------------------------------------------------------------

timestamps = sorted(
    debug_clean_quotes[
        "timestamp"
    ].unique()
)


example_timestamp = (
    timestamps[
        len(timestamps) // 2
    ]
)


print(
    "\nExample timestamp:",
    example_timestamp
)


example_expiry_support = (
    expiry_support[
        expiry_support[
            "timestamp"
        ]
        == example_timestamp
    ]
    .sort_values(
        "dte"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nExpiry support at example timestamp:"
)

display(
    example_expiry_support
)


# ------------------------------------------------------------
# 3. Summary across all timestamps
# ------------------------------------------------------------

summary_by_expiry = (
    expiry_support
    .groupby(
        "optionExpirationDate",
        observed=True,
    )
    .agg(

        median_dte=(
            "dte",
            "median",
        ),

        median_k_min=(
            "k_min",
            "median",
        ),

        median_k_max=(
            "k_max",
            "median",
        ),

        q90_k_min=(
            "k_min",
            lambda x: x.quantile(
                0.90
            ),
        ),

        q10_k_max=(
            "k_max",
            lambda x: x.quantile(
                0.10
            ),
        ),

        median_n_strikes=(
            "n_strikes",
            "median",
        ),
    )
    .sort_values(
        "median_dte"
    )
    .reset_index()
)


print(
    "\nSummary by actual expiry:"
)

display(
    summary_by_expiry
)


# ------------------------------------------------------------
# 4. Check target K_GRID support per actual expiry
# ------------------------------------------------------------

coverage_rows = []


for _, row in (
    example_expiry_support
    .iterrows()
):

    k_min = float(
        row[
            "k_min"
        ]
    )

    k_max = float(
        row[
            "k_max"
        ]
    )


    supported_grid_points = int(
        (
            (
                K_GRID
                >= k_min
            )
            &
            (
                K_GRID
                <= k_max
            )
        )
        .sum()
    )


    coverage_rows.append(
        {
            "expiration": (
                row[
                    "optionExpirationDate"
                ]
            ),

            "dte": float(
                row[
                    "dte"
                ]
            ),

            "k_min": (
                k_min
            ),

            "k_max": (
                k_max
            ),

            "supported_K_GRID_points": (
                supported_grid_points
            ),

            "total_K_GRID_points": (
                N_K
            ),

            "support_fraction": (
                supported_grid_points
                / N_K
            ),
        }
    )


grid_support_example = (
    pd.DataFrame(
        coverage_rows
    )
)


print(
    "\nK_GRID support at example timestamp:"
)

display(
    grid_support_example
)


# ------------------------------------------------------------
# 5. Plot each expiry's k range
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 6)
)


for _, row in (
    example_expiry_support
    .iterrows()
):

    plt.plot(
        [
            row[
                "k_min"
            ],
            row[
                "k_max"
            ],
        ],
        [
            row[
                "dte"
            ],
            row[
                "dte"
            ],
        ],
        linewidth=3,
    )


plt.axvline(
    K_GRID.min(),
    linestyle="--",
    label="K_GRID min",
)


plt.axvline(
    K_GRID.max(),
    linestyle="--",
    label="K_GRID max",
)


plt.xlabel(
    "log-moneyness"
)

plt.ylabel(
    "DTE"
)

plt.title(
    f"Actual moneyness support by expiry\n"
    f"{example_timestamp}"
)

plt.legend()

plt.show()


print(
    "\nBLOCK 6 finished successfully."
)

In [ ]:
# ============================================================
# BLOCK 7
# Build one masked 41 x 10 total-variance surface
#
# Input:
#     one timestamp's cleaned option quotes
#
# Output:
#     surface : [41, 10]
#     mask    : [41, 10]
#
# mask = 1:
#     this grid point is supported by real option data
#
# mask = 0:
#     unsupported
# ============================================================


# ------------------------------------------------------------
# 1. Helper:
# deduplicate very close moneyness observations
# ------------------------------------------------------------

def prepare_expiry_smile(
    expiry_slice: pd.DataFrame,
) -> pd.DataFrame:

    data = (
        expiry_slice[
            [
                "log_moneyness",
                "total_variance",
                "weight",
            ]
        ]
        .dropna()
        .copy()
    )


    if data.empty:

        return data


    # Floating point coordinates may differ slightly,
    # so round before grouping.
    data["k_key"] = (
        data[
            "log_moneyness"
        ]
        .round(7)
    )


    data["weighted_w"] = (
        data[
            "total_variance"
        ]
        * data[
            "weight"
        ]
    )


    data = (
        data
        .groupby(
            "k_key",
            as_index=False,
        )
        .agg(
            total_weight=(
                "weight",
                "sum",
            ),

            weighted_w=(
                "weighted_w",
                "sum",
            ),
        )
    )


    data[
        "total_variance"
    ] = (
        data[
            "weighted_w"
        ]
        / data[
            "total_weight"
        ].clip(
            lower=1e-12
        )
    )


    return (
        data[
            [
                "k_key",
                "total_variance",
            ]
        ]
        .sort_values(
            "k_key"
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------
# 2. Helper:
# small linear extrapolation in log(tau)
# ------------------------------------------------------------

def linear_extrapolate(
    x0,
    y0,
    x1,
    y1,
    x_target,
):

    denominator = (
        x1 - x0
    )


    if abs(
        denominator
    ) < 1e-12:

        return np.nan


    slope = (
        y1 - y0
    ) / denominator


    return (
        y0
        + slope
        * (
            x_target
            - x0
        )
    )


# ------------------------------------------------------------
# 3. Main function
# ------------------------------------------------------------

def build_masked_surface(
    snapshot: pd.DataFrame,
):
    """
    Build one volatility surface at one timestamp.

    Returns
    -------
    surface : ndarray, shape [41, 10]

        Total variance values.

        Unsupported positions are stored as 0,
        but must always be interpreted together with mask.

    mask : ndarray, shape [41, 10]

        1 = supported
        0 = unsupported

    diagnostics : dict
    """

    # ========================================================
    # A. Basic checks
    # ========================================================

    if (
        len(snapshot)
        < MIN_POINTS_PER_SNAPSHOT
    ):

        return None


    if (
        snapshot[
            "optionExpirationDate"
        ].nunique()
        < MIN_EXPIRIES_PER_SNAPSHOT
    ):

        return None


    # ========================================================
    # B. First interpolation:
    #
    # for each REAL expiry:
    #
    # irregular strikes
    #       ↓
    # fixed K_GRID
    #
    # ONLY inside actual k_min/k_max.
    # ========================================================

    expiry_rows = []


    for (
        expiration,
        expiry_slice,
    ) in snapshot.groupby(
        "optionExpirationDate",
        observed=True,
    ):

        smile = (
            prepare_expiry_smile(
                expiry_slice
            )
        )


        if len(
            smile
        ) < 4:

            continue


        k_values = (
            smile[
                "k_key"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        w_values = (
            smile[
                "total_variance"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        tau_value = float(
            expiry_slice[
                "tau"
            ].median()
        )


        if (
            not np.isfinite(
                tau_value
            )
            or tau_value <= 0
        ):

            continue


        # ----------------------------------------------------
        # NO k extrapolation
        # ----------------------------------------------------

        k_supported = (
            (
                K_GRID
                >= k_values.min()
            )
            &
            (
                K_GRID
                <= k_values.max()
            )
        )


        if (
            k_supported.sum()
            < 2
        ):

            continue


        smile_on_grid = np.full(
            N_K,
            np.nan,
            dtype=np.float64,
        )


        smile_on_grid[
            k_supported
        ] = np.interp(
            K_GRID[
                k_supported
            ],
            k_values,
            w_values,
        )


        expiry_rows.append(
            {
                "expiration": (
                    expiration
                ),

                "tau": (
                    tau_value
                ),

                "dte": (
                    tau_value * 365.0
                ),

                "smile": (
                    smile_on_grid
                ),
            }
        )


    if (
        len(expiry_rows)
        < MIN_EXPIRIES_PER_SNAPSHOT
    ):

        return None


    # ========================================================
    # C. Sort actual maturities
    # ========================================================

    expiry_rows = sorted(
        expiry_rows,
        key=lambda row: row[
            "tau"
        ],
    )


    actual_tau = np.array(
        [
            row["tau"]
            for row
            in expiry_rows
        ],
        dtype=np.float64,
    )


    actual_log_tau = np.log(
        actual_tau
    )


    smile_matrix = np.stack(
        [
            row[
                "smile"
            ]
            for row
            in expiry_rows
        ],
        axis=0,
    )

    # shape:
    #
    # [number of actual expiries, 41]


    # ========================================================
    # D. Output
    # ========================================================

    surface = np.zeros(
        (
            N_K,
            N_TAU,
        ),
        dtype=np.float64,
    )


    mask = np.zeros(
        (
            N_K,
            N_TAU,
        ),
        dtype=np.float32,
    )


    # ========================================================
    # E. Second interpolation:
    #
    # for every fixed k coordinate,
    # interpolate across maturity.
    # ========================================================

    for k_index in range(
        N_K
    ):

        values = (
            smile_matrix[
                :,
                k_index,
            ]
        )


        valid = np.isfinite(
            values
        )


        # Need at least two actual maturities
        if (
            valid.sum()
            < 2
        ):

            continue


        x = (
            actual_log_tau[
                valid
            ]
        )


        y = (
            values[
                valid
            ]
        )


        order = np.argsort(
            x
        )


        x = x[
            order
        ]

        y = y[
            order
        ]


        supported_tau = np.exp(
            x
        )


        min_tau = float(
            supported_tau.min()
        )

        max_tau = float(
            supported_tau.max()
        )


        # ----------------------------------------------------
        # Evaluate each target maturity
        # ----------------------------------------------------

        for (
            tau_index,
            target_tau,
        ) in enumerate(
            TAU_GRID
        ):

            target_tau = float(
                target_tau
            )


            target_log_tau = float(
                np.log(
                    target_tau
                )
            )


            # =================================================
            # CASE 1
            # Inside maturity range:
            # normal interpolation
            # =================================================

            if (
                min_tau
                <= target_tau
                <= max_tau
            ):

                value = float(
                    np.interp(
                        target_log_tau,
                        x,
                        y,
                    )
                )


                if (
                    np.isfinite(
                        value
                    )
                    and value > 0
                ):

                    surface[
                        k_index,
                        tau_index,
                    ] = value


                    mask[
                        k_index,
                        tau_index,
                    ] = 1.0


                continue


            # =================================================
            # CASE 2
            # Slightly below shortest maturity
            #
            # Example:
            # target = 7D
            # actual = 7.125D
            # =================================================

            if (
                target_tau
                < min_tau
            ):

                relative_gap = (
                    (
                        min_tau
                        - target_tau
                    )
                    / target_tau
                )


                if (
                    relative_gap
                    <= MAX_BOUNDARY_DTE_RELATIVE_GAP
                ):

                    value = (
                        linear_extrapolate(
                            x0=float(
                                x[0]
                            ),

                            y0=float(
                                y[0]
                            ),

                            x1=float(
                                x[1]
                            ),

                            y1=float(
                                y[1]
                            ),

                            x_target=(
                                target_log_tau
                            ),
                        )
                    )


                    if (
                        np.isfinite(
                            value
                        )
                        and value > 0
                    ):

                        surface[
                            k_index,
                            tau_index,
                        ] = float(
                            value
                        )


                        mask[
                            k_index,
                            tau_index,
                        ] = 1.0


                continue


            # =================================================
            # CASE 3
            # Slightly above longest maturity
            #
            # Example:
            # target = 270D
            # actual = 267D
            # =================================================

            if (
                target_tau
                > max_tau
            ):

                relative_gap = (
                    (
                        target_tau
                        - max_tau
                    )
                    / target_tau
                )


                if (
                    relative_gap
                    <= MAX_BOUNDARY_DTE_RELATIVE_GAP
                ):

                    value = (
                        linear_extrapolate(
                            x0=float(
                                x[-2]
                            ),

                            y0=float(
                                y[-2]
                            ),

                            x1=float(
                                x[-1]
                            ),

                            y1=float(
                                y[-1]
                            ),

                            x_target=(
                                target_log_tau
                            ),
                        )
                    )


                    if (
                        np.isfinite(
                            value
                        )
                        and value > 0
                    ):

                        surface[
                            k_index,
                            tau_index,
                        ] = float(
                            value
                        )


                        mask[
                            k_index,
                            tau_index,
                        ] = 1.0


                continue


    # ========================================================
    # F. Sanity checks
    # ========================================================

    if (
        mask.sum()
        == 0
    ):

        return None


    # Unsupported positions MUST stay zero
    if not np.allclose(
        surface[
            mask == 0
        ],
        0.0,
    ):

        raise RuntimeError(
            "mask=0 locations contain "
            "non-zero surface values."
        )


    supported_values = (
        surface[
            mask == 1
        ]
    )


    if not np.isfinite(
        supported_values
    ).all():

        raise RuntimeError(
            "Non-finite supported values."
        )


    # ========================================================
    # G. Diagnostics
    # ========================================================

    support_by_dte = (
        mask.mean(
            axis=0
        )
    )


    diagnostics = {
        "timestamp": (
            snapshot[
                "timestamp"
            ].iloc[0]
        ),

        "n_quotes": int(
            len(
                snapshot
            )
        ),

        "n_actual_expiries": int(
            snapshot[
                "optionExpirationDate"
            ].nunique()
        ),

        "usable_expiries": int(
            len(
                expiry_rows
            )
        ),

        "support_fraction": float(
            mask.mean()
        ),

        "support_by_dte": (
            support_by_dte
        ),
    }


    return (
        surface,
        mask,
        diagnostics,
    )


print(
    "build_masked_surface defined:",
    "build_masked_surface"
    in globals()
)

In [ ]:
# ============================================================
# TEST BLOCK 7
# ============================================================

test_timestamp = pd.Timestamp(
    "2023-01-05 13:00:00"
)


test_snapshot = (
    debug_clean_quotes[
        debug_clean_quotes[
            "timestamp"
        ]
        == test_timestamp
    ]
    .copy()
)


result = (
    build_masked_surface(
        test_snapshot
    )
)


if result is None:

    raise RuntimeError(
        "Masked surface construction failed."
    )


(
    test_surface,
    test_mask,
    test_diagnostics,
) = result


print(
    "\nSurface shape:",
    test_surface.shape
)

print(
    "Mask shape:",
    test_mask.shape
)

print(
    "Mask values:",
    np.unique(
        test_mask
    )
)

print(
    "Overall support:",
    test_diagnostics[
        "support_fraction"
    ]
)

print(
    "Usable expiries:",
    test_diagnostics[
        "usable_expiries"
    ]
)


# ------------------------------------------------------------
# Support by maturity
# ------------------------------------------------------------

support_table = pd.DataFrame(
    {
        "DTE": (
            DTE_GRID.astype(
                int
            )
        ),

        "supported_k_points": (
            test_mask
            .sum(
                axis=0
            )
            .astype(
                int
            )
        ),

        "total_k_points": (
            N_K
        ),

        "support_fraction": (
            test_mask.mean(
                axis=0
            )
        ),
    }
)


print(
    "\nSupport by target maturity:"
)

display(
    support_table
)


# ------------------------------------------------------------
# Verify unsupported values are zero
# ------------------------------------------------------------

print(
    "\nMax abs value where mask=0:"
)

print(
    np.max(
        np.abs(
            test_surface[
                test_mask == 0
            ]
        )
    )
)


# ------------------------------------------------------------
# Plot mask
# ------------------------------------------------------------

plt.figure(
    figsize=(9, 5)
)


plt.imshow(
    test_mask.T,
    origin="lower",
    aspect="auto",
    extent=[
        K_GRID.min(),
        K_GRID.max(),
        DTE_GRID.min(),
        DTE_GRID.max(),
    ],
    interpolation="nearest",
)


plt.xlabel(
    "log-moneyness"
)

plt.ylabel(
    "DTE"
)

plt.title(
    "Support mask — 2023-01-05 13:00"
)

plt.colorbar(
    label="1 = supported"
)

plt.show()

In [ ]:
# ============================================================
# BLOCK 8
# Build all masked surfaces for one trading day
# ============================================================


def build_day_masked_surfaces(
    clean_quotes: pd.DataFrame,
) -> pd.DataFrame:
    """
    Convert one day's cleaned option quotes
    into one masked 41x10 surface per timestamp.

    Output:
        one row per timestamp

    Contains:
        metadata
        410 surface columns
        410 mask columns
        per-DTE support diagnostics
    """

    rows = []


    grouped = (
        clean_quotes
        .groupby(
            "timestamp",
            observed=True,
        )
    )


    total_timestamps = (
        clean_quotes[
            "timestamp"
        ].nunique()
    )


    print(
        "timestamps to build:",
        total_timestamps
    )


    for index, (
        timestamp,
        snapshot,
    ) in enumerate(
        grouped,
        start=1,
    ):

        result = (
            build_masked_surface(
                snapshot
            )
        )


        if result is None:

            print(
                f"WARNING: failed surface at "
                f"{timestamp}"
            )

            continue


        (
            surface,
            mask,
            diagnostics,
        ) = result


        row = {
            "timestamp": (
                pd.Timestamp(
                    timestamp
                )
            ),

            "trade_date": (
                pd.Timestamp(
                    timestamp
                ).normalize()
            ),

            "clock_time": (
                pd.Timestamp(
                    timestamp
                ).strftime(
                    "%H:%M:%S"
                )
            ),

            "underlying": float(
                snapshot[
                    "underlyingPrice"
                ].median()
            ),

            "n_quotes": int(
                len(
                    snapshot
                )
            ),

            "n_actual_expiries": int(
                diagnostics[
                    "n_actual_expiries"
                ]
            ),

            "usable_expiries": int(
                diagnostics[
                    "usable_expiries"
                ]
            ),

            "support_fraction": float(
                diagnostics[
                    "support_fraction"
                ]
            ),
        }


        # ----------------------------------------------------
        # Store surface values
        # ----------------------------------------------------

        for k_index in range(
            N_K
        ):

            for tau_index in range(
                N_TAU
            ):

                surface_column = (
                    f"w_k{k_index:02d}_"
                    f"t{tau_index:02d}"
                )

                mask_column = (
                    f"m_k{k_index:02d}_"
                    f"t{tau_index:02d}"
                )


                row[
                    surface_column
                ] = float(
                    surface[
                        k_index,
                        tau_index,
                    ]
                )


                row[
                    mask_column
                ] = int(
                    mask[
                        k_index,
                        tau_index,
                    ]
                )


        # ----------------------------------------------------
        # Store support by DTE
        # ----------------------------------------------------

        support_by_dte = (
            diagnostics[
                "support_by_dte"
            ]
        )


        for tau_index, dte in enumerate(
            DTE_GRID
        ):

            row[
                f"support_dte_{int(dte)}"
            ] = float(
                support_by_dte[
                    tau_index
                ]
            )


        rows.append(
            row
        )


        if (
            index == 1
            or
            index % 5 == 0
            or
            index == total_timestamps
        ):

            print(
                f"  built "
                f"{index}/"
                f"{total_timestamps}"
            )


    if len(
        rows
    ) == 0:

        raise RuntimeError(
            "No intraday masked surfaces "
            "were successfully built."
        )


    day_table = (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "timestamp"
        )
        .reset_index(
            drop=True
        )
    )


    return (
        day_table
    )


print(
    "build_day_masked_surfaces defined:",
    "build_day_masked_surfaces"
    in globals()
)

In [ ]:
# ============================================================
# TEST BLOCK 8
# ============================================================

debug_day_surfaces = (
    build_day_masked_surfaces(
        debug_clean_quotes
    )
)


print(
    "\nDay surface table shape:"
)

print(
    debug_day_surfaces.shape
)


print(
    "\nNumber of timestamps:"
)

print(
    debug_day_surfaces[
        "timestamp"
    ].nunique()
)


print(
    "\nFirst / last timestamp:"
)

print(
    debug_day_surfaces[
        "timestamp"
    ].min(),
    "->",
    debug_day_surfaces[
        "timestamp"
    ].max()
)


print(
    "\nSupport fraction summary:"
)

display(
    debug_day_surfaces[
        "support_fraction"
    ].describe()
)


support_columns = [
    f"support_dte_{int(dte)}"
    for dte in DTE_GRID
]


print(
    "\nAverage support by DTE:"
)

display(
    debug_day_surfaces[
        support_columns
    ]
    .mean()
    .rename(
        "mean_support"
    )
    .to_frame()
)


print(
    "\nFirst rows:"
)

display(
    debug_day_surfaces[
        [
            "timestamp",
            "underlying",
            "n_quotes",
            "usable_expiries",
            "support_fraction",
        ]
    ]
    .head()
)

In [ ]:
# ============================================================
# BLOCK 9
# Save and reload ONE day's masked surfaces
#
# Test only:
#     2023-01-05
#
# Do NOT batch process all dates yet.
# ============================================================


# ------------------------------------------------------------
# 1. Save helper
# ------------------------------------------------------------

def save_surface_table(
    table: pd.DataFrame,
    trade_date: str,
) -> Path:
    """
    Prefer parquet.

    If pyarrow / parquet engine is unavailable,
    automatically fall back to CSV.
    """

    parquet_path = (
        SURFACE_ROOT
        / f"{trade_date}.parquet"
    )


    try:

        table.to_parquet(
            parquet_path,
            index=False,
        )

        return parquet_path


    except (
        ImportError,
        ModuleNotFoundError,
    ):

        csv_path = (
            SURFACE_ROOT
            / f"{trade_date}.csv"
        )


        table.to_csv(
            csv_path,
            index=False,
        )


        return csv_path


# ------------------------------------------------------------
# 2. Read helper
# ------------------------------------------------------------

def read_surface_table(
    path: Path,
) -> pd.DataFrame:

    if (
        path.suffix.lower()
        == ".parquet"
    ):

        table = pd.read_parquet(
            path
        )

    elif (
        path.suffix.lower()
        == ".csv"
    ):

        table = pd.read_csv(
            path
        )

    else:

        raise ValueError(
            f"Unsupported file type: "
            f"{path.suffix}"
        )


    # Restore datetime columns
    table[
        "timestamp"
    ] = pd.to_datetime(
        table[
            "timestamp"
        ],
        errors="coerce",
    )


    table[
        "trade_date"
    ] = pd.to_datetime(
        table[
            "trade_date"
        ],
        errors="coerce",
    )


    return table


# ============================================================
# 3. Save DEBUG_DATE
# ============================================================

debug_saved_path = (
    save_surface_table(
        debug_day_surfaces,
        DEBUG_DATE,
    )
)


print(
    "Saved file:"
)

print(
    debug_saved_path
)


print(
    "\nFile exists:"
)

print(
    debug_saved_path.exists()
)


print(
    "\nFile size (MB):"
)

print(
    round(
        debug_saved_path.stat().st_size
        / 1024
        / 1024,
        3,
    )
)


# ============================================================
# 4. Reload from disk
# ============================================================

debug_reloaded = (
    read_surface_table(
        debug_saved_path
    )
)


print(
    "\nReloaded shape:"
)

print(
    debug_reloaded.shape
)


print(
    "\nReloaded timestamps:"
)

print(
    debug_reloaded[
        "timestamp"
    ].nunique()
)


print(
    "\nFirst / last timestamp:"
)

print(
    debug_reloaded[
        "timestamp"
    ].min(),
    "->",
    debug_reloaded[
        "timestamp"
    ].max()
)


# ============================================================
# 5. Verify all expected columns exist
# ============================================================

missing_surface_columns = [
    column
    for column in GRID_COLUMNS
    if column
    not in debug_reloaded.columns
]


missing_mask_columns = [
    column
    for column in MASK_COLUMNS
    if column
    not in debug_reloaded.columns
]


print(
    "\nMissing surface columns:"
)

print(
    len(
        missing_surface_columns
    )
)


print(
    "Missing mask columns:"
)

print(
    len(
        missing_mask_columns
    )
)


# ============================================================
# 6. Verify masks are binary
# ============================================================

reloaded_masks = (
    debug_reloaded[
        MASK_COLUMNS
    ]
    .to_numpy(
        dtype=np.float32
    )
)


print(
    "\nMask unique values:"
)

print(
    np.unique(
        reloaded_masks
    )
)


# ============================================================
# 7. Verify unsupported surface locations remain zero
# ============================================================

reloaded_surfaces = (
    debug_reloaded[
        GRID_COLUMNS
    ]
    .to_numpy(
        dtype=np.float64
    )
)


unsupported_values = (
    reloaded_surfaces[
        reloaded_masks == 0
    ]
)


print(
    "\nMax abs surface value where mask=0:"
)

print(
    np.max(
        np.abs(
            unsupported_values
        )
    )
)


# ============================================================
# 8. Compare saved/reloaded numeric values
# ============================================================

original_surfaces = (
    debug_day_surfaces[
        GRID_COLUMNS
    ]
    .to_numpy(
        dtype=np.float64
    )
)


surface_difference = (
    reloaded_surfaces
    - original_surfaces
)


print(
    "\nMax surface difference after reload:"
)

print(
    np.max(
        np.abs(
            surface_difference
        )
    )
)


original_masks = (
    debug_day_surfaces[
        MASK_COLUMNS
    ]
    .to_numpy(
        dtype=np.float32
    )
)


mask_difference = (
    reloaded_masks
    - original_masks
)


print(
    "\nMax mask difference after reload:"
)

print(
    np.max(
        np.abs(
            mask_difference
        )
    )
)


# ============================================================
# 9. Final sanity checks
# ============================================================

assert (
    debug_reloaded.shape
    == debug_day_surfaces.shape
)


assert (
    len(
        missing_surface_columns
    )
    == 0
)


assert (
    len(
        missing_mask_columns
    )
    == 0
)


assert set(
    np.unique(
        reloaded_masks
    ).tolist()
).issubset(
    {
        0.0,
        1.0,
    }
)


assert np.allclose(
    unsupported_values,
    0.0,
)


assert np.allclose(
    reloaded_surfaces,
    original_surfaces,
)


assert np.array_equal(
    reloaded_masks,
    original_masks,
)


print(
    "\nBLOCK 9 PASSED."
)

print(
    "The masked surface file "
    "was saved and reloaded correctly."
)

In [ ]:
# ============================================================
# BLOCK 10
# Build + save masked surfaces for ONE arbitrary trading day
#
# Pipeline:
#   raw contract CSVs
#       ↓
#   load_day_raw_quotes()
#       ↓
#   clean_day_quotes()
#       ↓
#   build_day_masked_surfaces()
#       ↓
#   save daily parquet/csv
#
# First test only:
#   2023-01-03
# ============================================================


def save_surface_table(
    table: pd.DataFrame,
    trade_date: str,
) -> Path:
    """
    Prefer parquet.
    Fall back to CSV if parquet engine is unavailable.
    """

    parquet_path = (
        SURFACE_ROOT
        / f"{trade_date}.parquet"
    )

    try:

        table.to_parquet(
            parquet_path,
            index=False,
        )

        return parquet_path

    except (
        ImportError,
        ModuleNotFoundError,
    ):

        csv_path = (
            SURFACE_ROOT
            / f"{trade_date}.csv"
        )

        table.to_csv(
            csv_path,
            index=False,
        )

        return csv_path


def find_existing_surface_file(
    trade_date: str,
) -> Optional[Path]:
    """
    Check whether this day's surface file already exists.
    """

    parquet_path = (
        SURFACE_ROOT
        / f"{trade_date}.parquet"
    )

    csv_path = (
        SURFACE_ROOT
        / f"{trade_date}.csv"
    )

    if parquet_path.exists():
        return parquet_path

    if csv_path.exists():
        return csv_path

    return None


def build_and_save_one_day(
    trade_date: str,
    overwrite: bool = False,
) -> dict:
    """
    Build and save masked SPY surfaces for one day.

    Returns a diagnostic dictionary.
    """

    # --------------------------------------------------------
    # 1. Cache check
    # --------------------------------------------------------

    existing_file = (
        find_existing_surface_file(
            trade_date
        )
    )

    if (
        existing_file is not None
        and not overwrite
    ):

        return {
            "trade_date": trade_date,
            "status": "cached",
            "saved_path": str(
                existing_file
            ),
        }


    try:

        # ====================================================
        # 2. Load raw contracts
        # ====================================================

        raw_quotes = (
            load_day_raw_quotes(
                trade_date
            )
        )


        # ====================================================
        # 3. Clean quotes
        # ====================================================

        (
            clean_quotes,
            audit,
        ) = clean_day_quotes(
            raw_quotes
        )


        if clean_quotes.empty:

            return {
                "trade_date": trade_date,
                "status": "no_clean_quotes",
                "raw_rows": int(
                    len(
                        raw_quotes
                    )
                ),
            }


        # ====================================================
        # 4. Build all intraday masked surfaces
        # ====================================================

        day_surfaces = (
            build_day_masked_surfaces(
                clean_quotes
            )
        )


        if day_surfaces.empty:

            return {
                "trade_date": trade_date,
                "status": "no_surfaces",
                "raw_rows": int(
                    len(
                        raw_quotes
                    )
                ),
                "clean_rows": int(
                    len(
                        clean_quotes
                    )
                ),
            }


        # ====================================================
        # 5. Sanity checks before saving
        # ====================================================

        missing_surface_columns = [
            column
            for column
            in GRID_COLUMNS
            if column
            not in day_surfaces.columns
        ]

        missing_mask_columns = [
            column
            for column
            in MASK_COLUMNS
            if column
            not in day_surfaces.columns
        ]


        if missing_surface_columns:

            raise RuntimeError(
                "Missing surface columns. "
                f"Example: "
                f"{missing_surface_columns[:5]}"
            )


        if missing_mask_columns:

            raise RuntimeError(
                "Missing mask columns. "
                f"Example: "
                f"{missing_mask_columns[:5]}"
            )


        mask_values = (
            day_surfaces[
                MASK_COLUMNS
            ]
            .to_numpy(
                dtype=np.float32
            )
        )


        surface_values = (
            day_surfaces[
                GRID_COLUMNS
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        unique_mask_values = (
            np.unique(
                mask_values
            )
        )


        if not set(
            unique_mask_values.tolist()
        ).issubset(
            {
                0.0,
                1.0,
            }
        ):

            raise RuntimeError(
                "Mask contains invalid values: "
                f"{unique_mask_values}"
            )


        if not np.allclose(
            surface_values[
                mask_values == 0
            ],
            0.0,
        ):

            raise RuntimeError(
                "mask=0 locations contain "
                "non-zero surface values."
            )


        # ====================================================
        # 6. Save daily file
        # ====================================================

        saved_path = (
            save_surface_table(
                day_surfaces,
                trade_date,
            )
        )


        # ====================================================
        # 7. Summary
        # ====================================================

        support_values = (
            day_surfaces[
                "support_fraction"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        result = {
            "trade_date": trade_date,

            "status": "saved",

            "raw_rows": int(
                len(
                    raw_quotes
                )
            ),

            "clean_rows": int(
                len(
                    clean_quotes
                )
            ),

            "n_surfaces": int(
                len(
                    day_surfaces
                )
            ),

            "first_timestamp": str(
                day_surfaces[
                    "timestamp"
                ].min()
            ),

            "last_timestamp": str(
                day_surfaces[
                    "timestamp"
                ].max()
            ),

            "mean_support": float(
                support_values.mean()
            ),

            "median_support": float(
                np.median(
                    support_values
                )
            ),

            "min_support": float(
                support_values.min()
            ),

            "max_support": float(
                support_values.max()
            ),

            "saved_path": str(
                saved_path
            ),
        }


        # ----------------------------------------------------
        # support by DTE
        # ----------------------------------------------------

        for dte in DTE_GRID:

            support_column = (
                f"support_dte_"
                f"{int(dte)}"
            )

            if (
                support_column
                in day_surfaces.columns
            ):

                result[
                    f"mean_support_{int(dte)}D"
                ] = float(
                    day_surfaces[
                        support_column
                    ].mean()
                )


        return result


    except Exception as error:

        return {
            "trade_date": trade_date,
            "status": "failed",
            "error": repr(
                error
            ),
        }


    finally:

        gc.collect()


print(
    "build_and_save_one_day defined:",
    "build_and_save_one_day"
    in globals()
)

In [ ]:
# ============================================================
# TEST BLOCK 10
# ============================================================

TEST_DATE = "2023-01-03"


test_day_result = (
    build_and_save_one_day(
        TEST_DATE,
        overwrite=True,
    )
)


print(
    "\nTest day result:"
)

print(
    json.dumps(
        test_day_result,
        indent=2,
        ensure_ascii=False,
    )
)

In [ ]:
# ============================================================
# BLOCK 11
# Batch build masked surfaces for all available dates
# ============================================================


def discover_trade_dates(
    raw_root: Path = RAW_ROOT,
) -> list[str]:
    """
    Discover all YYYY-MM-DD folders under RAW_ROOT.
    """

    dates = []

    for path in raw_root.iterdir():

        if not path.is_dir():
            continue

        if not re.fullmatch(
            r"\d{4}-\d{2}-\d{2}",
            path.name,
        ):
            continue

        try:
            date = pd.Timestamp(
                path.name
            )
        except Exception:
            continue

        if (
            BUILD_START_DATE
            <= date
            <= BUILD_END_DATE
        ):
            dates.append(
                path.name
            )

    return sorted(
        dates
    )


trade_dates = (
    discover_trade_dates()
)


print(
    "Number of trade-date folders:",
    len(
        trade_dates
    )
)


print(
    "First date:",
    trade_dates[0]
)


print(
    "Last date:",
    trade_dates[-1]
)


print(
    "First 10 dates:"
)

print(
    trade_dates[:10]
)


# ============================================================
# Check existing cached files
# ============================================================

cached_dates = [
    trade_date
    for trade_date in trade_dates
    if (
        find_existing_surface_file(
            trade_date
        )
        is not None
    )
]


print(
    "\nAlready cached days:",
    len(
        cached_dates
    )
)


# ============================================================
# Batch settings
# ============================================================

# False:
# existing files are skipped.
#
# Recommended.
OVERWRITE_ALL = True


print(
    "\nOVERWRITE_ALL:",
    OVERWRITE_ALL
)


# ============================================================
# Full batch run
# ============================================================

build_results = []


for day_index, trade_date in enumerate(
    trade_dates,
    start=1,
):

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"[{day_index}/{len(trade_dates)}] "
        f"{trade_date}"
    )


    result = (
        build_and_save_one_day(
            trade_date,
            overwrite=(
                OVERWRITE_ALL
            ),
        )
    )


    build_results.append(
        result
    )


    print(
        "status:",
        result.get(
            "status"
        )
    )


    if (
        result.get(
            "status"
        )
        == "saved"
    ):

        print(
            "n_surfaces:",
            result.get(
                "n_surfaces"
            )
        )

        print(
            "mean_support:",
            round(
                result.get(
                    "mean_support",
                    np.nan,
                ),
                4,
            )
        )


    elif (
        result.get(
            "status"
        )
        == "failed"
    ):

        print(
            "ERROR:",
            result.get(
                "error"
            )
        )


    # --------------------------------------------------------
    # Save progress every 10 processed days
    # --------------------------------------------------------

    if (
        day_index % 10 == 0
        or
        day_index
        == len(
            trade_dates
        )
    ):

        progress_log = (
            pd.DataFrame(
                build_results
            )
        )


        progress_path = (
            OUTPUT_ROOT
            / "masked_surface_build_progress.csv"
        )


        progress_log.to_csv(
            progress_path,
            index=False,
        )


        print(
            "progress saved:",
            progress_path
        )


print(
    "\nFULL BATCH BUILD FINISHED."
)

In [ ]:
# ============================================================
# Batch summary
# ============================================================

build_log = pd.DataFrame(
    build_results
)


print(
    "\nStatus summary:"
)

display(
    build_log[
        "status"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nProblem days:"
)

problem_days = (
    build_log[
        ~build_log[
            "status"
        ].isin(
            [
                "saved",
                "cached",
            ]
        )
    ]
)


display(
    problem_days
)


successful_days = (
    build_log[
        build_log[
            "status"
        ]
        .eq(
            "saved"
        )
    ]
)


if not successful_days.empty:

    print(
        "\nSupport summary:"
    )

    display(
        successful_days[
            [
                "mean_support",
                "median_support",
                "min_support",
                "max_support",
            ]
        ].describe()
    )


final_log_path = (
    OUTPUT_ROOT
    / "masked_surface_build_log.csv"
)


build_log.to_csv(
    final_log_path,
    index=False,
)


print(
    "\nFinal log saved to:"
)

print(
    final_log_path
)

## 2. Masked total-variance surface construction

Quotes are cleaned using bid/ask consistency, relative spread, stale-quote,
DTE, moneyness, and OTM filters.  The model works with total variance,

\[
w(k,\tau)=\tau\,\sigma_{\mathrm{IV}}^2,
\]

on a fixed 41-by-10 grid.  Unsupported locations remain masked rather than being
artificially extrapolated.


In [ ]:
# ============================================================
# BLOCK 12
# Load all daily masked surface files
# and create train / validation / test splits
# ============================================================


def split_from_date(
    date_value,
) -> Optional[str]:

    date = (
        pd.Timestamp(
            date_value
        )
        .normalize()
    )


    if (
        date
        <= TRAIN_END
    ):

        return "train"


    if (
        date
        <= VALID_END
    ):

        return "validation"


    if (
        date
        <= TEST_END
    ):

        return "test"


    return None


def load_all_masked_surfaces(
    surface_root: Path = SURFACE_ROOT,
) -> pd.DataFrame:

    # --------------------------------------------------------
    # 1. Find daily files
    # --------------------------------------------------------

    file_map = {}


    for path in sorted(
        surface_root.glob(
            "????-??-??.csv"
        )
    ):

        file_map[
            path.stem
        ] = path


    for path in sorted(
        surface_root.glob(
            "????-??-??.parquet"
        )
    ):

        # parquet has priority
        file_map[
            path.stem
        ] = path


    paths = [
        file_map[
            key
        ]
        for key
        in sorted(
            file_map
        )
    ]


    if not paths:

        raise FileNotFoundError(
            f"No masked surface files found in:\n"
            f"{surface_root}"
        )


    print(
        "Daily surface files:",
        len(
            paths
        )
    )


    print(
        "First file:",
        paths[0].name
    )


    print(
        "Last file:",
        paths[-1].name
    )


    # --------------------------------------------------------
    # 2. Read files
    # --------------------------------------------------------

    frames = []


    for index, path in enumerate(
        paths,
        start=1,
    ):

        if (
            path.suffix.lower()
            == ".parquet"
        ):

            frame = pd.read_parquet(
                path
            )

        else:

            frame = pd.read_csv(
                path
            )


        frame[
            "timestamp"
        ] = pd.to_datetime(
            frame[
                "timestamp"
            ],
            errors="coerce",
        )


        frame[
            "trade_date"
        ] = pd.to_datetime(
            frame[
                "trade_date"
            ],
            errors="coerce",
        )


        frames.append(
            frame
        )


        if (
            index == 1
            or
            index % 100 == 0
            or
            index == len(
                paths
            )
        ):

            print(
                f"  loaded "
                f"{index}/"
                f"{len(paths)}"
            )


    # --------------------------------------------------------
    # 3. Concatenate
    # --------------------------------------------------------

    table = pd.concat(
        frames,
        ignore_index=True,
    )


    table = (
        table
        .sort_values(
            [
                "trade_date",
                "timestamp",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # 4. Check expected columns
    # --------------------------------------------------------

    missing_surface_columns = [
        column
        for column
        in GRID_COLUMNS
        if column
        not in table.columns
    ]


    missing_mask_columns = [
        column
        for column
        in MASK_COLUMNS
        if column
        not in table.columns
    ]


    if missing_surface_columns:

        raise RuntimeError(
            "Missing surface columns."
        )


    if missing_mask_columns:

        raise RuntimeError(
            "Missing mask columns."
        )


    # --------------------------------------------------------
    # 5. Basic numeric conversion
    # --------------------------------------------------------

    for column in (
        GRID_COLUMNS
        + MASK_COLUMNS
        + [
            "underlying",
            "support_fraction",
        ]
    ):

        table[
            column
        ] = pd.to_numeric(
            table[
                column
            ],
            errors="coerce",
        )


    # --------------------------------------------------------
    # 6. Keep model session
    #
    # 第一版先只用：
    # 09:45 -> 15:45
    #
    # 16:00 / 16:15 不进入预测模型。
    # --------------------------------------------------------

    table[
        "clock_time"
    ] = (
        table[
            "timestamp"
        ]
        .dt.strftime(
            "%H:%M:%S"
        )
    )


    table = table[
        table[
            "clock_time"
        ]
        .between(
            MODEL_SESSION_START,
            MODEL_SESSION_END,
        )
    ].copy()


    # --------------------------------------------------------
    # 7. Assign split
    # --------------------------------------------------------

    table[
        "split"
    ] = (
        table[
            "trade_date"
        ]
        .map(
            split_from_date
        )
    )


    table = table[
        table[
            "split"
        ].notna()
    ].copy()


    # --------------------------------------------------------
    # 8. Non-destructive exclusion of known unusable dates
    # --------------------------------------------------------

    excluded_dates = {
        pd.Timestamp(x).normalize()
        for x in EXCLUDED_DATES
    }

    table = table[
        ~table[
            "trade_date"
        ].dt.normalize().isin(
            excluded_dates
        )
    ].copy()


    table = (
        table
        .sort_values(
            [
                "trade_date",
                "timestamp",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    return table


# ============================================================
# Run BLOCK 12
# ============================================================

surface_table = (
    load_all_masked_surfaces()
)


print(
    "\nCombined table shape:"
)

print(
    surface_table.shape
)


print(
    "\nUnique trading days:"
)

print(
    surface_table[
        "trade_date"
    ].nunique()
)


print(
    "\nTimestamp range:"
)

print(
    surface_table[
        "timestamp"
    ].min(),
    "->",
    surface_table[
        "timestamp"
    ].max()
)


print(
    "\nSplit counts:"
)

display(
    surface_table[
        "split"
    ].value_counts()
)


print(
    "\nTrading days per split:"
)

display(
    surface_table
    .groupby(
        "split"
    )[
        "trade_date"
    ]
    .nunique()
)


print(
    "\nSurfaces per day:"
)

surfaces_per_day = (
    surface_table
    .groupby(
        "trade_date"
    )
    .size()
)


display(
    surfaces_per_day.describe()
)


print(
    "\nSupport fraction summary:"
)

display(
    surface_table[
        "support_fraction"
    ].describe()
)


print(
    "\nBLOCK 12 finished successfully."
)

## 3. Strict 15-minute forecasting samples

Each sample uses four historical surfaces,
\(W_{t-45},W_{t-30},W_{t-15},W_t\), and predicts the roll-down-adjusted
15-minute innovation.  Only temporally contiguous windows are admitted.


In [ ]:
# ============================================================
# BLOCK 13
# Build strict 15-minute sequence samples
#
# Model input:
#
#   W_{t-45}
#   W_{t-30}
#   W_{t-15}
#   W_t
#
# plus:
#
#   15-min SPY return
#   1-hour realized volatility
#   time-of-day
#
# Target:
#
#   Y_t
#   =
#   W_{t+15}
#   -
#   RollDown(W_t)
#
# Training loss later uses:
#
#   transition_mask
#
# A sample is created ONLY if all required timestamps
# are strictly 15 minutes apart.
# ============================================================


EXPECTED_GAP = pd.Timedelta(
    minutes=INTERVAL_MINUTES
)


# ============================================================
# 1. Mask-aware maturity roll-down
# ============================================================

def roll_down_masked_surface(
    surface: np.ndarray,
    mask: np.ndarray,
    delta_minutes: int = INTERVAL_MINUTES,
):
    """
    Compute approximately:

        R_delta W_t(k, tau)
        =
        W_t(k, tau + delta)

    using only currently supported maturity points.

    Parameters
    ----------
    surface:
        shape [41, 10]

    mask:
        shape [41, 10]

    Returns
    -------
    rolled_surface:
        shape [41, 10]

    rolled_mask:
        shape [41, 10]

    Important
    ---------
    No interpolation is performed through unsupported
    values as if they were observations.

    For each fixed k:
        - use only maturities where mask=1
        - interpolate in log(tau)
        - very small boundary extrapolation is allowed
          using the SAME tolerance as surface construction
    """

    rolled_surface = np.zeros(
        (
            N_K,
            N_TAU,
        ),
        dtype=np.float64,
    )


    rolled_mask = np.zeros(
        (
            N_K,
            N_TAU,
        ),
        dtype=np.float32,
    )


    delta_years = (
        delta_minutes
        / (
            365.0
            * 24.0
            * 60.0
        )
    )


    # Future target tau corresponds to
    # current tau + delta
    query_tau_grid = (
        TAU_GRID
        + delta_years
    )


    for k_index in range(
        N_K
    ):

        valid = (
            mask[
                k_index
            ]
            > 0
        )


        # Need at least two supported maturity points
        if (
            valid.sum()
            < 2
        ):

            continue


        actual_tau = (
            TAU_GRID[
                valid
            ]
        )


        actual_values = (
            surface[
                k_index,
                valid,
            ]
        )


        # Remove anything numerically invalid
        finite = (
            np.isfinite(
                actual_tau
            )
            &
            np.isfinite(
                actual_values
            )
            &
            (
                actual_values
                > 0
            )
        )


        actual_tau = (
            actual_tau[
                finite
            ]
        )

        actual_values = (
            actual_values[
                finite
            ]
        )


        if (
            len(
                actual_tau
            )
            < 2
        ):

            continue


        order = np.argsort(
            actual_tau
        )


        actual_tau = (
            actual_tau[
                order
            ]
        )

        actual_values = (
            actual_values[
                order
            ]
        )


        x = np.log(
            actual_tau
        )

        y = actual_values


        min_tau = float(
            actual_tau.min()
        )

        max_tau = float(
            actual_tau.max()
        )


        for tau_index, query_tau in enumerate(
            query_tau_grid
        ):

            query_tau = float(
                query_tau
            )


            query_log_tau = float(
                np.log(
                    query_tau
                )
            )


            # ------------------------------------------------
            # Case 1:
            # query maturity is inside supported range
            # ------------------------------------------------

            if (
                min_tau
                <= query_tau
                <= max_tau
            ):

                value = float(
                    np.interp(
                        query_log_tau,
                        x,
                        y,
                    )
                )


                if (
                    np.isfinite(
                        value
                    )
                    and value > 0
                ):

                    rolled_surface[
                        k_index,
                        tau_index,
                    ] = value


                    rolled_mask[
                        k_index,
                        tau_index,
                    ] = 1.0


                continue


            # ------------------------------------------------
            # Case 2:
            # slightly beyond longest supported maturity
            #
            # This is mainly relevant for the last DTE.
            # ------------------------------------------------

            if (
                query_tau
                > max_tau
            ):

                relative_gap = (
                    (
                        query_tau
                        - max_tau
                    )
                    / query_tau
                )


                if (
                    relative_gap
                    <= MAX_BOUNDARY_DTE_RELATIVE_GAP
                    and len(x) >= 2
                ):

                    denominator = (
                        x[-1]
                        - x[-2]
                    )


                    if (
                        abs(
                            denominator
                        )
                        > 1e-12
                    ):

                        slope = (
                            y[-1]
                            - y[-2]
                        ) / denominator


                        value = (
                            y[-1]
                            + slope
                            * (
                                query_log_tau
                                - x[-1]
                            )
                        )


                        if (
                            np.isfinite(
                                value
                            )
                            and value > 0
                        ):

                            rolled_surface[
                                k_index,
                                tau_index,
                            ] = float(
                                value
                            )


                            rolled_mask[
                                k_index,
                                tau_index,
                            ] = 1.0


    return (
        rolled_surface,
        rolled_mask,
    )


# ============================================================
# 2. Main sample builder
# ============================================================

def build_sequence_samples(
    table: pd.DataFrame,
):
    """
    Build strict 15-minute forecasting samples.

    For each current time t:

        model history:
            t-45
            t-30
            t-15
            t

        future:
            t+15

    To calculate 1-hour RV we additionally need:
            t-60

    Therefore each accepted sample requires:

        t-60
        t-45
        t-30
        t-15
        t
        t+15

    ALL adjacent timestamps must be exactly 15 minutes apart.

    This automatically handles:
        - early close
        - stale-quote gaps
        - missing timestamps
    """

    history_surface_list = []

    history_mask_list = []

    scalar_feature_list = []

    target_innovation_list = []

    transition_mask_list = []

    rolled_surface_list = []

    rolled_mask_list = []

    future_surface_list = []

    future_mask_list = []


    date_list = []

    timestamp_list = []

    future_timestamp_list = []

    clock_time_list = []

    split_list = []


    skipped_gap_windows = 0

    skipped_no_transition_support = 0

    total_candidate_windows = 0


    # ========================================================
    # Process each trading day independently
    # ========================================================

    for (
        trade_date,
        day,
    ) in table.groupby(
        "trade_date",
        observed=True,
    ):

        day = (
            day
            .sort_values(
                "timestamp"
            )
            .reset_index(
                drop=True
            )
        )


        # ----------------------------------------------------
        # Surface tensor
        # [time, 41, 10]
        # ----------------------------------------------------

        surfaces = (
            day[
                GRID_COLUMNS
            ]
            .to_numpy(
                dtype=np.float64
            )
            .reshape(
                -1,
                N_K,
                N_TAU,
            )
        )


        # ----------------------------------------------------
        # Mask tensor
        # [time, 41, 10]
        # ----------------------------------------------------

        masks = (
            day[
                MASK_COLUMNS
            ]
            .to_numpy(
                dtype=np.float32
            )
            .reshape(
                -1,
                N_K,
                N_TAU,
            )
        )


        # ----------------------------------------------------
        # Underlying
        # ----------------------------------------------------

        underlying = (
            day[
                "underlying"
            ]
            .to_numpy(
                dtype=np.float64
            )
        )


        log_underlying = np.log(
            np.clip(
                underlying,
                1e-12,
                None,
            )
        )


        returns = np.full(
            len(
                day
            ),
            np.nan,
            dtype=np.float64,
        )


        returns[
            1:
        ] = np.diff(
            log_underlying
        )


        # ====================================================
        # Current index starts at 4 because:
        #
        # index current_index-4 = t-60
        # index current_index-3 = t-45
        # index current_index-2 = t-30
        # index current_index-1 = t-15
        # index current_index   = t
        # index current_index+1 = t+15
        # ====================================================

        for current_index in range(
            4,
            len(day) - 1,
        ):

            total_candidate_windows += 1


            start_check_index = (
                current_index
                - 4
            )


            future_index = (
                current_index
                + 1
            )


            # ------------------------------------------------
            # Require SIX timestamps:
            #
            # t-60 ... t+15
            # ------------------------------------------------

            window_times = (
                day.loc[
                    start_check_index:
                    future_index,
                    "timestamp",
                ]
                .reset_index(
                    drop=True
                )
            )


            gaps = (
                window_times
                .diff()
                .dropna()
            )


            # exactly 5 adjacent gaps,
            # each exactly 15 minutes
            if (
                len(gaps)
                != 5
                or not (
                    gaps
                    == EXPECTED_GAP
                ).all()
            ):

                skipped_gap_windows += 1

                continue


            # =================================================
            # History surfaces:
            #
            # t-45,t-30,t-15,t
            #
            # Note:
            # t-60 is only for RV calculation.
            # =================================================

            history_start = (
                current_index
                - HISTORY_LENGTH
                + 1
            )


            history_surfaces = (
                surfaces[
                    history_start:
                    current_index + 1
                ]
            )


            history_masks = (
                masks[
                    history_start:
                    current_index + 1
                ]
            )


            # Should be exactly 4
            if (
                history_surfaces.shape[0]
                != HISTORY_LENGTH
            ):

                continue


            current_surface = (
                surfaces[
                    current_index
                ]
            )


            current_mask = (
                masks[
                    current_index
                ]
            )


            future_surface = (
                surfaces[
                    future_index
                ]
            )


            future_mask = (
                masks[
                    future_index
                ]
            )


            # =================================================
            # Roll-down current surface
            # =================================================

            (
                rolled_surface,
                rolled_mask,
            ) = roll_down_masked_surface(
                surface=(
                    current_surface
                ),
                mask=(
                    current_mask
                ),
                delta_minutes=(
                    INTERVAL_MINUTES
                ),
            )


            # =================================================
            # Transition mask
            #
            # A target point is valid ONLY when:
            #
            # 1. rolled current value is supported
            # 2. future observation is supported
            # =================================================

            transition_mask = (
                (
                    rolled_mask
                    > 0
                )
                &
                (
                    future_mask
                    > 0
                )
            ).astype(
                np.float32
            )


            if (
                transition_mask.sum()
                == 0
            ):

                skipped_no_transition_support += 1

                continue


            # =================================================
            # Innovation target
            #
            # Unsupported values remain zero.
            #
            # Loss later MUST use transition_mask.
            # =================================================

            innovation = np.zeros(
                (
                    N_K,
                    N_TAU,
                ),
                dtype=np.float64,
            )


            valid_transition = (
                transition_mask
                > 0
            )


            innovation[
                valid_transition
            ] = (
                future_surface[
                    valid_transition
                ]
                - rolled_surface[
                    valid_transition
                ]
            )


            # =================================================
            # Market features
            # =================================================

            # Current 15-minute SPY return:
            #
            # S_t / S_{t-15}
            return_15m = (
                log_underlying[
                    current_index
                ]
                - log_underlying[
                    current_index - 1
                ]
            )


            # -------------------------------------------------
            # Past 1-hour realized volatility
            #
            # Four 15-minute returns:
            #
            # t-60 -> t-45
            # t-45 -> t-30
            # t-30 -> t-15
            # t-15 -> t
            # -------------------------------------------------

            one_hour_returns = (
                returns[
                    current_index - 3:
                    current_index + 1
                ]
            )


            if (
                len(
                    one_hour_returns
                )
                != 4
                or not np.isfinite(
                    one_hour_returns
                ).all()
            ):

                continue


            rv_1h = float(
                np.sqrt(
                    np.sum(
                        one_hour_returns**2
                    )
                )
            )


            # =================================================
            # Time-of-day
            # =================================================

            timestamp = pd.Timestamp(
                day.loc[
                    current_index,
                    "timestamp",
                ]
            )


            minute_from_open = (
                timestamp.hour
                * 60
                + timestamp.minute
                - (
                    9 * 60
                    + 30
                )
            )


            time_fraction = (
                minute_from_open
                / 390.0
            )


            sin_time = float(
                np.sin(
                    2.0
                    * np.pi
                    * time_fraction
                )
            )


            cos_time = float(
                np.cos(
                    2.0
                    * np.pi
                    * time_fraction
                )
            )


            scalar_features = np.array(
                [
                    return_15m,
                    rv_1h,
                    sin_time,
                    cos_time,
                ],
                dtype=np.float32,
            )


            # =================================================
            # Save sample
            # =================================================

            history_surface_list.append(
                history_surfaces.astype(
                    np.float32
                )
            )


            history_mask_list.append(
                history_masks.astype(
                    np.float32
                )
            )


            scalar_feature_list.append(
                scalar_features
            )


            target_innovation_list.append(
                innovation.astype(
                    np.float32
                )
            )


            transition_mask_list.append(
                transition_mask.astype(
                    np.float32
                )
            )


            rolled_surface_list.append(
                rolled_surface.astype(
                    np.float32
                )
            )


            rolled_mask_list.append(
                rolled_mask.astype(
                    np.float32
                )
            )


            future_surface_list.append(
                future_surface.astype(
                    np.float32
                )
            )


            future_mask_list.append(
                future_mask.astype(
                    np.float32
                )
            )


            date_list.append(
                trade_date.strftime(
                    "%Y-%m-%d"
                )
            )


            timestamp_list.append(
                str(
                    timestamp
                )
            )


            future_timestamp_list.append(
                str(
                    day.loc[
                        future_index,
                        "timestamp",
                    ]
                )
            )


            clock_time_list.append(
                timestamp.strftime(
                    "%H:%M:%S"
                )
            )


            split_list.append(
                day.loc[
                    current_index,
                    "split",
                ]
            )


    # ========================================================
    # Final checks
    # ========================================================

    if not history_surface_list:

        raise RuntimeError(
            "No valid strict 15-minute samples "
            "were constructed."
        )


    samples = {

        # [N,4,41,10]
        "history_surface": (
            np.stack(
                history_surface_list
            )
        ),

        # [N,4,41,10]
        "history_mask": (
            np.stack(
                history_mask_list
            )
        ),

        # [N,4]
        #
        # return_15m
        # rv_1h
        # sin_time
        # cos_time
        "scalars": (
            np.stack(
                scalar_feature_list
            )
        ),

        # [N,41,10]
        "target": (
            np.stack(
                target_innovation_list
            )
        ),

        # [N,41,10]
        "transition_mask": (
            np.stack(
                transition_mask_list
            )
        ),

        # [N,41,10]
        "rolled": (
            np.stack(
                rolled_surface_list
            )
        ),

        # [N,41,10]
        "rolled_mask": (
            np.stack(
                rolled_mask_list
            )
        ),

        # [N,41,10]
        "future": (
            np.stack(
                future_surface_list
            )
        ),

        # [N,41,10]
        "future_mask": (
            np.stack(
                future_mask_list
            )
        ),

        "date": np.asarray(
            date_list
        ),

        "timestamp": np.asarray(
            timestamp_list
        ),

        "future_timestamp": np.asarray(
            future_timestamp_list
        ),

        "clock_time": np.asarray(
            clock_time_list
        ),

        "split": np.asarray(
            split_list
        ),
    }


    diagnostics = {

        "total_candidate_windows": (
            total_candidate_windows
        ),

        "valid_samples": (
            len(
                history_surface_list
            )
        ),

        "skipped_gap_windows": (
            skipped_gap_windows
        ),

        "skipped_no_transition_support": (
            skipped_no_transition_support
        ),
    }


    return (
        samples,
        diagnostics,
    )


# ============================================================
# 3. BUILD SAMPLES
# ============================================================

(
    samples,
    sequence_diagnostics,
) = build_sequence_samples(
    surface_table
)


# ============================================================
# 4. PRINT DIAGNOSTICS
# ============================================================

print(
    "Sequence diagnostics:"
)

print(
    sequence_diagnostics
)


print(
    "\nArray shapes:"
)


print(
    "history_surface:",
    samples[
        "history_surface"
    ].shape
)


print(
    "history_mask:",
    samples[
        "history_mask"
    ].shape
)


print(
    "scalars:",
    samples[
        "scalars"
    ].shape
)


print(
    "target:",
    samples[
        "target"
    ].shape
)


print(
    "transition_mask:",
    samples[
        "transition_mask"
    ].shape
)


print(
    "rolled:",
    samples[
        "rolled"
    ].shape
)


print(
    "future:",
    samples[
        "future"
    ].shape
)


# ============================================================
# 5. Split counts
# ============================================================

print(
    "\nSample counts by split:"
)

display(
    pd.Series(
        samples[
            "split"
        ]
    )
    .value_counts()
)


# ============================================================
# 6. Average transition support
# ============================================================

transition_support_per_sample = (
    samples[
        "transition_mask"
    ]
    .mean(
        axis=(1, 2)
    )
)


print(
    "\nTransition support per sample:"
)

display(
    pd.Series(
        transition_support_per_sample
    )
    .describe()
)


# ============================================================
# 7. Samples per trading day
# ============================================================

samples_per_day = (
    pd.Series(
        samples[
            "date"
        ]
    )
    .value_counts()
)


print(
    "\nSamples per day:"
)

display(
    samples_per_day.describe()
)


# ============================================================
# 8. Verify every sample is really 15 minutes ahead
# ============================================================

current_ts = pd.to_datetime(
    samples[
        "timestamp"
    ]
)


future_ts = pd.to_datetime(
    samples[
        "future_timestamp"
    ]
)


horizon_minutes = (
    (
        future_ts
        - current_ts
    )
    .total_seconds()
    / 60.0
)


print(
    "\nUnique forecast horizons:"
)

print(
    np.unique(
        horizon_minutes
    )
)


assert np.allclose(
    horizon_minutes,
    15.0,
)


# ============================================================
# 9. Show a few examples
# ============================================================

sample_metadata = pd.DataFrame(
    {
        "date": (
            samples[
                "date"
            ][:10]
        ),

        "current_time": (
            samples[
                "timestamp"
            ][:10]
        ),

        "future_time": (
            samples[
                "future_timestamp"
            ][:10]
        ),

        "clock_time": (
            samples[
                "clock_time"
            ][:10]
        ),

        "split": (
            samples[
                "split"
            ][:10]
        ),

        "transition_support": (
            transition_support_per_sample[
                :10
            ]
        ),
    }
)


print(
    "\nFirst samples:"
)

display(
    sample_metadata
)


print(
    "\nBLOCK 13 finished successfully."
)

In [ ]:
# ============================================================
# BLOCK 14
# Train-only normalization
# + transition-support threshold diagnostics
#
# Input:
#     samples from Block 13
#
# This block does NOT train CNN yet.
# ============================================================


# ============================================================
# 1. Split masks
# ============================================================

train_sample_mask = (
    samples["split"]
    == "train"
)

validation_sample_mask = (
    samples["split"]
    == "validation"
)

test_sample_mask = (
    samples["split"]
    == "test"
)


print(
    "train samples:",
    int(
        train_sample_mask.sum()
    )
)

print(
    "validation samples:",
    int(
        validation_sample_mask.sum()
    )
)

print(
    "test samples:",
    int(
        test_sample_mask.sum()
    )
)


# ============================================================
# 2. Transition support diagnostics
# ============================================================

transition_support = (
    samples[
        "transition_mask"
    ]
    .mean(
        axis=(1, 2)
    )
)


print(
    "\nOverall transition-support summary:"
)

display(
    pd.Series(
        transition_support
    ).describe()
)


# ------------------------------------------------------------
# Check how many samples survive different thresholds
# ------------------------------------------------------------

threshold_rows = []


for threshold in [
    0.00,
    0.02,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
]:

    keep = (
        transition_support
        >= threshold
    )


    train_keep = (
        keep
        & train_sample_mask
    )

    validation_keep = (
        keep
        & validation_sample_mask
    )

    test_keep = (
        keep
        & test_sample_mask
    )


    threshold_rows.append(
        {
            "threshold": threshold,

            "all_kept": int(
                keep.sum()
            ),

            "all_keep_pct": float(
                keep.mean()
            ),

            "train_kept": int(
                train_keep.sum()
            ),

            "validation_kept": int(
                validation_keep.sum()
            ),

            "test_kept": int(
                test_keep.sum()
            ),
        }
    )


threshold_table = pd.DataFrame(
    threshold_rows
)


print(
    "\nTransition-support threshold table:"
)

display(
    threshold_table
)


# ============================================================
# 3. Choose threshold
#
# First version:
# use 10%.
#
# You can change this after looking at the table.
# ============================================================

MIN_TRANSITION_SUPPORT = 0.10


keep_sample_mask = (
    transition_support
    >= MIN_TRANSITION_SUPPORT
)


print(
    "\nSelected MIN_TRANSITION_SUPPORT:"
)

print(
    MIN_TRANSITION_SUPPORT
)


print(
    "Samples kept:"
)

print(
    int(
        keep_sample_mask.sum()
    ),
    "/",
    len(
        keep_sample_mask
    )
)


print(
    "Keep percentage:"
)

print(
    float(
        keep_sample_mask.mean()
    )
)


# ============================================================
# 4. Effective train/validation/test masks
# ============================================================

effective_train_mask = (
    train_sample_mask
    & keep_sample_mask
)

effective_validation_mask = (
    validation_sample_mask
    & keep_sample_mask
)

effective_test_mask = (
    test_sample_mask
    & keep_sample_mask
)


print(
    "\nEffective split counts:"
)

print(
    "train:",
    int(
        effective_train_mask.sum()
    )
)

print(
    "validation:",
    int(
        effective_validation_mask.sum()
    )
)

print(
    "test:",
    int(
        effective_test_mask.sum()
    )
)


# ============================================================
# 5. Train-only normalization
#
# IMPORTANT:
# all means/stds are computed ONLY from effective TRAIN samples.
# ============================================================


# ------------------------------------------------------------
# 5A. Surface normalization
#
# Use only points with history_mask = 1.
#
# We calculate one mean/std per grid point:
# shape [41,10]
# ------------------------------------------------------------

train_history_surface = (
    samples[
        "history_surface"
    ][
        effective_train_mask
    ]
)


train_history_mask = (
    samples[
        "history_mask"
    ][
        effective_train_mask
    ]
)


# Sum over:
# sample dimension
# history-time dimension
surface_weight = (
    train_history_mask.sum(
        axis=(0, 1)
    )
)


surface_sum = (
    (
        train_history_surface
        * train_history_mask
    )
    .sum(
        axis=(0, 1)
    )
)


surface_mean = (
    surface_sum
    / np.maximum(
        surface_weight,
        1.0,
    )
)


surface_squared_sum = (
    (
        train_history_surface**2
        * train_history_mask
    )
    .sum(
        axis=(0, 1)
    )
)


surface_second_moment = (
    surface_squared_sum
    / np.maximum(
        surface_weight,
        1.0,
    )
)


surface_variance = (
    surface_second_moment
    - surface_mean**2
)


surface_variance = np.maximum(
    surface_variance,
    1e-12,
)


surface_std = np.sqrt(
    surface_variance
)


# ------------------------------------------------------------
# 5B. Scalar normalization
#
# shape [4]
# ------------------------------------------------------------

train_scalars = (
    samples[
        "scalars"
    ][
        effective_train_mask
    ]
)


scalar_mean = (
    train_scalars.mean(
        axis=0
    )
)


scalar_std = (
    train_scalars.std(
        axis=0
    )
)


scalar_std = np.maximum(
    scalar_std,
    1e-8,
)


# ------------------------------------------------------------
# 5C. Target innovation normalization
#
# Use ONLY transition_mask=1.
#
# One mean/std per grid point.
# shape [41,10]
# ------------------------------------------------------------

train_target = (
    samples[
        "target"
    ][
        effective_train_mask
    ]
)


train_transition_mask = (
    samples[
        "transition_mask"
    ][
        effective_train_mask
    ]
)


target_weight = (
    train_transition_mask.sum(
        axis=0
    )
)


target_sum = (
    (
        train_target
        * train_transition_mask
    )
    .sum(
        axis=0
    )
)


target_mean = (
    target_sum
    / np.maximum(
        target_weight,
        1.0,
    )
)


target_squared_sum = (
    (
        train_target**2
        * train_transition_mask
    )
    .sum(
        axis=0
    )
)


target_second_moment = (
    target_squared_sum
    / np.maximum(
        target_weight,
        1.0,
    )
)


target_variance = (
    target_second_moment
    - target_mean**2
)


target_variance = np.maximum(
    target_variance,
    1e-14,
)


target_std = np.sqrt(
    target_variance
)


# ============================================================
# 6. Fallback for grid points with insufficient train support
# ============================================================

# ------------------------------------------------------------
# Some grid points may have zero or tiny observations.
# For those points we cannot trust pointwise std.
#
# Use global supported train std as fallback.
# ------------------------------------------------------------

supported_surface_values = (
    train_history_surface[
        train_history_mask
        > 0
    ]
)


if len(
    supported_surface_values
) == 0:

    raise RuntimeError(
        "No supported surface values "
        "in training set."
    )


global_surface_std = float(
    np.std(
        supported_surface_values
    )
)


global_surface_std = max(
    global_surface_std,
    1e-8,
)


low_surface_support = (
    surface_weight
    < 20
)


surface_std[
    low_surface_support
] = global_surface_std


# Target fallback
supported_target_values = (
    train_target[
        train_transition_mask
        > 0
    ]
)


if len(
    supported_target_values
) == 0:

    raise RuntimeError(
        "No supported target values "
        "in training set."
    )


global_target_std = float(
    np.std(
        supported_target_values
    )
)


global_target_std = max(
    global_target_std,
    1e-8,
)


low_target_support = (
    target_weight
    < 20
)


target_std[
    low_target_support
] = global_target_std


# ============================================================
# 7. Store normalizers
# ============================================================

normalizers = {

    "surface_mean": (
        surface_mean.astype(
            np.float32
        )
    ),

    "surface_std": (
        surface_std.astype(
            np.float32
        )
    ),

    "scalar_mean": (
        scalar_mean.astype(
            np.float32
        )
    ),

    "scalar_std": (
        scalar_std.astype(
            np.float32
        )
    ),

    "target_mean": (
        target_mean.astype(
            np.float32
        )
    ),

    "target_std": (
        target_std.astype(
            np.float32
        )
    ),

    "surface_weight": (
        surface_weight.astype(
            np.float32
        )
    ),

    "target_weight": (
        target_weight.astype(
            np.float32
        )
    ),
}


# ============================================================
# 8. Diagnostics
# ============================================================

print(
    "\nSurface normalizer shapes:"
)

print(
    "surface_mean:",
    normalizers[
        "surface_mean"
    ].shape
)

print(
    "surface_std:",
    normalizers[
        "surface_std"
    ].shape
)


print(
    "\nScalar mean:"
)

print(
    normalizers[
        "scalar_mean"
    ]
)


print(
    "\nScalar std:"
)

print(
    normalizers[
        "scalar_std"
    ]
)


print(
    "\nTarget normalizer shapes:"
)

print(
    "target_mean:",
    normalizers[
        "target_mean"
    ].shape
)

print(
    "target_std:",
    normalizers[
        "target_std"
    ].shape
)


print(
    "\nTrain support counts per grid point:"
)

print(
    "surface_weight min / median / max:"
)

print(
    float(
        surface_weight.min()
    ),
    float(
        np.median(
            surface_weight
        )
    ),
    float(
        surface_weight.max()
    ),
)


print(
    "target_weight min / median / max:"
)

print(
    float(
        target_weight.min()
    ),
    float(
        np.median(
            target_weight
        )
    ),
    float(
        target_weight.max()
    ),
)


print(
    "\nLow-support surface grid points (<20 obs):"
)

print(
    int(
        low_surface_support.sum()
    ),
    "/",
    N_K * N_TAU
)


print(
    "Low-support target grid points (<20 obs):"
)

print(
    int(
        low_target_support.sum()
    ),
    "/",
    N_K * N_TAU
)


# ============================================================
# 9. Final consistency checks
# ============================================================

assert (
    normalizers[
        "surface_mean"
    ].shape
    == (
        N_K,
        N_TAU,
    )
)


assert (
    normalizers[
        "target_mean"
    ].shape
    == (
        N_K,
        N_TAU,
    )
)


assert (
    normalizers[
        "scalar_mean"
    ].shape
    == (4,)
)


assert np.isfinite(
    normalizers[
        "surface_mean"
    ]
).all()


assert np.isfinite(
    normalizers[
        "surface_std"
    ]
).all()


assert np.isfinite(
    normalizers[
        "target_mean"
    ]
).all()


assert np.isfinite(
    normalizers[
        "target_std"
    ]
).all()


print(
    "\nBLOCK 14 finished successfully."
)

In [ ]:
print("First sample date:", samples["date"][0])
print("Last sample date:", samples["date"][-1])

print(
    "Unique sample dates:",
    len(np.unique(samples["date"]))
)

print(
    "First 10 unique dates:",
    np.unique(samples["date"])[:10]
)

print(
    "Last 10 unique dates:",
    np.unique(samples["date"])[-10:]
)

In [ ]:
# ============================================================
# BLOCK 15
# Mask-aware PyTorch Dataset + DataLoaders
#
# CNN input channels:
#
#   4 history surfaces
#   4 history masks
#   4 scalar features
#   1 k coordinate
#   1 tau coordinate
#
# Total = 14 channels
#
# IMPORTANT:
# transition_mask is NOT used as model input.
# It contains future support information and is used
# ONLY for loss calculation.
# ============================================================


# ============================================================
# 1. Coordinate channels
# ============================================================

# ------------------------------------------------------------
# k coordinate
# scale approximately to [-1, 1]
# ------------------------------------------------------------

k_scale = max(
    abs(K_GRID.min()),
    abs(K_GRID.max()),
)

k_coordinate = (
    K_GRID
    / k_scale
)

k_channel = np.repeat(
    k_coordinate[:, None],
    N_TAU,
    axis=1,
).astype(
    np.float32
)


# ------------------------------------------------------------
# tau coordinate
#
# maturities are unevenly spaced,
# so use log(tau), then scale to [-1, 1]
# ------------------------------------------------------------

log_tau = np.log(
    TAU_GRID
)

tau_coordinate = (
    2.0
    * (
        log_tau
        - log_tau.min()
    )
    / (
        log_tau.max()
        - log_tau.min()
    )
    - 1.0
)

tau_channel = np.repeat(
    tau_coordinate[None, :],
    N_K,
    axis=0,
).astype(
    np.float32
)


print(
    "k_channel shape:",
    k_channel.shape
)

print(
    "tau_channel shape:",
    tau_channel.shape
)


# ============================================================
# 2. Dataset
# ============================================================

class MaskedSurfaceDataset(
    Dataset
):

    def __init__(
        self,
        samples,
        normalizers,
        split,
        min_transition_support=(
            MIN_TRANSITION_SUPPORT
        ),
    ):

        self.samples = samples
        self.normalizers = normalizers
        self.split = split

        # ----------------------------------------------------
        # split filter
        # ----------------------------------------------------

        split_mask = (
            samples["split"]
            == split
        )

        # ----------------------------------------------------
        # transition-support filter
        # ----------------------------------------------------

        transition_support = (
            samples[
                "transition_mask"
            ]
            .mean(
                axis=(1, 2)
            )
        )

        support_filter = (
            transition_support
            >= min_transition_support
        )

        self.indices = np.where(
            split_mask
            & support_filter
        )[0]


    def __len__(
        self,
    ):

        return len(
            self.indices
        )


    def __getitem__(
        self,
        item,
    ):

        sample_index = int(
            self.indices[item]
        )

        # ====================================================
        # Raw arrays
        # ====================================================

        history_surface = (
            self.samples[
                "history_surface"
            ][sample_index]
            .astype(
                np.float32
            )
        )
        # [4,41,10]


        history_mask = (
            self.samples[
                "history_mask"
            ][sample_index]
            .astype(
                np.float32
            )
        )
        # [4,41,10]


        scalar_features = (
            self.samples[
                "scalars"
            ][sample_index]
            .astype(
                np.float32
            )
        )
        # [4]


        target = (
            self.samples[
                "target"
            ][sample_index]
            .astype(
                np.float32
            )
        )
        # [41,10]


        transition_mask = (
            self.samples[
                "transition_mask"
            ][sample_index]
            .astype(
                np.float32
            )
        )
        # [41,10]


        # ====================================================
        # 3. Normalize history surfaces
        #
        # unsupported positions remain exactly zero
        # ====================================================

        surface_mean = (
            self.normalizers[
                "surface_mean"
            ]
        )

        surface_std = (
            self.normalizers[
                "surface_std"
            ]
        )

        normalized_history = (
            (
                history_surface
                - surface_mean[
                    None,
                    :,
                    :,
                ]
            )
            / surface_std[
                None,
                :,
                :,
            ]
        )

        normalized_history = (
            normalized_history
            * history_mask
        )


        # ====================================================
        # 4. Normalize scalar features
        # ====================================================

        normalized_scalars = (
            (
                scalar_features
                - self.normalizers[
                    "scalar_mean"
                ]
            )
            / self.normalizers[
                "scalar_std"
            ]
        )

        # broadcast:
        # [4] -> [4,41,10]

        scalar_channels = np.broadcast_to(
            normalized_scalars[
                :,
                None,
                None,
            ],
            (
                4,
                N_K,
                N_TAU,
            ),
        ).astype(
            np.float32
        )


        # ====================================================
        # 5. Normalize target innovation
        #
        # unsupported transition points remain zero
        # ====================================================

        normalized_target = (
            (
                target
                - self.normalizers[
                    "target_mean"
                ]
            )
            / self.normalizers[
                "target_std"
            ]
        )

        normalized_target = (
            normalized_target
            * transition_mask
        )


        # ====================================================
        # 6. Assemble CNN input
        #
        # 4 history surfaces
        # 4 history masks
        # 4 scalar channels
        # 1 k coordinate
        # 1 tau coordinate
        #
        # total = 14 channels
        # ====================================================

        model_input = np.concatenate(
            [
                # Channels 0-3
                normalized_history,

                # Channels 4-7
                history_mask,

                # Channels 8-11
                scalar_channels,

                # Channel 12
                k_channel[
                    None,
                    :,
                    :,
                ],

                # Channel 13
                tau_channel[
                    None,
                    :,
                    :,
                ],
            ],
            axis=0,
        ).astype(
            np.float32
        )


        # ====================================================
        # 7. Convert to torch tensors
        # ====================================================

        model_input_tensor = (
            torch.from_numpy(
                model_input
            )
        )


        target_tensor = (
            torch.from_numpy(
                normalized_target[
                    None,
                    :,
                    :,
                ]
            )
        )


        transition_mask_tensor = (
            torch.from_numpy(
                transition_mask[
                    None,
                    :,
                    :,
                ]
            )
        )


        sample_index_tensor = (
            torch.tensor(
                sample_index,
                dtype=torch.long,
            )
        )


        return (
            model_input_tensor,
            target_tensor,
            transition_mask_tensor,
            sample_index_tensor,
        )


# ============================================================
# 8. Build datasets
# ============================================================

train_dataset = (
    MaskedSurfaceDataset(
        samples=samples,
        normalizers=normalizers,
        split="train",
    )
)

validation_dataset = (
    MaskedSurfaceDataset(
        samples=samples,
        normalizers=normalizers,
        split="validation",
    )
)

test_dataset = (
    MaskedSurfaceDataset(
        samples=samples,
        normalizers=normalizers,
        split="test",
    )
)


print(
    "\nDataset sizes:"
)

print(
    "train:",
    len(
        train_dataset
    )
)

print(
    "validation:",
    len(
        validation_dataset
    )
)

print(
    "test:",
    len(
        test_dataset
    )
)


# ============================================================
# 9. DataLoaders
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    drop_last=False,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    drop_last=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    drop_last=False,
)


# ============================================================
# 10. Inspect first train batch
# ============================================================

(
    batch_input,
    batch_target,
    batch_transition_mask,
    batch_indices,
) = next(
    iter(
        train_loader
    )
)


print(
    "\nFirst batch shapes:"
)

print(
    "input:",
    batch_input.shape
)

print(
    "target:",
    batch_target.shape
)

print(
    "transition mask:",
    batch_transition_mask.shape
)

print(
    "indices:",
    batch_indices.shape
)


# ============================================================
# 11. Number of input channels
# ============================================================

print(
    "\nNumber of CNN input channels:"
)

print(
    batch_input.shape[1]
)

assert (
    batch_input.shape[1]
    == 14
)


# ============================================================
# 12. Check transition mask
# ============================================================

print(
    "\nTransition mask unique values:"
)

print(
    torch.unique(
        batch_transition_mask
    )
)


# ============================================================
# 13. Unsupported target should be zero
# ============================================================

unsupported_target = (
    batch_target[
        batch_transition_mask
        == 0
    ]
)

if (
    unsupported_target.numel()
    > 0
):

    max_unsupported_target = (
        unsupported_target
        .abs()
        .max()
        .item()
    )

else:

    max_unsupported_target = 0.0


print(
    "\nMax |normalized target| "
    "where transition_mask=0:"
)

print(
    max_unsupported_target
)

assert (
    max_unsupported_target
    < 1e-7
)


# ============================================================
# 14. Unsupported history should be zero
# ============================================================

history_surface_batch = (
    batch_input[
        :,
        0:4,
        :,
        :,
    ]
)

history_mask_batch = (
    batch_input[
        :,
        4:8,
        :,
        :,
    ]
)

unsupported_history = (
    history_surface_batch[
        history_mask_batch
        == 0
    ]
)

if (
    unsupported_history.numel()
    > 0
):

    max_unsupported_history = (
        unsupported_history
        .abs()
        .max()
        .item()
    )

else:

    max_unsupported_history = 0.0


print(
    "\nMax |normalized history| "
    "where history_mask=0:"
)

print(
    max_unsupported_history
)

assert (
    max_unsupported_history
    < 1e-7
)


# ============================================================
# 15. Check transition support in first batch
# ============================================================

batch_transition_support = (
    batch_transition_mask
    .mean(
        dim=(1, 2, 3)
    )
)


print(
    "\nFirst batch transition support:"
)

print(
    "min:",
    float(
        batch_transition_support.min()
    )
)

print(
    "mean:",
    float(
        batch_transition_support.mean()
    )
)

print(
    "max:",
    float(
        batch_transition_support.max()
    )
)


print(
    "\nBLOCK 15 finished successfully."
)

## 4. Persistence-anchored CNN

The baseline forecast is the maturity roll-down
\(\mathcal{R}_{15}(W_t)\).  The neural network learns only the additional
surface innovation.  Unsupported surface locations are excluded from the loss.


In [ ]:
# ============================================================
# BLOCK 16
# Define Masked CNN
# + masked MSE loss
# + one forward-pass sanity check
# ============================================================


# ============================================================
# 1. Residual CNN block
# ============================================================

class ResidualBlock(
    nn.Module
):

    def __init__(
        self,
        channels: int,
    ):

        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=channels,
            out_channels=channels,
            kernel_size=3,
            padding=1,
            padding_mode="replicate",
        )

        self.norm1 = nn.GroupNorm(
            num_groups=8,
            num_channels=channels,
        )

        self.conv2 = nn.Conv2d(
            in_channels=channels,
            out_channels=channels,
            kernel_size=3,
            padding=1,
            padding_mode="replicate",
        )

        self.norm2 = nn.GroupNorm(
            num_groups=8,
            num_channels=channels,
        )

        self.activation = nn.GELU()


    def forward(
        self,
        x,
    ):

        residual = x

        x = self.conv1(
            x
        )

        x = self.norm1(
            x
        )

        x = self.activation(
            x
        )

        x = self.conv2(
            x
        )

        x = self.norm2(
            x
        )

        x = (
            x
            + residual
        )

        x = self.activation(
            x
        )

        return x


# ============================================================
# 2. Surface mean CNN
#
# Input:
#   [B,14,41,10]
#
# Output:
#   [B,1,41,10]
#
# Output is normalized innovation prediction.
# ============================================================

class SurfaceMeanCNN(
    nn.Module
):

    def __init__(
        self,
        in_channels=14,
        hidden_channels=64,
        num_blocks=4,
    ):

        super().__init__()

        self.input_layer = nn.Sequential(

            nn.Conv2d(
                in_channels=in_channels,
                out_channels=hidden_channels,
                kernel_size=3,
                padding=1,
                padding_mode="replicate",
            ),

            nn.GELU(),
        )


        self.blocks = nn.Sequential(
            *[
                ResidualBlock(
                    hidden_channels
                )
                for _ in range(
                    num_blocks
                )
            ]
        )


        self.output_layer = nn.Conv2d(
            in_channels=hidden_channels,
            out_channels=1,
            kernel_size=1,
        )


    def forward(
        self,
        x,
    ):

        x = self.input_layer(
            x
        )

        x = self.blocks(
            x
        )

        x = self.output_layer(
            x
        )

        return x


# ============================================================
# 3. Masked MSE
#
# Only transition_mask = 1 contributes to loss.
# ============================================================

def masked_mse_loss(
    prediction,
    target,
    mask,
):
    """
    Parameters
    ----------
    prediction:
        [B,1,41,10]

    target:
        [B,1,41,10]

    mask:
        [B,1,41,10]

    Returns
    -------
    scalar loss
    """

    squared_error = (
        prediction
        - target
    ) ** 2


    masked_squared_error = (
        squared_error
        * mask
    )


    valid_count = (
        mask.sum()
    )


    if (
        valid_count.item()
        <= 0
    ):

        raise RuntimeError(
            "Batch contains zero valid "
            "transition-mask points."
        )


    loss = (
        masked_squared_error.sum()
        / valid_count
    )


    return loss


# ============================================================
# 4. Create model
# ============================================================

model = SurfaceMeanCNN(
    in_channels=14,
    hidden_channels=64,
    num_blocks=4,
).to(
    DEVICE
)


print(
    "DEVICE:",
    DEVICE
)


print(
    "\nModel:"
)

print(
    model
)


# ============================================================
# 5. Parameter count
# ============================================================

total_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)


trainable_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
    if parameter.requires_grad
)


print(
    "\nTotal parameters:",
    f"{total_parameters:,}"
)


print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)


# ============================================================
# 6. One forward pass
# ============================================================

batch_input = (
    batch_input.to(
        DEVICE
    )
)

batch_target = (
    batch_target.to(
        DEVICE
    )
)

batch_transition_mask = (
    batch_transition_mask.to(
        DEVICE
    )
)


model.eval()


with torch.no_grad():

    batch_prediction = (
        model(
            batch_input
        )
    )


print(
    "\nPrediction shape:"
)

print(
    batch_prediction.shape
)


assert (
    batch_prediction.shape
    == batch_target.shape
)


# ============================================================
# 7. Compute masked loss
# ============================================================

with torch.no_grad():

    test_loss = (
        masked_mse_loss(
            prediction=(
                batch_prediction
            ),

            target=(
                batch_target
            ),

            mask=(
                batch_transition_mask
            ),
        )
    )


print(
    "\nInitial masked MSE:"
)

print(
    float(
        test_loss.cpu()
    )
)


# ============================================================
# 8. Check prediction finite
# ============================================================

print(
    "\nPrediction finite:"
)

print(
    torch.isfinite(
        batch_prediction
    ).all().item()
)


assert (
    torch.isfinite(
        batch_prediction
    ).all()
)


# ============================================================
# 9. Check valid-point count
# ============================================================

valid_points = int(
    batch_transition_mask
    .sum()
    .item()
)


total_points = (
    batch_transition_mask
    .numel()
)


print(
    "\nValid target grid points in batch:"
)

print(
    valid_points,
    "/",
    total_points
)


print(
    "Batch valid fraction:"
)

print(
    valid_points
    / total_points
)


# ============================================================
# 10. Compare against zero-innovation baseline
#
# In standardized target space:
#
# prediction = 0
#
# is NOT exactly the raw persistence baseline
# because target was standardized around train mean.
#
# This is only a numerical sanity check here.
# Formal persistence evaluation comes later.
# ============================================================

zero_prediction = (
    torch.zeros_like(
        batch_target
    )
)


zero_loss = (
    masked_mse_loss(
        prediction=(
            zero_prediction
        ),

        target=(
            batch_target
        ),

        mask=(
            batch_transition_mask
        ),
    )
)


print(
    "\nZero-prediction masked MSE:"
)

print(
    float(
        zero_loss.cpu()
    )
)


# ============================================================
# 11. Final
# ============================================================

print(
    "\nBLOCK 16 finished successfully."
)

In [ ]:
# ============================================================
# BLOCK 17
# Train masked CNN
# + validation early stopping
# ============================================================


# ============================================================
# 1. Training configuration
# ============================================================

TRAIN_EPOCHS = 80

EARLY_STOPPING_PATIENCE = 12

LEARNING_RATE = 1e-3

WEIGHT_DECAY = 1e-5

GRADIENT_CLIP = 1.0


# ============================================================
# 2. Fresh model
#
# IMPORTANT:
# Re-initialize model here so training always starts cleanly.
# ============================================================

model = SurfaceMeanCNN(
    in_channels=14,
    hidden_channels=64,
    num_blocks=4,
).to(
    DEVICE
)


# ============================================================
# 3. Optimizer
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


# ============================================================
# 4. Learning-rate scheduler
# ============================================================

scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=4,
    )
)


# ============================================================
# 5. One training epoch
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
):

    model.train()


    total_weighted_loss = 0.0

    total_valid_points = 0.0


    for (
        model_input,
        target,
        transition_mask,
        _,
    ) in loader:

        model_input = (
            model_input.to(
                DEVICE
            )
        )

        target = (
            target.to(
                DEVICE
            )
        )

        transition_mask = (
            transition_mask.to(
                DEVICE
            )
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        prediction = (
            model(
                model_input
            )
        )


        loss = (
            masked_mse_loss(
                prediction=prediction,
                target=target,
                mask=transition_mask,
            )
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP,
        )


        optimizer.step()


        valid_points = float(
            transition_mask
            .sum()
            .item()
        )


        total_weighted_loss += (
            float(
                loss.detach().cpu()
            )
            * valid_points
        )


        total_valid_points += (
            valid_points
        )


    epoch_loss = (
        total_weighted_loss
        / max(
            total_valid_points,
            1.0,
        )
    )


    return epoch_loss


# ============================================================
# 6. Validation epoch
# ============================================================

def evaluate_epoch(
    model,
    loader,
):

    model.eval()


    total_weighted_loss = 0.0

    total_valid_points = 0.0


    with torch.no_grad():

        for (
            model_input,
            target,
            transition_mask,
            _,
        ) in loader:

            model_input = (
                model_input.to(
                    DEVICE
                )
            )

            target = (
                target.to(
                    DEVICE
                )
            )

            transition_mask = (
                transition_mask.to(
                    DEVICE
                )
            )


            prediction = (
                model(
                    model_input
                )
            )


            loss = (
                masked_mse_loss(
                    prediction=prediction,
                    target=target,
                    mask=transition_mask,
                )
            )


            valid_points = float(
                transition_mask
                .sum()
                .item()
            )


            total_weighted_loss += (
                float(
                    loss.cpu()
                )
                * valid_points
            )


            total_valid_points += (
                valid_points
            )


    epoch_loss = (
        total_weighted_loss
        / max(
            total_valid_points,
            1.0,
        )
    )


    return epoch_loss


# ============================================================
# 7. Zero-prediction baseline on validation set
#
# Again:
# this is zero in STANDARDIZED innovation space.
# Formal raw persistence evaluation is later.
# ============================================================

def zero_baseline_loss(
    loader,
):

    total_loss = 0.0

    total_valid_points = 0.0


    for (
        _,
        target,
        transition_mask,
        _,
    ) in loader:

        zero_prediction = (
            torch.zeros_like(
                target
            )
        )


        squared_error = (
            (
                zero_prediction
                - target
            ) ** 2
            * transition_mask
        )


        valid_points = float(
            transition_mask
            .sum()
            .item()
        )


        total_loss += float(
            squared_error
            .sum()
            .item()
        )


        total_valid_points += (
            valid_points
        )


    return (
        total_loss
        / max(
            total_valid_points,
            1.0,
        )
    )


validation_zero_loss = (
    zero_baseline_loss(
        validation_loader
    )
)


print(
    "Validation zero-prediction baseline:"
)

print(
    validation_zero_loss
)


# ============================================================
# 8. Training loop
# ============================================================

best_validation_loss = float(
    "inf"
)

best_model_state = None

epochs_without_improvement = 0

training_history = []


for epoch in range(
    1,
    TRAIN_EPOCHS + 1,
):

    train_loss = (
        train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
        )
    )


    validation_loss = (
        evaluate_epoch(
            model=model,
            loader=validation_loader,
        )
    )


    scheduler.step(
        validation_loss
    )


    current_lr = (
        optimizer
        .param_groups[0]["lr"]
    )


    training_history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": (
                validation_loss
            ),
            "learning_rate": (
                current_lr
            ),
        }
    )


    improvement_vs_zero = (
        1.0
        - (
            validation_loss
            / validation_zero_loss
        )
    )


    print(
        f"epoch={epoch:03d} | "
        f"train={train_loss:.6f} | "
        f"val={validation_loss:.6f} | "
        f"zero={validation_zero_loss:.6f} | "
        f"val improvement="
        f"{100 * improvement_vs_zero:.2f}% | "
        f"lr={current_lr:.2e}"
    )


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if (
        validation_loss
        < best_validation_loss
        - 1e-6
    ):

        best_validation_loss = (
            validation_loss
        )


        best_model_state = (
            copy.deepcopy(
                model.state_dict()
            )
        )


        epochs_without_improvement = 0


    else:

        epochs_without_improvement += 1


    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print(
            "\nEarly stopping."
        )

        break


# ============================================================
# 9. Restore best model
# ============================================================

if (
    best_model_state
    is None
):

    raise RuntimeError(
        "No valid best model state "
        "was recorded."
    )


model.load_state_dict(
    best_model_state
)


print(
    "\nBest validation loss:"
)

print(
    best_validation_loss
)


print(
    "Validation zero baseline:"
)

print(
    validation_zero_loss
)


print(
    "Best validation improvement vs zero:"
)

print(
    1.0
    - (
        best_validation_loss
        / validation_zero_loss
    )
)


# ============================================================
# 10. Save training history
# ============================================================

training_history_df = (
    pd.DataFrame(
        training_history
    )
)


display(
    training_history_df.tail(
        20
    )
)


# ============================================================
# 11. Plot training curve
# ============================================================

plt.figure(
    figsize=(8, 5)
)


plt.plot(
    training_history_df[
        "epoch"
    ],
    training_history_df[
        "train_loss"
    ],
    label="train",
)


plt.plot(
    training_history_df[
        "epoch"
    ],
    training_history_df[
        "validation_loss"
    ],
    label="validation",
)


plt.axhline(
    validation_zero_loss,
    linestyle="--",
    label="zero baseline",
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Masked standardized MSE"
)

plt.title(
    "Masked CNN training"
)

plt.legend()

plt.show()


print(
    "\nBLOCK 17 finished successfully."
)

## 5. Evaluation and regime analysis

The learned correction is compared with roll-down persistence in raw total-
variance space.  Diagnostics include MSE/RMSE/MAE, day-level bootstrap tests,
clock-time effects, volatility regimes, and persistence-failure regimes.


In [ ]:
# ============================================================
# BLOCK 18
# Raw-space evaluation:
#
# CNN vs roll-down persistence
#
# Metrics:
#   - masked RMSE
#   - masked MAE
#   - per-sample MSE
#   - per-day average MSE
#   - paired day-level bootstrap
#
# IMPORTANT:
# Evaluation is performed in RAW total-variance space.
# ============================================================


# ============================================================
# 1. Helper:
# predict raw innovation from a DataLoader
# ============================================================

def predict_raw_innovation(
    model,
    loader,
    samples,
    normalizers,
    device,
):

    model.eval()


    all_indices = []

    all_prediction = []


    with torch.no_grad():

        for (
            model_input,
            _,
            _,
            sample_indices,
        ) in loader:

            model_input = (
                model_input.to(
                    device
                )
            )


            prediction_normalized = (
                model(
                    model_input
                )
                .cpu()
                .numpy()[:, 0]
            )
            # [B,41,10]


            # ------------------------------------------------
            # Convert normalized prediction
            # back to RAW innovation space
            # ------------------------------------------------

            prediction_raw = (
                prediction_normalized
                * normalizers[
                    "target_std"
                ][
                    None,
                    :,
                    :,
                ]
                + normalizers[
                    "target_mean"
                ][
                    None,
                    :,
                    :,
                ]
            )


            all_prediction.append(
                prediction_raw.astype(
                    np.float32
                )
            )


            all_indices.append(
                sample_indices.numpy()
            )


    all_prediction = np.concatenate(
        all_prediction,
        axis=0,
    )


    all_indices = np.concatenate(
        all_indices,
        axis=0,
    )


    return (
        all_indices,
        all_prediction,
    )


# ============================================================
# 2. Evaluate one split
# ============================================================

def evaluate_raw_split(
    split_name,
    loader,
    model,
    samples,
    normalizers,
    device,
):

    (
        indices,
        predicted_innovation,
    ) = predict_raw_innovation(
        model=model,
        loader=loader,
        samples=samples,
        normalizers=normalizers,
        device=device,
    )


    # --------------------------------------------------------
    # Ground truth objects
    # --------------------------------------------------------

    true_innovation = (
        samples[
            "target"
        ][
            indices
        ]
        .astype(
            np.float64
        )
    )


    transition_mask = (
        samples[
            "transition_mask"
        ][
            indices
        ]
        .astype(
            np.float64
        )
    )


    rolled_surface = (
        samples[
            "rolled"
        ][
            indices
        ]
        .astype(
            np.float64
        )
    )


    true_future = (
        samples[
            "future"
        ][
            indices
        ]
        .astype(
            np.float64
        )
    )


    predicted_innovation = (
        predicted_innovation.astype(
            np.float64
        )
    )


    # ========================================================
    # 3. Persistence prediction
    #
    # innovation = 0
    # ========================================================

    persistence_innovation = np.zeros_like(
        true_innovation
    )


    # ========================================================
    # 4. Future surface predictions
    # ========================================================

    cnn_future = (
        rolled_surface
        + predicted_innovation
    )


    persistence_future = (
        rolled_surface
    )


    # ========================================================
    # 5. Errors
    # ========================================================

    cnn_innovation_error = (
        predicted_innovation
        - true_innovation
    )


    persistence_innovation_error = (
        persistence_innovation
        - true_innovation
    )


    cnn_future_error = (
        cnn_future
        - true_future
    )


    persistence_future_error = (
        persistence_future
        - true_future
    )


    # ========================================================
    # 6. Global masked metrics
    # ========================================================

    valid_count = float(
        transition_mask.sum()
    )


    if (
        valid_count
        <= 0
    ):

        raise RuntimeError(
            f"No valid transition points "
            f"in split {split_name}"
        )


    cnn_mse = float(
        (
            cnn_future_error**2
            * transition_mask
        ).sum()
        / valid_count
    )


    persistence_mse = float(
        (
            persistence_future_error**2
            * transition_mask
        ).sum()
        / valid_count
    )


    cnn_rmse = float(
        np.sqrt(
            cnn_mse
        )
    )


    persistence_rmse = float(
        np.sqrt(
            persistence_mse
        )
    )


    cnn_mae = float(
        (
            np.abs(
                cnn_future_error
            )
            * transition_mask
        ).sum()
        / valid_count
    )


    persistence_mae = float(
        (
            np.abs(
                persistence_future_error
            )
            * transition_mask
        ).sum()
        / valid_count
    )


    relative_mse_improvement = float(
        1.0
        - (
            cnn_mse
            / persistence_mse
        )
    )


    relative_rmse_improvement = float(
        1.0
        - (
            cnn_rmse
            / persistence_rmse
        )
    )


    # ========================================================
    # 7. Per-sample metrics
    # ========================================================

    sample_rows = []


    for position, sample_index in enumerate(
        indices
    ):

        mask = (
            transition_mask[
                position
            ]
        )


        n_valid = float(
            mask.sum()
        )


        if (
            n_valid
            <= 0
        ):

            continue


        cnn_sample_mse = float(
            (
                cnn_future_error[
                    position
                ] ** 2
                * mask
            ).sum()
            / n_valid
        )


        persistence_sample_mse = float(
            (
                persistence_future_error[
                    position
                ] ** 2
                * mask
            ).sum()
            / n_valid
        )


        cnn_sample_mae = float(
            (
                np.abs(
                    cnn_future_error[
                        position
                    ]
                )
                * mask
            ).sum()
            / n_valid
        )


        persistence_sample_mae = float(
            (
                np.abs(
                    persistence_future_error[
                        position
                    ]
                )
                * mask
            ).sum()
            / n_valid
        )


        sample_rows.append(
            {
                "split": (
                    split_name
                ),

                "sample_index": int(
                    sample_index
                ),

                "trade_date": (
                    samples[
                        "date"
                    ][
                        sample_index
                    ]
                ),

                "timestamp": (
                    samples[
                        "timestamp"
                    ][
                        sample_index
                    ]
                ),

                "clock_time": (
                    samples[
                        "clock_time"
                    ][
                        sample_index
                    ]
                ),

                "transition_support": float(
                    mask.mean()
                ),

                "cnn_mse": (
                    cnn_sample_mse
                ),

                "persistence_mse": (
                    persistence_sample_mse
                ),

                "mse_difference": (
                    cnn_sample_mse
                    - persistence_sample_mse
                ),

                "cnn_mae": (
                    cnn_sample_mae
                ),

                "persistence_mae": (
                    persistence_sample_mae
                ),
            }
        )


    sample_metrics = pd.DataFrame(
        sample_rows
    )


    # ========================================================
    # 8. Summary
    # ========================================================

    summary = {
        "split": (
            split_name
        ),

        "samples": int(
            len(
                sample_metrics
            )
        ),

        "valid_grid_points": int(
            valid_count
        ),

        "cnn_rmse": (
            cnn_rmse
        ),

        "persistence_rmse": (
            persistence_rmse
        ),

        "cnn_mae": (
            cnn_mae
        ),

        "persistence_mae": (
            persistence_mae
        ),

        "relative_mse_improvement": (
            relative_mse_improvement
        ),

        "relative_rmse_improvement": (
            relative_rmse_improvement
        ),
    }


    return (
        summary,
        sample_metrics,
    )


# ============================================================
# 9. Run validation
# ============================================================

(
    validation_raw_summary,
    validation_sample_metrics,
) = evaluate_raw_split(
    split_name="validation",
    loader=validation_loader,
    model=model,
    samples=samples,
    normalizers=normalizers,
    device=DEVICE,
)


print(
    "\nVALIDATION RAW SUMMARY:"
)

print(
    json.dumps(
        validation_raw_summary,
        indent=2,
    )
)


# ============================================================
# 10. Run test
# ============================================================

(
    test_raw_summary,
    test_sample_metrics,
) = evaluate_raw_split(
    split_name="test",
    loader=test_loader,
    model=model,
    samples=samples,
    normalizers=normalizers,
    device=DEVICE,
)


print(
    "\nTEST RAW SUMMARY:"
)

print(
    json.dumps(
        test_raw_summary,
        indent=2,
    )
)


# ============================================================
# 11. Day-level paired comparison
#
# For each trading day:
#
#   mean CNN MSE
#   mean persistence MSE
#
# then compare:
#
#   CNN - persistence
#
# Negative = CNN better
# ============================================================

daily_test = (
    test_sample_metrics
    .groupby(
        "trade_date",
        observed=True,
    )
    .agg(
        cnn_mse=(
            "cnn_mse",
            "mean",
        ),

        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        n_samples=(
            "sample_index",
            "size",
        ),
    )
    .reset_index()
)


daily_test[
    "mse_difference"
] = (
    daily_test[
        "cnn_mse"
    ]
    - daily_test[
        "persistence_mse"
    ]
)


print(
    "\nDaily test MSE difference summary:"
)

display(
    daily_test[
        "mse_difference"
    ].describe()
)


print(
    "\nFraction of test days where CNN beats persistence:"
)

cnn_win_fraction = float(
    (
        daily_test[
            "mse_difference"
        ]
        < 0
    ).mean()
)


print(
    cnn_win_fraction
)


# ============================================================
# 12. Paired day-level bootstrap
# ============================================================

def bootstrap_daily_difference(
    daily_difference,
    repetitions=5000,
    seed=1234,
):

    values = np.asarray(
        daily_difference,
        dtype=np.float64,
    )


    if (
        len(values)
        == 0
    ):

        raise RuntimeError(
            "No daily differences."
        )


    rng = np.random.default_rng(
        seed
    )


    bootstrap_means = np.empty(
        repetitions,
        dtype=np.float64,
    )


    for repetition in range(
        repetitions
    ):

        sampled = rng.choice(
            values,
            size=len(values),
            replace=True,
        )


        bootstrap_means[
            repetition
        ] = sampled.mean()


    result = {
        "n_days": int(
            len(
                values
            )
        ),

        "mean_difference": float(
            values.mean()
        ),

        "median_difference": float(
            np.median(
                values
            )
        ),

        "ci_2.5": float(
            np.quantile(
                bootstrap_means,
                0.025,
            )
        ),

        "ci_97.5": float(
            np.quantile(
                bootstrap_means,
                0.975,
            )
        ),

        "probability_mean_below_zero": float(
            (
                bootstrap_means
                < 0
            ).mean()
        ),
    }


    return result


bootstrap_result = (
    bootstrap_daily_difference(
        daily_test[
            "mse_difference"
        ].to_numpy()
    )
)


print(
    "\nDAY-LEVEL BOOTSTRAP:"
)

print(
    json.dumps(
        bootstrap_result,
        indent=2,
    )
)


# ============================================================
# 13. Clock-time analysis
#
# Check whether signal exists only at certain times.
# ============================================================

clock_time_metrics = (
    test_sample_metrics
    .groupby(
        "clock_time",
        observed=True,
    )
    .agg(
        cnn_mse=(
            "cnn_mse",
            "mean",
        ),

        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        samples=(
            "sample_index",
            "size",
        ),
    )
    .reset_index()
)


clock_time_metrics[
    "relative_mse_improvement"
] = (
    1.0
    - (
        clock_time_metrics[
            "cnn_mse"
        ]
        / clock_time_metrics[
            "persistence_mse"
        ]
    )
)


print(
    "\nTest results by clock time:"
)

display(
    clock_time_metrics
)


# ============================================================
# 14. Final interpretation
# ============================================================

test_improvement = (
    test_raw_summary[
        "relative_mse_improvement"
    ]
)


ci_upper = (
    bootstrap_result[
        "ci_97.5"
    ]
)


if (
    test_improvement
    > 0
    and ci_upper
    < 0
):

    decision = "GO"


    interpretation = (
        "CNN beats roll-down persistence "
        "on raw-space test MSE, and the "
        "day-level bootstrap confidence interval "
        "for CNN minus persistence is entirely below zero."
    )


elif (
    test_improvement
    > 0
):

    decision = "REVIEW"


    interpretation = (
        "CNN has lower overall raw-space test MSE, "
        "but the day-level bootstrap does not provide "
        "strong enough evidence that the improvement "
        "is stable across trading days."
    )


else:

    decision = "NO_GO_FOR_MEAN"


    interpretation = (
        "CNN does not beat roll-down persistence "
        "on raw-space test MSE. "
        "The current conditional-mean model should "
        "not be made more complicated before "
        "revisiting the forecasting target or "
        "moving toward conditional variance/distribution modeling."
    )


print(
    "\nFINAL DECISION:"
)

print(
    decision
)


print(
    "\nInterpretation:"
)

print(
    interpretation
)


print(
    "\nBLOCK 18 finished successfully."
)

In [ ]:
# ============================================================
# BLOCK 19
# Regime analysis:
#
# 1. RV quintiles
# 2. Absolute SPY 15-min move quintiles
# 3. Persistence-error quintiles
# 4. Joint interpretation
#
# Requires:
#   test_sample_metrics   from Block 18
#   samples               from Block 13
# ============================================================


# ============================================================
# 1. Attach market features to test sample metrics
# ============================================================

regime_df = (
    test_sample_metrics
    .copy()
)


# sample_index maps directly back to Block 13 samples
sample_indices = (
    regime_df[
        "sample_index"
    ]
    .to_numpy(
        dtype=int
    )
)


# ------------------------------------------------------------
# Scalars from Block 13:
#
# column 0 = 15-min SPY return
# column 1 = 1-hour realized volatility
# column 2 = sin(time)
# column 3 = cos(time)
# ------------------------------------------------------------

regime_df[
    "spy_return_15m"
] = (
    samples[
        "scalars"
    ][
        sample_indices,
        0,
    ]
)


regime_df[
    "abs_spy_return_15m"
] = np.abs(
    regime_df[
        "spy_return_15m"
    ]
)


regime_df[
    "rv_1h"
] = (
    samples[
        "scalars"
    ][
        sample_indices,
        1,
    ]
)


# ------------------------------------------------------------
# Relative improvement per sample
#
# > 0 : CNN better
# < 0 : persistence better
# ------------------------------------------------------------

regime_df[
    "relative_mse_improvement"
] = (
    1.0
    - (
        regime_df[
            "cnn_mse"
        ]
        / regime_df[
            "persistence_mse"
        ]
    )
)


# Avoid pathological division if persistence error = 0
regime_df.loc[
    regime_df[
        "persistence_mse"
    ]
    <= 0,
    "relative_mse_improvement",
] = np.nan


# ============================================================
# 2. Helper: robust quantile buckets
# ============================================================

def make_quantile_bucket(
    series,
    q=5,
    prefix="Q",
):
    """
    qcut with duplicate handling.
    """

    ranked = (
        series
        .rank(
            method="first"
        )
    )


    labels = [
        f"{prefix}{i}"
        for i in range(
            1,
            q + 1
        )
    ]


    return pd.qcut(
        ranked,
        q=q,
        labels=labels,
    )


# ============================================================
# 3. RV quintiles
# ============================================================

regime_df[
    "rv_bucket"
] = make_quantile_bucket(
    regime_df[
        "rv_1h"
    ],
    q=5,
    prefix="RV_Q",
)


rv_summary = (
    regime_df
    .groupby(
        "rv_bucket",
        observed=True,
    )
    .agg(
        samples=(
            "sample_index",
            "size",
        ),

        mean_rv=(
            "rv_1h",
            "mean",
        ),

        cnn_mse=(
            "cnn_mse",
            "mean",
        ),

        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        cnn_mae=(
            "cnn_mae",
            "mean",
        ),

        persistence_mae=(
            "persistence_mae",
            "mean",
        ),

        cnn_win_rate=(
            "mse_difference",
            lambda x: float(
                (
                    x < 0
                ).mean()
            ),
        ),
    )
    .reset_index()
)


rv_summary[
    "relative_mse_improvement"
] = (
    1.0
    - (
        rv_summary[
            "cnn_mse"
        ]
        / rv_summary[
            "persistence_mse"
        ]
    )
)


rv_summary[
    "relative_mae_improvement"
] = (
    1.0
    - (
        rv_summary[
            "cnn_mae"
        ]
        / rv_summary[
            "persistence_mae"
        ]
    )
)


print(
    "\n================ RV QUINTILES ================"
)

display(
    rv_summary
)


# ============================================================
# 4. Absolute 15-min SPY move quintiles
# ============================================================

regime_df[
    "move_bucket"
] = make_quantile_bucket(
    regime_df[
        "abs_spy_return_15m"
    ],
    q=5,
    prefix="MOVE_Q",
)


move_summary = (
    regime_df
    .groupby(
        "move_bucket",
        observed=True,
    )
    .agg(
        samples=(
            "sample_index",
            "size",
        ),

        mean_abs_return=(
            "abs_spy_return_15m",
            "mean",
        ),

        cnn_mse=(
            "cnn_mse",
            "mean",
        ),

        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        cnn_mae=(
            "cnn_mae",
            "mean",
        ),

        persistence_mae=(
            "persistence_mae",
            "mean",
        ),

        cnn_win_rate=(
            "mse_difference",
            lambda x: float(
                (
                    x < 0
                ).mean()
            ),
        ),
    )
    .reset_index()
)


move_summary[
    "relative_mse_improvement"
] = (
    1.0
    - (
        move_summary[
            "cnn_mse"
        ]
        / move_summary[
            "persistence_mse"
        ]
    )
)


move_summary[
    "relative_mae_improvement"
] = (
    1.0
    - (
        move_summary[
            "cnn_mae"
        ]
        / move_summary[
            "persistence_mae"
        ]
    )
)


print(
    "\n================ ABS SPY MOVE QUINTILES ================"
)

display(
    move_summary
)


# ============================================================
# 5. Persistence-error quintiles
#
# This tells us:
# when persistence is already good,
# versus when persistence fails badly.
# ============================================================

regime_df[
    "persistence_error_bucket"
] = make_quantile_bucket(
    regime_df[
        "persistence_mse"
    ],
    q=5,
    prefix="PERR_Q",
)


persistence_error_summary = (
    regime_df
    .groupby(
        "persistence_error_bucket",
        observed=True,
    )
    .agg(
        samples=(
            "sample_index",
            "size",
        ),

        mean_persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        cnn_mse=(
            "cnn_mse",
            "mean",
        ),

        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        cnn_mae=(
            "cnn_mae",
            "mean",
        ),

        persistence_mae=(
            "persistence_mae",
            "mean",
        ),

        cnn_win_rate=(
            "mse_difference",
            lambda x: float(
                (
                    x < 0
                ).mean()
            ),
        ),
    )
    .reset_index()
)


persistence_error_summary[
    "relative_mse_improvement"
] = (
    1.0
    - (
        persistence_error_summary[
            "cnn_mse"
        ]
        / persistence_error_summary[
            "persistence_mse"
        ]
    )
)


persistence_error_summary[
    "relative_mae_improvement"
] = (
    1.0
    - (
        persistence_error_summary[
            "cnn_mae"
        ]
        / persistence_error_summary[
            "persistence_mae"
        ]
    )
)


print(
    "\n================ PERSISTENCE ERROR QUINTILES ================"
)

display(
    persistence_error_summary
)


# ============================================================
# 6. Daily regime analysis
#
# Aggregate within each trading day first.
# ============================================================

daily_regime = (
    regime_df
    .groupby(
        "trade_date",
        observed=True,
    )
    .agg(
        mean_rv=(
            "rv_1h",
            "mean",
        ),

        mean_abs_return=(
            "abs_spy_return_15m",
            "mean",
        ),

        cnn_mse=(
            "cnn_mse",
            "mean",
        ),

        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        samples=(
            "sample_index",
            "size",
        ),
    )
    .reset_index()
)


daily_regime[
    "mse_difference"
] = (
    daily_regime[
        "cnn_mse"
    ]
    - daily_regime[
        "persistence_mse"
    ]
)


daily_regime[
    "relative_mse_improvement"
] = (
    1.0
    - (
        daily_regime[
            "cnn_mse"
        ]
        / daily_regime[
            "persistence_mse"
        ]
    )
)


daily_regime[
    "rv_bucket"
] = make_quantile_bucket(
    daily_regime[
        "mean_rv"
    ],
    q=5,
    prefix="DAY_RV_Q",
)


daily_rv_summary = (
    daily_regime
    .groupby(
        "rv_bucket",
        observed=True,
    )
    .agg(
        days=(
            "trade_date",
            "size",
        ),

        mean_rv=(
            "mean_rv",
            "mean",
        ),

        cnn_mse=(
            "cnn_mse",
            "mean",
        ),

        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        cnn_win_rate=(
            "mse_difference",
            lambda x: float(
                (
                    x < 0
                ).mean()
            ),
        ),
    )
    .reset_index()
)


daily_rv_summary[
    "relative_mse_improvement"
] = (
    1.0
    - (
        daily_rv_summary[
            "cnn_mse"
        ]
        / daily_rv_summary[
            "persistence_mse"
        ]
    )
)


print(
    "\n================ DAILY RV QUINTILES ================"
)

display(
    daily_rv_summary
)


# ============================================================
# 7. Correlations
# ============================================================

correlation_table = pd.DataFrame(
    {
        "variable": [
            "rv_1h",
            "abs_spy_return_15m",
            "persistence_mse",
        ],

        "corr_with_mse_difference": [
            regime_df[
                "rv_1h"
            ].corr(
                regime_df[
                    "mse_difference"
                ]
            ),

            regime_df[
                "abs_spy_return_15m"
            ].corr(
                regime_df[
                    "mse_difference"
                ]
            ),

            regime_df[
                "persistence_mse"
            ].corr(
                regime_df[
                    "mse_difference"
                ]
            ),
        ],
    }
)


print(
    "\n================ CORRELATIONS ================"
)

display(
    correlation_table
)


# ============================================================
# 8. Plot RV regime improvement
# ============================================================

plt.figure(
    figsize=(8, 5)
)


plt.plot(
    rv_summary[
        "rv_bucket"
    ].astype(str),

    100.0
    * rv_summary[
        "relative_mse_improvement"
    ],

    marker="o",
)


plt.axhline(
    0.0,
    linestyle="--",
)


plt.xlabel(
    "1-hour RV quintile"
)

plt.ylabel(
    "CNN MSE improvement vs persistence (%)"
)

plt.title(
    "CNN improvement by RV regime"
)

plt.show()


# ============================================================
# 9. Plot SPY-move regime improvement
# ============================================================

plt.figure(
    figsize=(8, 5)
)


plt.plot(
    move_summary[
        "move_bucket"
    ].astype(str),

    100.0
    * move_summary[
        "relative_mse_improvement"
    ],

    marker="o",
)


plt.axhline(
    0.0,
    linestyle="--",
)


plt.xlabel(
    "|SPY 15-min return| quintile"
)

plt.ylabel(
    "CNN MSE improvement vs persistence (%)"
)

plt.title(
    "CNN improvement by SPY move regime"
)

plt.show()


# ============================================================
# 10. Plot persistence-error regime
# ============================================================

plt.figure(
    figsize=(8, 5)
)


plt.plot(
    persistence_error_summary[
        "persistence_error_bucket"
    ].astype(str),

    100.0
    * persistence_error_summary[
        "relative_mse_improvement"
    ],

    marker="o",
)


plt.axhline(
    0.0,
    linestyle="--",
)


plt.xlabel(
    "Persistence error quintile"
)

plt.ylabel(
    "CNN MSE improvement vs persistence (%)"
)

plt.title(
    "CNN improvement when persistence is weak/strong"
)

plt.show()


# ============================================================
# 11. Final concise diagnostics
# ============================================================

print(
    "\n================ KEY NUMBERS ================"
)


print(
    "Lowest RV quintile improvement:",
    float(
        rv_summary[
            "relative_mse_improvement"
        ].iloc[0]
    )
)


print(
    "Highest RV quintile improvement:",
    float(
        rv_summary[
            "relative_mse_improvement"
        ].iloc[-1]
    )
)


print(
    "Smallest SPY-move quintile improvement:",
    float(
        move_summary[
            "relative_mse_improvement"
        ].iloc[0]
    )
)


print(
    "Largest SPY-move quintile improvement:",
    float(
        move_summary[
            "relative_mse_improvement"
        ].iloc[-1]
    )
)


print(
    "Lowest persistence-error quintile improvement:",
    float(
        persistence_error_summary[
            "relative_mse_improvement"
        ].iloc[0]
    )
)


print(
    "Highest persistence-error quintile improvement:",
    float(
        persistence_error_summary[
            "relative_mse_improvement"
        ].iloc[-1]
    )
)


print(
    "\nBLOCK 19 finished successfully."
)

In [ ]:
# ============================================================
# BLOCK 20
# Persistence-failure prediction
#
# Goal:
#   Predict whether persistence will fail badly at t+15
#   using ONLY information available at time t.
#
# Label:
#   failure = 1 if persistence MSE is above
#   the 80th percentile of TRAIN persistence MSE
#
# Baseline classifier:
#   Logistic Regression
#
# Features:
#   - 15-min SPY return
#   - 1h RV
#   - time-of-day sin/cos
#   - recent surface change magnitude
#   - recent mask overlap
#
# IMPORTANT:
#   NO future information is allowed in features.
# ============================================================


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.preprocessing import StandardScaler


# ============================================================
# 1. Build persistence MSE for ALL samples
#
# Persistence prediction:
#   future_hat = rolled
#
# Therefore:
#   persistence error = future - rolled
#
# This is used ONLY to create labels.
# ============================================================

transition_mask_all = (
    samples[
        "transition_mask"
    ].astype(
        np.float64
    )
)


future_all = (
    samples[
        "future"
    ].astype(
        np.float64
    )
)


rolled_all = (
    samples[
        "rolled"
    ].astype(
        np.float64
    )
)


persistence_error_all = (
    future_all
    - rolled_all
)


valid_counts = (
    transition_mask_all
    .sum(
        axis=(1, 2)
    )
)


persistence_mse_all = (
    (
        persistence_error_all**2
        * transition_mask_all
    )
    .sum(
        axis=(1, 2)
    )
    / np.maximum(
        valid_counts,
        1.0,
    )
)


print(
    "Persistence MSE shape:",
    persistence_mse_all.shape
)


# ============================================================
# 2. TRAIN-only failure threshold
# ============================================================

train_indices_all = np.where(
    samples[
        "split"
    ]
    == "train"
)[0]


train_persistence_mse = (
    persistence_mse_all[
        train_indices_all
    ]
)


FAILURE_QUANTILE = 0.80


failure_threshold = float(
    np.quantile(
        train_persistence_mse,
        FAILURE_QUANTILE,
    )
)


print(
    "\nFailure quantile:",
    FAILURE_QUANTILE
)


print(
    "TRAIN-only persistence MSE threshold:"
)

print(
    failure_threshold
)


# ============================================================
# 3. Create failure labels
# ============================================================

failure_label = (
    persistence_mse_all
    > failure_threshold
).astype(
    np.int64
)


print(
    "\nFailure rate overall:"
)

print(
    failure_label.mean()
)


for split_name in [
    "train",
    "validation",
    "test",
]:

    split_mask = (
        samples[
            "split"
        ]
        == split_name
    )

    print(
        split_name,
        "failure rate =",
        failure_label[
            split_mask
        ].mean()
    )


# ============================================================
# 4. Build CURRENT-information-only features
# ============================================================

def build_failure_features(
    samples,
):
    """
    Create one feature vector per sample.

    All features are available at current time t.

    No future surface or future mask is used.
    """

    history_surface = (
        samples[
            "history_surface"
        ].astype(
            np.float64
        )
    )
    # [N,4,41,10]


    history_mask = (
        samples[
            "history_mask"
        ].astype(
            np.float64
        )
    )


    scalars = (
        samples[
            "scalars"
        ].astype(
            np.float64
        )
    )


    # --------------------------------------------------------
    # A. Existing scalar features
    #
    # 0 = return_15m
    # 1 = rv_1h
    # 2 = sin_time
    # 3 = cos_time
    # --------------------------------------------------------

    return_15m = (
        scalars[:, 0]
    )


    abs_return_15m = np.abs(
        return_15m
    )


    rv_1h = (
        scalars[:, 1]
    )


    sin_time = (
        scalars[:, 2]
    )


    cos_time = (
        scalars[:, 3]
    )


    # --------------------------------------------------------
    # B. Recent surface change:
    #
    # W_t - W_{t-15}
    #
    # only where both history masks are valid
    # --------------------------------------------------------

    previous_surface = (
        history_surface[
            :,
            -2,
            :,
            :,
        ]
    )


    current_surface = (
        history_surface[
            :,
            -1,
            :,
            :,
        ]
    )


    previous_mask = (
        history_mask[
            :,
            -2,
            :,
            :,
        ]
    )


    current_mask = (
        history_mask[
            :,
            -1,
            :,
            :,
        ]
    )


    common_mask = (
        (
            previous_mask
            > 0
        )
        &
        (
            current_mask
            > 0
        )
    ).astype(
        np.float64
    )


    common_count = (
        common_mask
        .sum(
            axis=(1, 2)
        )
    )


    recent_surface_change = (
        current_surface
        - previous_surface
    )


    # RMSE-like magnitude
    surface_change_rms = (
        np.sqrt(
            (
                recent_surface_change**2
                * common_mask
            )
            .sum(
                axis=(1, 2)
            )
            / np.maximum(
                common_count,
                1.0,
            )
        )
    )


    # Mean absolute recent surface move
    surface_change_mae = (
        (
            np.abs(
                recent_surface_change
            )
            * common_mask
        )
        .sum(
            axis=(1, 2)
        )
        / np.maximum(
            common_count,
            1.0,
        )
    )


    # --------------------------------------------------------
    # C. Current surface support
    # --------------------------------------------------------

    current_support = (
        current_mask
        .mean(
            axis=(1, 2)
        )
    )


    previous_support = (
        previous_mask
        .mean(
            axis=(1, 2)
        )
    )


    common_support = (
        common_mask
        .mean(
            axis=(1, 2)
        )
    )


    # --------------------------------------------------------
    # D. Surface level summary
    #
    # masked mean current total variance
    # --------------------------------------------------------

    current_mask_count = (
        current_mask
        .sum(
            axis=(1, 2)
        )
    )


    current_surface_mean = (
        (
            current_surface
            * current_mask
        )
        .sum(
            axis=(1, 2)
        )
        / np.maximum(
            current_mask_count,
            1.0,
        )
    )


    # --------------------------------------------------------
    # E. Assemble features
    # --------------------------------------------------------

    feature_names = [
        "return_15m",
        "abs_return_15m",
        "rv_1h",
        "sin_time",
        "cos_time",
        "surface_change_rms",
        "surface_change_mae",
        "current_support",
        "previous_support",
        "common_support",
        "current_surface_mean",
    ]


    X = np.column_stack(
        [
            return_15m,
            abs_return_15m,
            rv_1h,
            sin_time,
            cos_time,
            surface_change_rms,
            surface_change_mae,
            current_support,
            previous_support,
            common_support,
            current_surface_mean,
        ]
    )


    return (
        X,
        feature_names,
    )


X_all, feature_names = (
    build_failure_features(
        samples
    )
)


print(
    "\nFeature matrix shape:"
)

print(
    X_all.shape
)


print(
    "\nFeature names:"
)

print(
    feature_names
)


# ============================================================
# 5. Remove any non-finite rows
# ============================================================

finite_rows = (
    np.isfinite(
        X_all
    ).all(
        axis=1
    )
)


print(
    "\nFinite feature rows:"
)

print(
    finite_rows.sum(),
    "/",
    len(
        finite_rows
    )
)


# ============================================================
# 6. Split
# ============================================================

train_mask_failure = (
    (
        samples[
            "split"
        ]
        == "train"
    )
    &
    finite_rows
)


validation_mask_failure = (
    (
        samples[
            "split"
        ]
        == "validation"
    )
    &
    finite_rows
)


test_mask_failure = (
    (
        samples[
            "split"
        ]
        == "test"
    )
    &
    finite_rows
)


X_train = (
    X_all[
        train_mask_failure
    ]
)

y_train = (
    failure_label[
        train_mask_failure
    ]
)


X_validation = (
    X_all[
        validation_mask_failure
    ]
)

y_validation = (
    failure_label[
        validation_mask_failure
    ]
)


X_test = (
    X_all[
        test_mask_failure
    ]
)

y_test = (
    failure_label[
        test_mask_failure
    ]
)


print(
    "\nFailure classifier split sizes:"
)

print(
    "train:",
    len(
        X_train
    )
)

print(
    "validation:",
    len(
        X_validation
    )
)

print(
    "test:",
    len(
        X_test
    )
)


# ============================================================
# 7. Standardize features using TRAIN only
# ============================================================

feature_scaler = (
    StandardScaler()
)


X_train_scaled = (
    feature_scaler
    .fit_transform(
        X_train
    )
)


X_validation_scaled = (
    feature_scaler
    .transform(
        X_validation
    )
)


X_test_scaled = (
    feature_scaler
    .transform(
        X_test
    )
)


# ============================================================
# 8. Logistic Regression
#
# class_weight balanced because failures are minority class
# ============================================================

failure_classifier = (
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=1234,
    )
)


failure_classifier.fit(
    X_train_scaled,
    y_train,
)


# ============================================================
# 9. Evaluation helper
# ============================================================

def evaluate_failure_classifier(
    model,
    X,
    y,
    split_name,
):

    probability = (
        model.predict_proba(
            X
        )[:, 1]
    )


    prediction = (
        probability
        >= 0.5
    ).astype(
        int
    )


    metrics = {
        "split": (
            split_name
        ),

        "failure_rate": float(
            y.mean()
        ),

        "roc_auc": float(
            roc_auc_score(
                y,
                probability,
            )
        ),

        "average_precision": float(
            average_precision_score(
                y,
                probability,
            )
        ),

        "accuracy": float(
            accuracy_score(
                y,
                prediction,
            )
        ),

        "precision": float(
            precision_score(
                y,
                prediction,
                zero_division=0,
            )
        ),

        "recall": float(
            recall_score(
                y,
                prediction,
                zero_division=0,
            )
        ),

        "f1": float(
            f1_score(
                y,
                prediction,
                zero_division=0,
            )
        ),
    }


    return (
        metrics,
        probability,
        prediction,
    )


# ============================================================
# 10. Validation evaluation
# ============================================================

(
    validation_failure_metrics,
    validation_failure_probability,
    validation_failure_prediction,
) = evaluate_failure_classifier(
    model=(
        failure_classifier
    ),
    X=(
        X_validation_scaled
    ),
    y=(
        y_validation
    ),
    split_name=(
        "validation"
    ),
)


print(
    "\nVALIDATION FAILURE CLASSIFIER:"
)

print(
    json.dumps(
        validation_failure_metrics,
        indent=2,
    )
)


print(
    "\nValidation confusion matrix:"
)

print(
    confusion_matrix(
        y_validation,
        validation_failure_prediction,
    )
)


# ============================================================
# 11. Test evaluation
# ============================================================

(
    test_failure_metrics,
    test_failure_probability,
    test_failure_prediction,
) = evaluate_failure_classifier(
    model=(
        failure_classifier
    ),
    X=(
        X_test_scaled
    ),
    y=(
        y_test
    ),
    split_name=(
        "test"
    ),
)


print(
    "\nTEST FAILURE CLASSIFIER:"
)

print(
    json.dumps(
        test_failure_metrics,
        indent=2,
    )
)


print(
    "\nTest confusion matrix:"
)

print(
    confusion_matrix(
        y_test,
        test_failure_prediction,
    )
)


# ============================================================
# 12. Coefficients
# ============================================================

coefficient_table = pd.DataFrame(
    {
        "feature": (
            feature_names
        ),

        "coefficient": (
            failure_classifier
            .coef_[0]
        ),
    }
)


coefficient_table[
    "abs_coefficient"
] = (
    coefficient_table[
        "coefficient"
    ]
    .abs()
)


coefficient_table = (
    coefficient_table
    .sort_values(
        "abs_coefficient",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


print(
    "\nFailure classifier coefficients:"
)

display(
    coefficient_table
)


# ============================================================
# 13. Probability bucket diagnostic on TEST
#
# If classifier is useful:
# predicted failure probability should be monotonic with
# realized failure rate.
# ============================================================

test_failure_df = pd.DataFrame(
    {
        "probability": (
            test_failure_probability
        ),

        "actual_failure": (
            y_test
        ),
    }
)


test_failure_df[
    "probability_bucket"
] = pd.qcut(
    test_failure_df[
        "probability"
    ]
    .rank(
        method="first"
    ),

    q=5,

    labels=[
        "P_Q1",
        "P_Q2",
        "P_Q3",
        "P_Q4",
        "P_Q5",
    ],
)


probability_bucket_summary = (
    test_failure_df
    .groupby(
        "probability_bucket",
        observed=True,
    )
    .agg(
        samples=(
            "actual_failure",
            "size",
        ),

        mean_predicted_probability=(
            "probability",
            "mean",
        ),

        actual_failure_rate=(
            "actual_failure",
            "mean",
        ),
    )
    .reset_index()
)


print(
    "\nTEST probability buckets:"
)

display(
    probability_bucket_summary
)


print(
    "\nBLOCK 20 finished successfully."
)

In [ ]:
# ============================================================
# BLOCK 21
# Gated CNN evaluation
#
# Compare:
#   1. Persistence
#   2. Raw CNN
#   3. Hard-gated CNN
#   4. Soft-gated CNN
#
# IMPORTANT:
#   - Gate hyperparameters are selected on VALIDATION only.
#   - TEST is untouched until final evaluation.
# ============================================================


# ============================================================
# 1. Helper:
# get CNN raw innovation predictions for arbitrary loader
# ============================================================

def get_raw_cnn_predictions(
    model,
    loader,
    normalizers,
    device,
):
    model.eval()

    all_indices = []
    all_predictions = []

    with torch.no_grad():

        for (
            model_input,
            _,
            _,
            sample_indices,
        ) in loader:

            model_input = (
                model_input.to(
                    device
                )
            )

            pred_normalized = (
                model(
                    model_input
                )
                .cpu()
                .numpy()[:, 0]
            )

            pred_raw = (
                pred_normalized
                * normalizers[
                    "target_std"
                ][
                    None,
                    :,
                    :,
                ]
                + normalizers[
                    "target_mean"
                ][
                    None,
                    :,
                    :,
                ]
            )

            all_predictions.append(
                pred_raw.astype(
                    np.float64
                )
            )

            all_indices.append(
                sample_indices.numpy()
            )


    return (
        np.concatenate(
            all_indices,
            axis=0,
        ),
        np.concatenate(
            all_predictions,
            axis=0,
        ),
    )


# ============================================================
# 2. Get validation/test CNN predictions
# ============================================================

(
    val_indices_gate,
    val_cnn_innovation,
) = get_raw_cnn_predictions(
    model=model,
    loader=validation_loader,
    normalizers=normalizers,
    device=DEVICE,
)


(
    test_indices_gate,
    test_cnn_innovation,
) = get_raw_cnn_predictions(
    model=model,
    loader=test_loader,
    normalizers=normalizers,
    device=DEVICE,
)


# ============================================================
# 3. Build failure-feature matrix for selected indices
#
# X_all / feature_scaler / failure_classifier
# already come from Block 20
# ============================================================

val_X_gate = (
    feature_scaler
    .transform(
        X_all[
            val_indices_gate
        ]
    )
)


test_X_gate = (
    feature_scaler
    .transform(
        X_all[
            test_indices_gate
        ]
    )
)


val_failure_probability = (
    failure_classifier
    .predict_proba(
        val_X_gate
    )[:, 1]
)


test_failure_probability = (
    failure_classifier
    .predict_proba(
        test_X_gate
    )[:, 1]
)


print(
    "Validation probability range:",
    float(
        val_failure_probability.min()
    ),
    "->",
    float(
        val_failure_probability.max()
    )
)


print(
    "Test probability range:",
    float(
        test_failure_probability.min()
    ),
    "->",
    float(
        test_failure_probability.max()
    )
)


# ============================================================
# 4. Generic raw evaluation helper
# ============================================================

def evaluate_innovation_prediction(
    indices,
    predicted_innovation,
    samples,
):

    true_future = (
        samples[
            "future"
        ][
            indices
        ]
        .astype(
            np.float64
        )
    )


    rolled = (
        samples[
            "rolled"
        ][
            indices
        ]
        .astype(
            np.float64
        )
    )


    transition_mask = (
        samples[
            "transition_mask"
        ][
            indices
        ]
        .astype(
            np.float64
        )
    )


    predicted_future = (
        rolled
        + predicted_innovation
    )


    error = (
        predicted_future
        - true_future
    )


    valid_count = float(
        transition_mask.sum()
    )


    mse = float(
        (
            error**2
            * transition_mask
        ).sum()
        / valid_count
    )


    mae = float(
        (
            np.abs(
                error
            )
            * transition_mask
        ).sum()
        / valid_count
    )


    # --------------------------------------------------------
    # Per-sample MSE
    # --------------------------------------------------------

    valid_per_sample = (
        transition_mask
        .sum(
            axis=(1, 2)
        )
    )


    sample_mse = (
        (
            error**2
            * transition_mask
        )
        .sum(
            axis=(1, 2)
        )
        / np.maximum(
            valid_per_sample,
            1.0,
        )
    )


    return {
        "mse": (
            mse
        ),

        "rmse": float(
            np.sqrt(
                mse
            )
        ),

        "mae": (
            mae
        ),

        "sample_mse": (
            sample_mse
        ),
    }


# ============================================================
# 5. Persistence/raw-CNN validation baselines
# ============================================================

val_zero_innovation = np.zeros_like(
    val_cnn_innovation
)


val_persistence_metrics = (
    evaluate_innovation_prediction(
        indices=(
            val_indices_gate
        ),

        predicted_innovation=(
            val_zero_innovation
        ),

        samples=(
            samples
        ),
    )
)


val_raw_cnn_metrics = (
    evaluate_innovation_prediction(
        indices=(
            val_indices_gate
        ),

        predicted_innovation=(
            val_cnn_innovation
        ),

        samples=(
            samples
        ),
    )
)


print(
    "\nVALIDATION BASELINES:"
)


print(
    "Persistence MSE:",
    val_persistence_metrics[
        "mse"
    ]
)


print(
    "Raw CNN MSE:",
    val_raw_cnn_metrics[
        "mse"
    ]
)


print(
    "Persistence MAE:",
    val_persistence_metrics[
        "mae"
    ]
)


print(
    "Raw CNN MAE:",
    val_raw_cnn_metrics[
        "mae"
    ]
)


# ============================================================
# 6. HARD GATE threshold search on VALIDATION
# ============================================================

hard_thresholds = np.arange(
    0.10,
    0.91,
    0.05,
)


hard_rows = []


for threshold in hard_thresholds:

    gate = (
        val_failure_probability
        >= threshold
    ).astype(
        np.float64
    )


    gated_innovation = (
        val_cnn_innovation
        * gate[
            :,
            None,
            None,
        ]
    )


    metrics = (
        evaluate_innovation_prediction(
            indices=(
                val_indices_gate
            ),

            predicted_innovation=(
                gated_innovation
            ),

            samples=(
                samples
            ),
        )
    )


    hard_rows.append(
        {
            "threshold": float(
                threshold
            ),

            "cnn_used_fraction": float(
                gate.mean()
            ),

            "mse": (
                metrics[
                    "mse"
                ]
            ),

            "rmse": (
                metrics[
                    "rmse"
                ]
            ),

            "mae": (
                metrics[
                    "mae"
                ]
            ),

            "mse_improvement_vs_persistence": float(
                1.0
                - (
                    metrics[
                        "mse"
                    ]
                    / val_persistence_metrics[
                        "mse"
                    ]
                )
            ),

            "mae_improvement_vs_persistence": float(
                1.0
                - (
                    metrics[
                        "mae"
                    ]
                    / val_persistence_metrics[
                        "mae"
                    ]
                )
            ),
        }
    )


hard_validation_table = (
    pd.DataFrame(
        hard_rows
    )
)


print(
    "\nHARD-GATE VALIDATION SEARCH:"
)

display(
    hard_validation_table
)


# ============================================================
# 7. Select hard threshold
#
# Primary criterion:
# lowest validation MSE
#
# Tie-breaker:
# lower MAE
# ============================================================

best_hard_row = (
    hard_validation_table
    .sort_values(
        [
            "mse",
            "mae",
        ]
    )
    .iloc[0]
)


BEST_HARD_THRESHOLD = float(
    best_hard_row[
        "threshold"
    ]
)


print(
    "\nBEST HARD THRESHOLD:"
)

print(
    BEST_HARD_THRESHOLD
)


print(
    "Validation CNN-use fraction:"
)

print(
    float(
        best_hard_row[
            "cnn_used_fraction"
        ]
    )
)


print(
    "Validation MSE improvement:"
)

print(
    float(
        best_hard_row[
            "mse_improvement_vs_persistence"
        ]
    )
)


print(
    "Validation MAE improvement:"
)

print(
    float(
        best_hard_row[
            "mae_improvement_vs_persistence"
        ]
    )
)


# ============================================================
# 8. SOFT GATE alpha search on VALIDATION
#
# innovation_hat =
#
#   p_failure ^ alpha
#   *
#   CNN correction
# ============================================================

soft_alphas = [
    0.25,
    0.50,
    0.75,
    1.00,
    1.50,
    2.00,
    3.00,
    4.00,
]


soft_rows = []


for alpha in soft_alphas:

    gate = (
        val_failure_probability
        ** alpha
    )


    gated_innovation = (
        val_cnn_innovation
        * gate[
            :,
            None,
            None,
        ]
    )


    metrics = (
        evaluate_innovation_prediction(
            indices=(
                val_indices_gate
            ),

            predicted_innovation=(
                gated_innovation
            ),

            samples=(
                samples
            ),
        )
    )


    soft_rows.append(
        {
            "alpha": float(
                alpha
            ),

            "mean_gate": float(
                gate.mean()
            ),

            "mse": (
                metrics[
                    "mse"
                ]
            ),

            "rmse": (
                metrics[
                    "rmse"
                ]
            ),

            "mae": (
                metrics[
                    "mae"
                ]
            ),

            "mse_improvement_vs_persistence": float(
                1.0
                - (
                    metrics[
                        "mse"
                    ]
                    / val_persistence_metrics[
                        "mse"
                    ]
                )
            ),

            "mae_improvement_vs_persistence": float(
                1.0
                - (
                    metrics[
                        "mae"
                    ]
                    / val_persistence_metrics[
                        "mae"
                    ]
                )
            ),
        }
    )


soft_validation_table = (
    pd.DataFrame(
        soft_rows
    )
)


print(
    "\nSOFT-GATE VALIDATION SEARCH:"
)

display(
    soft_validation_table
)


best_soft_row = (
    soft_validation_table
    .sort_values(
        [
            "mse",
            "mae",
        ]
    )
    .iloc[0]
)


BEST_SOFT_ALPHA = float(
    best_soft_row[
        "alpha"
    ]
)


print(
    "\nBEST SOFT ALPHA:"
)

print(
    BEST_SOFT_ALPHA
)


print(
    "Validation mean gate:"
)

print(
    float(
        best_soft_row[
            "mean_gate"
        ]
    )
)


print(
    "Validation MSE improvement:"
)

print(
    float(
        best_soft_row[
            "mse_improvement_vs_persistence"
        ]
    )
)


print(
    "Validation MAE improvement:"
)

print(
    float(
        best_soft_row[
            "mae_improvement_vs_persistence"
        ]
    )
)


# ============================================================
# 9. FINAL TEST predictions
#
# Hyperparameters are now frozen.
# ============================================================

test_zero_innovation = np.zeros_like(
    test_cnn_innovation
)


# ------------------------------------------------------------
# Persistence
# ------------------------------------------------------------

test_persistence = (
    evaluate_innovation_prediction(
        indices=(
            test_indices_gate
        ),

        predicted_innovation=(
            test_zero_innovation
        ),

        samples=(
            samples
        ),
    )
)


# ------------------------------------------------------------
# Raw CNN
# ------------------------------------------------------------

test_raw_cnn = (
    evaluate_innovation_prediction(
        indices=(
            test_indices_gate
        ),

        predicted_innovation=(
            test_cnn_innovation
        ),

        samples=(
            samples
        ),
    )
)


# ------------------------------------------------------------
# Hard gate
# ------------------------------------------------------------

test_hard_gate = (
    test_failure_probability
    >= BEST_HARD_THRESHOLD
).astype(
    np.float64
)


test_hard_innovation = (
    test_cnn_innovation
    * test_hard_gate[
        :,
        None,
        None,
    ]
)


test_hard = (
    evaluate_innovation_prediction(
        indices=(
            test_indices_gate
        ),

        predicted_innovation=(
            test_hard_innovation
        ),

        samples=(
            samples
        ),
    )
)


# ------------------------------------------------------------
# Soft gate
# ------------------------------------------------------------

test_soft_gate = (
    test_failure_probability
    ** BEST_SOFT_ALPHA
)


test_soft_innovation = (
    test_cnn_innovation
    * test_soft_gate[
        :,
        None,
        None,
    ]
)


test_soft = (
    evaluate_innovation_prediction(
        indices=(
            test_indices_gate
        ),

        predicted_innovation=(
            test_soft_innovation
        ),

        samples=(
            samples
        ),
    )
)


# ============================================================
# 10. Test summary
# ============================================================

test_comparison = pd.DataFrame(
    [
        {
            "model": (
                "Persistence"
            ),

            "mse": (
                test_persistence[
                    "mse"
                ]
            ),

            "rmse": (
                test_persistence[
                    "rmse"
                ]
            ),

            "mae": (
                test_persistence[
                    "mae"
                ]
            ),
        },

        {
            "model": (
                "Raw CNN"
            ),

            "mse": (
                test_raw_cnn[
                    "mse"
                ]
            ),

            "rmse": (
                test_raw_cnn[
                    "rmse"
                ]
            ),

            "mae": (
                test_raw_cnn[
                    "mae"
                ]
            ),
        },

        {
            "model": (
                "Hard-gated CNN"
            ),

            "mse": (
                test_hard[
                    "mse"
                ]
            ),

            "rmse": (
                test_hard[
                    "rmse"
                ]
            ),

            "mae": (
                test_hard[
                    "mae"
                ]
            ),
        },

        {
            "model": (
                "Soft-gated CNN"
            ),

            "mse": (
                test_soft[
                    "mse"
                ]
            ),

            "rmse": (
                test_soft[
                    "rmse"
                ]
            ),

            "mae": (
                test_soft[
                    "mae"
                ]
            ),
        },
    ]
)


test_comparison[
    "mse_improvement_vs_persistence"
] = (
    1.0
    - (
        test_comparison[
            "mse"
        ]
        / test_persistence[
            "mse"
        ]
    )
)


test_comparison[
    "mae_improvement_vs_persistence"
] = (
    1.0
    - (
        test_comparison[
            "mae"
        ]
        / test_persistence[
            "mae"
        ]
    )
)


print(
    "\n================ FINAL TEST COMPARISON ================"
)

display(
    test_comparison
)


print(
    "\nHard gate test CNN-use fraction:"
)

print(
    float(
        test_hard_gate.mean()
    )
)


print(
    "Soft gate test mean gate:"
)

print(
    float(
        test_soft_gate.mean()
    )
)


# ============================================================
# 11. Test sample-level comparisons
# ============================================================

test_gate_sample_df = pd.DataFrame(
    {
        "sample_index": (
            test_indices_gate
        ),

        "trade_date": (
            samples[
                "date"
            ][
                test_indices_gate
            ]
        ),

        "clock_time": (
            samples[
                "clock_time"
            ][
                test_indices_gate
            ]
        ),

        "failure_probability": (
            test_failure_probability
        ),

        "hard_gate": (
            test_hard_gate
        ),

        "soft_gate": (
            test_soft_gate
        ),

        "persistence_mse": (
            test_persistence[
                "sample_mse"
            ]
        ),

        "raw_cnn_mse": (
            test_raw_cnn[
                "sample_mse"
            ]
        ),

        "hard_gate_mse": (
            test_hard[
                "sample_mse"
            ]
        ),

        "soft_gate_mse": (
            test_soft[
                "sample_mse"
            ]
        ),
    }
)


# ============================================================
# 12. Win rates
# ============================================================

for model_name, mse_column in [
    (
        "Raw CNN",
        "raw_cnn_mse",
    ),

    (
        "Hard-gated CNN",
        "hard_gate_mse",
    ),

    (
        "Soft-gated CNN",
        "soft_gate_mse",
    ),
]:

    win_rate = float(
        (
            test_gate_sample_df[
                mse_column
            ]
            < test_gate_sample_df[
                "persistence_mse"
            ]
        ).mean()
    )


    print(
        f"{model_name} sample win rate:",
        win_rate
    )


# ============================================================
# 13. Day-level comparison
# ============================================================

daily_gate = (
    test_gate_sample_df
    .groupby(
        "trade_date",
        observed=True,
    )
    .agg(
        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        raw_cnn_mse=(
            "raw_cnn_mse",
            "mean",
        ),

        hard_gate_mse=(
            "hard_gate_mse",
            "mean",
        ),

        soft_gate_mse=(
            "soft_gate_mse",
            "mean",
        ),

        samples=(
            "sample_index",
            "size",
        ),
    )
    .reset_index()
)


for model_name, mse_column in [
    (
        "Raw CNN",
        "raw_cnn_mse",
    ),

    (
        "Hard-gated CNN",
        "hard_gate_mse",
    ),

    (
        "Soft-gated CNN",
        "soft_gate_mse",
    ),
]:

    daily_gate[
        f"{model_name}_difference"
    ] = (
        daily_gate[
            mse_column
        ]
        - daily_gate[
            "persistence_mse"
        ]
    )


    day_win_rate = float(
        (
            daily_gate[
                f"{model_name}_difference"
            ]
            < 0
        ).mean()
    )


    print(
        f"{model_name} day win rate:",
        day_win_rate
    )


# ============================================================
# 14. Bootstrap helper
# ============================================================

def bootstrap_gate_difference(
    differences,
    repetitions=5000,
    seed=1234,
):

    values = np.asarray(
        differences,
        dtype=np.float64,
    )


    rng = np.random.default_rng(
        seed
    )


    bootstrap_means = np.empty(
        repetitions,
        dtype=np.float64,
    )


    for i in range(
        repetitions
    ):

        sampled = rng.choice(
            values,
            size=len(values),
            replace=True,
        )


        bootstrap_means[
            i
        ] = sampled.mean()


    return {
        "mean_difference": float(
            values.mean()
        ),

        "ci_2.5": float(
            np.quantile(
                bootstrap_means,
                0.025,
            )
        ),

        "ci_97.5": float(
            np.quantile(
                bootstrap_means,
                0.975,
            )
        ),

        "prob_mean_below_zero": float(
            (
                bootstrap_means
                < 0
            ).mean()
        ),
    }


# ============================================================
# 15. Day-level bootstrap for all three learned models
# ============================================================

bootstrap_rows = []


for model_name in [
    "Raw CNN",
    "Hard-gated CNN",
    "Soft-gated CNN",
]:

    result = (
        bootstrap_gate_difference(
            daily_gate[
                f"{model_name}_difference"
            ]
        )
    )


    bootstrap_rows.append(
        {
            "model": (
                model_name
            ),

            **result,
        }
    )


bootstrap_table = pd.DataFrame(
    bootstrap_rows
)


print(
    "\n================ DAY-LEVEL BOOTSTRAP ================"
)

display(
    bootstrap_table
)


# ============================================================
# 16. Probability regime evaluation
#
# Does gating actually help especially in low/high
# predicted failure-probability regions?
# ============================================================

test_gate_sample_df[
    "probability_bucket"
] = pd.qcut(
    test_gate_sample_df[
        "failure_probability"
    ]
    .rank(
        method="first"
    ),

    q=5,

    labels=[
        "P_Q1",
        "P_Q2",
        "P_Q3",
        "P_Q4",
        "P_Q5",
    ],
)


probability_regime_summary = (
    test_gate_sample_df
    .groupby(
        "probability_bucket",
        observed=True,
    )
    .agg(
        samples=(
            "sample_index",
            "size",
        ),

        mean_probability=(
            "failure_probability",
            "mean",
        ),

        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        raw_cnn_mse=(
            "raw_cnn_mse",
            "mean",
        ),

        hard_gate_mse=(
            "hard_gate_mse",
            "mean",
        ),

        soft_gate_mse=(
            "soft_gate_mse",
            "mean",
        ),
    )
    .reset_index()
)


for col in [
    "raw_cnn",
    "hard_gate",
    "soft_gate",
]:

    probability_regime_summary[
        f"{col}_improvement"
    ] = (
        1.0
        - (
            probability_regime_summary[
                f"{col}_mse"
            ]
            / probability_regime_summary[
                "persistence_mse"
            ]
        )
    )


print(
    "\n================ TEST BY FAILURE-PROBABILITY BUCKET ================"
)

display(
    probability_regime_summary
)


print(
    "\nBLOCK 21 finished successfully."
)

## 6. Capacity and shrinkage ablations

This section documents the model-selection path:

- V1: larger CNN (304,641 parameters)
- V2: smaller persistence-anchored CNN (41,345 parameters)
- V3: tiny CNN (6,753 parameters)
- validation-selected constant and state-dependent shrinkage

The final deterministic specification keeps V2 and uses a constant correction
strength of 0.765 for parsimony.


In [ ]:
# ============================================================
# BLOCK 22
# Smaller CNN V2
#
# Main changes vs V1:
#
#   hidden channels: 64 -> 32
#   residual blocks: 4 -> 2
#   early-stop patience: 4
#
# Most importantly:
#
# V2 is initialized EXACTLY at raw persistence:
#
#       raw innovation prediction = 0
#
# even though the target is standardized as
#
#       (Y - target_mean) / target_std
#
# ============================================================


# ============================================================
# 0. Preserve V1 model before training V2
# ============================================================

model_v1 = copy.deepcopy(
    model
).to(
    DEVICE
)

model_v1.eval()


V1_PARAMETER_COUNT = sum(
    p.numel()
    for p in model_v1.parameters()
)


print(
    "V1 parameters:",
    f"{V1_PARAMETER_COUNT:,}"
)


# ============================================================
# 1. Smaller residual block
# ============================================================

class ResidualBlockV2(
    nn.Module
):

    def __init__(
        self,
        channels,
    ):

        super().__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            padding_mode="replicate",
        )

        self.norm1 = nn.GroupNorm(
            num_groups=4,
            num_channels=channels,
        )

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            padding_mode="replicate",
        )

        self.norm2 = nn.GroupNorm(
            num_groups=4,
            num_channels=channels,
        )

        self.activation = nn.GELU()


    def forward(
        self,
        x,
    ):

        residual = x

        x = self.conv1(
            x
        )

        x = self.norm1(
            x
        )

        x = self.activation(
            x
        )

        x = self.conv2(
            x
        )

        x = self.norm2(
            x
        )

        x = (
            x
            + residual
        )

        x = self.activation(
            x
        )

        return x


# ============================================================
# 2. Persistence-anchored smaller CNN
# ============================================================

class PersistenceAnchoredCNN(
    nn.Module
):

    def __init__(
        self,
        normalizers,
        in_channels=14,
        hidden_channels=32,
        num_blocks=2,
    ):

        super().__init__()


        # ----------------------------------------------------
        # In standardized target space:
        #
        # raw persistence means:
        #
        #       Y_hat_raw = 0
        #
        # therefore:
        #
        #       Y_hat_std
        #       =
        #       (0 - target_mean) / target_std
        #
        # This varies across the 41x10 grid.
        # ----------------------------------------------------

        persistence_standardized = (
            - normalizers[
                "target_mean"
            ]
            / normalizers[
                "target_std"
            ]
        ).astype(
            np.float32
        )


        self.register_buffer(
            "persistence_baseline",
            torch.from_numpy(
                persistence_standardized[
                    None,
                    None,
                    :,
                    :,
                ]
            ),
        )


        # ----------------------------------------------------
        # Small correction CNN
        # ----------------------------------------------------

        self.input_layer = nn.Sequential(

            nn.Conv2d(
                in_channels,
                hidden_channels,
                kernel_size=3,
                padding=1,
                padding_mode="replicate",
            ),

            nn.GELU(),
        )


        self.blocks = nn.Sequential(
            *[
                ResidualBlockV2(
                    hidden_channels
                )
                for _ in range(
                    num_blocks
                )
            ]
        )


        self.output_layer = nn.Conv2d(
            hidden_channels,
            1,
            kernel_size=1,
        )


        # ----------------------------------------------------
        # CRITICAL:
        # correction starts at exactly zero.
        #
        # Therefore model initially equals persistence.
        # ----------------------------------------------------

        nn.init.zeros_(
            self.output_layer.weight
        )

        nn.init.zeros_(
            self.output_layer.bias
        )


    def forward(
        self,
        x,
    ):

        hidden = self.input_layer(
            x
        )

        hidden = self.blocks(
            hidden
        )

        correction = (
            self.output_layer(
                hidden
            )
        )


        return (
            self.persistence_baseline
            + correction
        )


# ============================================================
# 3. Create V2
# ============================================================

model_v2 = PersistenceAnchoredCNN(
    normalizers=normalizers,
    in_channels=14,
    hidden_channels=32,
    num_blocks=2,
).to(
    DEVICE
)


V2_PARAMETER_COUNT = sum(
    p.numel()
    for p in model_v2.parameters()
)


print(
    "\nV2 parameters:",
    f"{V2_PARAMETER_COUNT:,}"
)


print(
    "Parameter reduction:"
)

print(
    1.0
    - (
        V2_PARAMETER_COUNT
        / V1_PARAMETER_COUNT
    )
)


# ============================================================
# 4. Verify epoch-0 model is EXACT raw persistence
# ============================================================

(
    check_input,
    check_target,
    check_transition_mask,
    _,
) = next(
    iter(
        validation_loader
    )
)


check_input = (
    check_input.to(
        DEVICE
    )
)

check_target = (
    check_target.to(
        DEVICE
    )
)

check_transition_mask = (
    check_transition_mask.to(
        DEVICE
    )
)


model_v2.eval()


with torch.no_grad():

    initial_prediction = (
        model_v2(
            check_input
        )
    )


# ------------------------------------------------------------
# Convert initial standardized prediction back to raw
# ------------------------------------------------------------

initial_prediction_raw = (
    initial_prediction
    .cpu()
    .numpy()[:, 0]
    * normalizers[
        "target_std"
    ][
        None,
        :,
        :,
    ]
    + normalizers[
        "target_mean"
    ][
        None,
        :,
        :,
    ]
)


print(
    "\nEpoch-0 max absolute raw innovation:"
)

print(
    float(
        np.max(
            np.abs(
                initial_prediction_raw
            )
        )
    )
)


assert np.allclose(
    initial_prediction_raw,
    0.0,
    atol=1e-9,
)


print(
    "Epoch-0 V2 equals raw persistence: TRUE"
)


# ============================================================
# 5. Correct standardized-space persistence loss
# ============================================================

def persistence_standardized_loss(
    loader,
    normalizers,
):

    persistence_std = (
        - normalizers[
            "target_mean"
        ]
        / normalizers[
            "target_std"
        ]
    )


    persistence_std = (
        torch.from_numpy(
            persistence_std[
                None,
                None,
                :,
                :,
            ].astype(
                np.float32
            )
        )
    )


    total_error = 0.0

    total_valid = 0.0


    for (
        _,
        target,
        mask,
        _,
    ) in loader:

        batch_baseline = (
            persistence_std.expand(
                target.shape[0],
                -1,
                -1,
                -1,
            )
        )


        squared_error = (
            (
                batch_baseline
                - target
            ) ** 2
            * mask
        )


        total_error += float(
            squared_error
            .sum()
            .item()
        )


        total_valid += float(
            mask
            .sum()
            .item()
        )


    return (
        total_error
        / total_valid
    )


v2_validation_persistence_loss = (
    persistence_standardized_loss(
        validation_loader,
        normalizers,
    )
)


print(
    "\nCorrect validation persistence "
    "loss in standardized space:"
)

print(
    v2_validation_persistence_loss
)


# ============================================================
# 6. V2 training configuration
# ============================================================

V2_EPOCHS = 40

V2_EARLY_STOPPING_PATIENCE = 4

V2_LEARNING_RATE = 1e-3

V2_WEIGHT_DECAY = 1e-5

V2_GRADIENT_CLIP = 1.0


optimizer_v2 = torch.optim.AdamW(
    model_v2.parameters(),
    lr=V2_LEARNING_RATE,
    weight_decay=V2_WEIGHT_DECAY,
)


scheduler_v2 = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer_v2,
        mode="min",
        factor=0.5,
        patience=2,
    )
)


# ============================================================
# 7. Training functions
# ============================================================

def train_one_epoch_v2(
    model,
    loader,
    optimizer,
):

    model.train()

    total_loss = 0.0

    total_valid = 0.0


    for (
        x,
        target,
        mask,
        _,
    ) in loader:

        x = x.to(
            DEVICE
        )

        target = target.to(
            DEVICE
        )

        mask = mask.to(
            DEVICE
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        prediction = model(
            x
        )


        loss = masked_mse_loss(
            prediction,
            target,
            mask,
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            V2_GRADIENT_CLIP,
        )


        optimizer.step()


        valid = float(
            mask.sum().item()
        )


        total_loss += (
            float(
                loss.detach().cpu()
            )
            * valid
        )


        total_valid += (
            valid
        )


    return (
        total_loss
        / total_valid
    )


def evaluate_epoch_v2(
    model,
    loader,
):

    model.eval()

    total_loss = 0.0

    total_valid = 0.0


    with torch.no_grad():

        for (
            x,
            target,
            mask,
            _,
        ) in loader:

            x = x.to(
                DEVICE
            )

            target = target.to(
                DEVICE
            )

            mask = mask.to(
                DEVICE
            )


            prediction = model(
                x
            )


            loss = masked_mse_loss(
                prediction,
                target,
                mask,
            )


            valid = float(
                mask.sum().item()
            )


            total_loss += (
                float(
                    loss.cpu()
                )
                * valid
            )


            total_valid += (
                valid
            )


    return (
        total_loss
        / total_valid
    )


# ============================================================
# 8. Train V2
# ============================================================

best_v2_validation_loss = float(
    "inf"
)

best_v2_state = None

best_v2_epoch = None

v2_wait = 0

v2_history = []


for epoch in range(
    1,
    V2_EPOCHS + 1,
):

    train_loss = (
        train_one_epoch_v2(
            model_v2,
            train_loader,
            optimizer_v2,
        )
    )


    val_loss = (
        evaluate_epoch_v2(
            model_v2,
            validation_loader,
        )
    )


    scheduler_v2.step(
        val_loss
    )


    current_lr = (
        optimizer_v2
        .param_groups[0]["lr"]
    )


    improvement_vs_persistence = (
        1.0
        - (
            val_loss
            / v2_validation_persistence_loss
        )
    )


    v2_history.append(
        {
            "epoch": (
                epoch
            ),

            "train_loss": (
                train_loss
            ),

            "validation_loss": (
                val_loss
            ),

            "improvement_vs_persistence": (
                improvement_vs_persistence
            ),

            "learning_rate": (
                current_lr
            ),
        }
    )


    print(
        f"V2 epoch={epoch:03d} | "
        f"train={train_loss:.6f} | "
        f"val={val_loss:.6f} | "
        f"persistence={v2_validation_persistence_loss:.6f} | "
        f"improvement="
        f"{100 * improvement_vs_persistence:.2f}% | "
        f"lr={current_lr:.2e}"
    )


    if (
        val_loss
        < best_v2_validation_loss
        - 1e-6
    ):

        best_v2_validation_loss = (
            val_loss
        )


        best_v2_epoch = (
            epoch
        )


        best_v2_state = (
            copy.deepcopy(
                model_v2.state_dict()
            )
        )


        v2_wait = 0


    else:

        v2_wait += 1


    if (
        v2_wait
        >= V2_EARLY_STOPPING_PATIENCE
    ):

        print(
            "\nV2 early stopping."
        )

        break


# ============================================================
# 9. Restore best V2
# ============================================================

if (
    best_v2_state
    is None
):

    raise RuntimeError(
        "No V2 best state found."
    )


model_v2.load_state_dict(
    best_v2_state
)


print(
    "\nBest V2 epoch:"
)

print(
    best_v2_epoch
)


print(
    "Best V2 validation loss:"
)

print(
    best_v2_validation_loss
)


print(
    "Best V2 validation improvement "
    "vs persistence:"
)

print(
    1.0
    - (
        best_v2_validation_loss
        / v2_validation_persistence_loss
    )
)


# ============================================================
# 10. RAW validation/test evaluation
#
# Uses Block 18 helper evaluate_raw_split().
# ============================================================

(
    v2_validation_raw_summary,
    v2_validation_sample_metrics,
) = evaluate_raw_split(
    split_name="validation",
    loader=validation_loader,
    model=model_v2,
    samples=samples,
    normalizers=normalizers,
    device=DEVICE,
)


(
    v2_test_raw_summary,
    v2_test_sample_metrics,
) = evaluate_raw_split(
    split_name="test",
    loader=test_loader,
    model=model_v2,
    samples=samples,
    normalizers=normalizers,
    device=DEVICE,
)


print(
    "\nV2 VALIDATION RAW:"
)

print(
    json.dumps(
        v2_validation_raw_summary,
        indent=2,
    )
)


print(
    "\nV2 TEST RAW:"
)

print(
    json.dumps(
        v2_test_raw_summary,
        indent=2,
    )
)


# ============================================================
# 11. Day win rate for V2
# ============================================================

v2_daily_test = (
    v2_test_sample_metrics
    .groupby(
        "trade_date",
        observed=True,
    )
    .agg(
        cnn_mse=(
            "cnn_mse",
            "mean",
        ),

        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        n_samples=(
            "sample_index",
            "size",
        ),
    )
    .reset_index()
)


v2_daily_test[
    "mse_difference"
] = (
    v2_daily_test[
        "cnn_mse"
    ]
    - v2_daily_test[
        "persistence_mse"
    ]
)


v2_day_win_rate = float(
    (
        v2_daily_test[
            "mse_difference"
        ]
        < 0
    ).mean()
)


print(
    "\nV2 test day win rate:"
)

print(
    v2_day_win_rate
)


# ============================================================
# 12. V2 bootstrap
# ============================================================

v2_bootstrap = (
    bootstrap_daily_difference(
        v2_daily_test[
            "mse_difference"
        ].to_numpy()
    )
)


print(
    "\nV2 DAY-LEVEL BOOTSTRAP:"
)

print(
    json.dumps(
        v2_bootstrap,
        indent=2,
    )
)


# ============================================================
# 13. V1 vs V2 comparison
#
# V1 results come from Block 18.
# ============================================================

comparison_v1_v2 = pd.DataFrame(
    [
        {
            "model": (
                "V1 CNN"
            ),

            "parameters": (
                V1_PARAMETER_COUNT
            ),

            "validation_rmse": (
                validation_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "validation_mae": (
                validation_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_rmse": (
                test_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "test_mae": (
                test_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_mse_improvement": (
                test_raw_summary[
                    "relative_mse_improvement"
                ]
            ),

            "test_mae_improvement": (
                1.0
                - (
                    test_raw_summary[
                        "cnn_mae"
                    ]
                    / test_raw_summary[
                        "persistence_mae"
                    ]
                )
            ),

            "test_day_win_rate": (
                cnn_win_fraction
            ),
        },

        {
            "model": (
                "V2 Smaller CNN"
            ),

            "parameters": (
                V2_PARAMETER_COUNT
            ),

            "validation_rmse": (
                v2_validation_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "validation_mae": (
                v2_validation_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_rmse": (
                v2_test_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "test_mae": (
                v2_test_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_mse_improvement": (
                v2_test_raw_summary[
                    "relative_mse_improvement"
                ]
            ),

            "test_mae_improvement": (
                1.0
                - (
                    v2_test_raw_summary[
                        "cnn_mae"
                    ]
                    / v2_test_raw_summary[
                        "persistence_mae"
                    ]
                )
            ),

            "test_day_win_rate": (
                v2_day_win_rate
            ),
        },
    ]
)


print(
    "\n================ V1 vs V2 ================"
)

display(
    comparison_v1_v2
)


# ============================================================
# 14. Training curve
# ============================================================

v2_history_df = pd.DataFrame(
    v2_history
)


plt.figure(
    figsize=(8, 5)
)


plt.plot(
    v2_history_df[
        "epoch"
    ],

    v2_history_df[
        "train_loss"
    ],

    label="V2 train",
)


plt.plot(
    v2_history_df[
        "epoch"
    ],

    v2_history_df[
        "validation_loss"
    ],

    label="V2 validation",
)


plt.axhline(
    v2_validation_persistence_loss,
    linestyle="--",
    label="persistence",
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Masked standardized MSE"
)

plt.title(
    "V2 smaller persistence-anchored CNN"
)

plt.legend()

plt.show()


print(
    "\nBLOCK 22 finished successfully."
)

In [ ]:
# ============================================================
# BLOCK 23
# Tiny CNN V3
#
# Capacity ablation:
#
# V1:
#   64 channels
#   4 residual blocks
#   ~304k params
#
# V2:
#   32 channels
#   2 residual blocks
#   ~41k params
#
# V3:
#   16 channels
#   1 residual block
#   tiny model
#
# V3 is still persistence-anchored:
#
# epoch 0 raw innovation prediction = 0
# ============================================================


# ============================================================
# 0. Preserve V2
# ============================================================

model_v2_saved = copy.deepcopy(
    model_v2
).to(
    DEVICE
)

model_v2_saved.eval()


V2_PARAMETER_COUNT = sum(
    p.numel()
    for p in model_v2_saved.parameters()
)


print(
    "V2 parameters:",
    f"{V2_PARAMETER_COUNT:,}"
)


# ============================================================
# 1. Tiny residual block
# ============================================================

class ResidualBlockV3(
    nn.Module
):

    def __init__(
        self,
        channels,
    ):

        super().__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            padding_mode="replicate",
        )

        self.norm1 = nn.GroupNorm(
            num_groups=4,
            num_channels=channels,
        )

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            padding_mode="replicate",
        )

        self.norm2 = nn.GroupNorm(
            num_groups=4,
            num_channels=channels,
        )

        self.activation = nn.GELU()


    def forward(
        self,
        x,
    ):

        residual = x

        x = self.conv1(
            x
        )

        x = self.norm1(
            x
        )

        x = self.activation(
            x
        )

        x = self.conv2(
            x
        )

        x = self.norm2(
            x
        )

        x = x + residual

        x = self.activation(
            x
        )

        return x


# ============================================================
# 2. Tiny persistence-anchored CNN
# ============================================================

class TinyPersistenceCNN(
    nn.Module
):

    def __init__(
        self,
        normalizers,
        in_channels=14,
        hidden_channels=16,
        num_blocks=1,
    ):

        super().__init__()


        # ----------------------------------------------------
        # Persistence baseline in STANDARDIZED target space
        #
        # raw innovation = 0
        #
        # standardized:
        #
        # (0 - target_mean) / target_std
        # ----------------------------------------------------

        persistence_standardized = (
            - normalizers[
                "target_mean"
            ]
            / normalizers[
                "target_std"
            ]
        ).astype(
            np.float32
        )


        self.register_buffer(
            "persistence_baseline",
            torch.from_numpy(
                persistence_standardized[
                    None,
                    None,
                    :,
                    :,
                ]
            ),
        )


        # ----------------------------------------------------
        # Tiny correction network
        # ----------------------------------------------------

        self.input_layer = nn.Sequential(

            nn.Conv2d(
                in_channels,
                hidden_channels,
                kernel_size=3,
                padding=1,
                padding_mode="replicate",
            ),

            nn.GELU(),
        )


        self.blocks = nn.Sequential(
            *[
                ResidualBlockV3(
                    hidden_channels
                )
                for _ in range(
                    num_blocks
                )
            ]
        )


        self.output_layer = nn.Conv2d(
            hidden_channels,
            1,
            kernel_size=1,
        )


        # ----------------------------------------------------
        # Zero correction at initialization
        # ----------------------------------------------------

        nn.init.zeros_(
            self.output_layer.weight
        )

        nn.init.zeros_(
            self.output_layer.bias
        )


    def forward(
        self,
        x,
    ):

        hidden = (
            self.input_layer(
                x
            )
        )

        hidden = (
            self.blocks(
                hidden
            )
        )

        correction = (
            self.output_layer(
                hidden
            )
        )


        return (
            self.persistence_baseline
            + correction
        )


# ============================================================
# 3. Create V3
# ============================================================

model_v3 = TinyPersistenceCNN(
    normalizers=normalizers,
    in_channels=14,
    hidden_channels=16,
    num_blocks=1,
).to(
    DEVICE
)


V3_PARAMETER_COUNT = sum(
    p.numel()
    for p in model_v3.parameters()
)


print(
    "\nV3 parameters:",
    f"{V3_PARAMETER_COUNT:,}"
)


print(
    "Reduction vs V2:"
)

print(
    1.0
    - (
        V3_PARAMETER_COUNT
        / V2_PARAMETER_COUNT
    )
)


print(
    "Reduction vs V1:"
)

print(
    1.0
    - (
        V3_PARAMETER_COUNT
        / V1_PARAMETER_COUNT
    )
)


# ============================================================
# 4. Verify epoch-0 = raw persistence
# ============================================================

(
    check_input_v3,
    _,
    _,
    _,
) = next(
    iter(
        validation_loader
    )
)


check_input_v3 = (
    check_input_v3.to(
        DEVICE
    )
)


model_v3.eval()


with torch.no_grad():

    initial_v3_std = (
        model_v3(
            check_input_v3
        )
    )


initial_v3_raw = (
    initial_v3_std
    .cpu()
    .numpy()[:, 0]
    * normalizers[
        "target_std"
    ][
        None,
        :,
        :,
    ]
    + normalizers[
        "target_mean"
    ][
        None,
        :,
        :,
    ]
)


print(
    "\nEpoch-0 max absolute raw innovation:"
)

print(
    float(
        np.max(
            np.abs(
                initial_v3_raw
            )
        )
    )
)


assert np.allclose(
    initial_v3_raw,
    0.0,
    atol=1e-9,
)


print(
    "Epoch-0 V3 equals raw persistence: TRUE"
)


# ============================================================
# 5. Training config
# ============================================================

V3_EPOCHS = 40

V3_EARLY_STOPPING_PATIENCE = 4

V3_LEARNING_RATE = 1e-3

V3_WEIGHT_DECAY = 1e-5

V3_GRADIENT_CLIP = 1.0


optimizer_v3 = torch.optim.AdamW(
    model_v3.parameters(),
    lr=V3_LEARNING_RATE,
    weight_decay=V3_WEIGHT_DECAY,
)


scheduler_v3 = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer_v3,
        mode="min",
        factor=0.5,
        patience=2,
    )
)


# ============================================================
# 6. Train/eval epoch helpers
# ============================================================

def train_one_epoch_v3(
    model,
    loader,
    optimizer,
):

    model.train()

    total_loss = 0.0
    total_valid = 0.0


    for (
        x,
        target,
        mask,
        _,
    ) in loader:

        x = x.to(
            DEVICE
        )

        target = target.to(
            DEVICE
        )

        mask = mask.to(
            DEVICE
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        prediction = (
            model(
                x
            )
        )


        loss = (
            masked_mse_loss(
                prediction,
                target,
                mask,
            )
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            V3_GRADIENT_CLIP,
        )


        optimizer.step()


        valid = float(
            mask.sum().item()
        )


        total_loss += (
            float(
                loss.detach().cpu()
            )
            * valid
        )


        total_valid += (
            valid
        )


    return (
        total_loss
        / total_valid
    )


def evaluate_epoch_v3(
    model,
    loader,
):

    model.eval()

    total_loss = 0.0
    total_valid = 0.0


    with torch.no_grad():

        for (
            x,
            target,
            mask,
            _,
        ) in loader:

            x = x.to(
                DEVICE
            )

            target = target.to(
                DEVICE
            )

            mask = mask.to(
                DEVICE
            )


            prediction = (
                model(
                    x
                )
            )


            loss = (
                masked_mse_loss(
                    prediction,
                    target,
                    mask,
                )
            )


            valid = float(
                mask.sum().item()
            )


            total_loss += (
                float(
                    loss.cpu()
                )
                * valid
            )


            total_valid += (
                valid
            )


    return (
        total_loss
        / total_valid
    )


# ============================================================
# 7. Train V3
# ============================================================

best_v3_validation_loss = float(
    "inf"
)

best_v3_state = None

best_v3_epoch = None

v3_wait = 0

v3_history = []


for epoch in range(
    1,
    V3_EPOCHS + 1,
):

    train_loss = (
        train_one_epoch_v3(
            model_v3,
            train_loader,
            optimizer_v3,
        )
    )


    val_loss = (
        evaluate_epoch_v3(
            model_v3,
            validation_loader,
        )
    )


    scheduler_v3.step(
        val_loss
    )


    current_lr = (
        optimizer_v3
        .param_groups[0]["lr"]
    )


    improvement_vs_persistence = (
        1.0
        - (
            val_loss
            / v2_validation_persistence_loss
        )
    )


    v3_history.append(
        {
            "epoch": (
                epoch
            ),

            "train_loss": (
                train_loss
            ),

            "validation_loss": (
                val_loss
            ),

            "improvement_vs_persistence": (
                improvement_vs_persistence
            ),

            "learning_rate": (
                current_lr
            ),
        }
    )


    print(
        f"V3 epoch={epoch:03d} | "
        f"train={train_loss:.6f} | "
        f"val={val_loss:.6f} | "
        f"persistence="
        f"{v2_validation_persistence_loss:.6f} | "
        f"improvement="
        f"{100 * improvement_vs_persistence:.2f}% | "
        f"lr={current_lr:.2e}"
    )


    if (
        val_loss
        < best_v3_validation_loss
        - 1e-6
    ):

        best_v3_validation_loss = (
            val_loss
        )

        best_v3_epoch = (
            epoch
        )

        best_v3_state = (
            copy.deepcopy(
                model_v3.state_dict()
            )
        )

        v3_wait = 0


    else:

        v3_wait += 1


    if (
        v3_wait
        >= V3_EARLY_STOPPING_PATIENCE
    ):

        print(
            "\nV3 early stopping."
        )

        break


# ============================================================
# 8. Restore best V3
# ============================================================

if (
    best_v3_state
    is None
):

    raise RuntimeError(
        "No V3 best state found."
    )


model_v3.load_state_dict(
    best_v3_state
)


print(
    "\nBest V3 epoch:"
)

print(
    best_v3_epoch
)


print(
    "Best V3 validation loss:"
)

print(
    best_v3_validation_loss
)


print(
    "Best V3 validation improvement "
    "vs persistence:"
)

print(
    1.0
    - (
        best_v3_validation_loss
        / v2_validation_persistence_loss
    )
)


# ============================================================
# 9. Raw validation/test evaluation
# ============================================================

(
    v3_validation_raw_summary,
    v3_validation_sample_metrics,
) = evaluate_raw_split(
    split_name="validation",
    loader=validation_loader,
    model=model_v3,
    samples=samples,
    normalizers=normalizers,
    device=DEVICE,
)


(
    v3_test_raw_summary,
    v3_test_sample_metrics,
) = evaluate_raw_split(
    split_name="test",
    loader=test_loader,
    model=model_v3,
    samples=samples,
    normalizers=normalizers,
    device=DEVICE,
)


print(
    "\nV3 VALIDATION RAW:"
)

print(
    json.dumps(
        v3_validation_raw_summary,
        indent=2,
    )
)


print(
    "\nV3 TEST RAW:"
)

print(
    json.dumps(
        v3_test_raw_summary,
        indent=2,
    )
)


# ============================================================
# 10. Day-level V3 results
# ============================================================

v3_daily_test = (
    v3_test_sample_metrics
    .groupby(
        "trade_date",
        observed=True,
    )
    .agg(
        cnn_mse=(
            "cnn_mse",
            "mean",
        ),

        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        n_samples=(
            "sample_index",
            "size",
        ),
    )
    .reset_index()
)


v3_daily_test[
    "mse_difference"
] = (
    v3_daily_test[
        "cnn_mse"
    ]
    - v3_daily_test[
        "persistence_mse"
    ]
)


v3_day_win_rate = float(
    (
        v3_daily_test[
            "mse_difference"
        ]
        < 0
    ).mean()
)


print(
    "\nV3 test day win rate:"
)

print(
    v3_day_win_rate
)


v3_bootstrap = (
    bootstrap_daily_difference(
        v3_daily_test[
            "mse_difference"
        ].to_numpy()
    )
)


print(
    "\nV3 DAY-LEVEL BOOTSTRAP:"
)

print(
    json.dumps(
        v3_bootstrap,
        indent=2,
    )
)


# ============================================================
# 11. V1 vs V2 vs V3 comparison
# ============================================================

comparison_v1_v2_v3 = pd.DataFrame(
    [
        {
            "model": (
                "V1 CNN"
            ),

            "parameters": (
                V1_PARAMETER_COUNT
            ),

            "validation_rmse": (
                validation_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "validation_mae": (
                validation_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_rmse": (
                test_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "test_mae": (
                test_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_mse_improvement": (
                test_raw_summary[
                    "relative_mse_improvement"
                ]
            ),

            "test_mae_improvement": (
                1.0
                - (
                    test_raw_summary[
                        "cnn_mae"
                    ]
                    / test_raw_summary[
                        "persistence_mae"
                    ]
                )
            ),

            "test_day_win_rate": (
                cnn_win_fraction
            ),
        },

        {
            "model": (
                "V2 Smaller CNN"
            ),

            "parameters": (
                V2_PARAMETER_COUNT
            ),

            "validation_rmse": (
                v2_validation_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "validation_mae": (
                v2_validation_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_rmse": (
                v2_test_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "test_mae": (
                v2_test_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_mse_improvement": (
                v2_test_raw_summary[
                    "relative_mse_improvement"
                ]
            ),

            "test_mae_improvement": (
                1.0
                - (
                    v2_test_raw_summary[
                        "cnn_mae"
                    ]
                    / v2_test_raw_summary[
                        "persistence_mae"
                    ]
                )
            ),

            "test_day_win_rate": (
                v2_day_win_rate
            ),
        },

        {
            "model": (
                "V3 Tiny CNN"
            ),

            "parameters": (
                V3_PARAMETER_COUNT
            ),

            "validation_rmse": (
                v3_validation_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "validation_mae": (
                v3_validation_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_rmse": (
                v3_test_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "test_mae": (
                v3_test_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_mse_improvement": (
                v3_test_raw_summary[
                    "relative_mse_improvement"
                ]
            ),

            "test_mae_improvement": (
                1.0
                - (
                    v3_test_raw_summary[
                        "cnn_mae"
                    ]
                    / v3_test_raw_summary[
                        "persistence_mae"
                    ]
                )
            ),

            "test_day_win_rate": (
                v3_day_win_rate
            ),
        },
    ]
)


print(
    "\n================ V1 vs V2 vs V3 ================"
)

display(
    comparison_v1_v2_v3
)


# ============================================================
# 12. V3 training curve
# ============================================================

v3_history_df = pd.DataFrame(
    v3_history
)


plt.figure(
    figsize=(8, 5)
)


plt.plot(
    v3_history_df[
        "epoch"
    ],
    v3_history_df[
        "train_loss"
    ],
    label="V3 train",
)


plt.plot(
    v3_history_df[
        "epoch"
    ],
    v3_history_df[
        "validation_loss"
    ],
    label="V3 validation",
)


plt.axhline(
    v2_validation_persistence_loss,
    linestyle="--",
    label="persistence",
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Masked standardized MSE"
)

plt.title(
    "V3 tiny persistence-anchored CNN"
)

plt.legend()

plt.show()


print(
    "\nBLOCK 23 finished successfully."
)

In [ ]:
# ============================================================
# BLOCK 24
# V2 Relaxed-Training Check
#
# Same architecture as V2:
#   hidden_channels = 32
#   num_blocks      = 2
#
# Only training schedule is relaxed:
#   max epochs             = 80
#   early stopping patience= 10
#   scheduler patience     = 4
#
# Goal:
# determine whether previous V2 stopped too early.
# ============================================================


# ============================================================
# 0. Preserve current V2 result
# ============================================================

model_v2_original = copy.deepcopy(
    model_v2
).to(
    DEVICE
)

model_v2_original.eval()


ORIGINAL_V2_BEST_EPOCH = (
    best_v2_epoch
)

ORIGINAL_V2_BEST_VAL_LOSS = (
    best_v2_validation_loss
)


print(
    "Original V2 best epoch:",
    ORIGINAL_V2_BEST_EPOCH
)

print(
    "Original V2 best validation loss:",
    ORIGINAL_V2_BEST_VAL_LOSS
)


# ============================================================
# 1. Re-create fresh V2 model
#
# IMPORTANT:
# do NOT continue training old V2.
# Start from fresh persistence-anchored initialization.
# ============================================================

model_v2_relaxed = PersistenceAnchoredCNN(
    normalizers=normalizers,
    in_channels=14,
    hidden_channels=32,
    num_blocks=2,
).to(
    DEVICE
)


RELAXED_V2_PARAMETER_COUNT = sum(
    p.numel()
    for p in model_v2_relaxed.parameters()
)


print(
    "\nRelaxed V2 parameters:",
    f"{RELAXED_V2_PARAMETER_COUNT:,}"
)


# ============================================================
# 2. Verify epoch-0 still equals raw persistence
# ============================================================

(
    check_input_relaxed,
    _,
    _,
    _,
) = next(
    iter(
        validation_loader
    )
)


check_input_relaxed = (
    check_input_relaxed.to(
        DEVICE
    )
)


model_v2_relaxed.eval()


with torch.no_grad():

    initial_relaxed_std = (
        model_v2_relaxed(
            check_input_relaxed
        )
    )


initial_relaxed_raw = (
    initial_relaxed_std
    .cpu()
    .numpy()[:, 0]
    * normalizers[
        "target_std"
    ][
        None,
        :,
        :,
    ]
    + normalizers[
        "target_mean"
    ][
        None,
        :,
        :,
    ]
)


print(
    "\nEpoch-0 max absolute raw innovation:"
)

print(
    float(
        np.max(
            np.abs(
                initial_relaxed_raw
            )
        )
    )
)


assert np.allclose(
    initial_relaxed_raw,
    0.0,
    atol=1e-9,
)


print(
    "Epoch-0 relaxed V2 equals raw persistence: TRUE"
)


# ============================================================
# 3. Relaxed training config
# ============================================================

V2_RELAXED_EPOCHS = 80

V2_RELAXED_EARLY_STOPPING_PATIENCE = 10

V2_RELAXED_SCHEDULER_PATIENCE = 4

V2_RELAXED_LEARNING_RATE = 1e-3

V2_RELAXED_WEIGHT_DECAY = 1e-5

V2_RELAXED_GRADIENT_CLIP = 1.0


optimizer_v2_relaxed = torch.optim.AdamW(
    model_v2_relaxed.parameters(),
    lr=V2_RELAXED_LEARNING_RATE,
    weight_decay=V2_RELAXED_WEIGHT_DECAY,
)


scheduler_v2_relaxed = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(
        optimizer_v2_relaxed,
        mode="min",
        factor=0.5,
        patience=V2_RELAXED_SCHEDULER_PATIENCE,
    )
)


# ============================================================
# 4. Training helpers
#
# Reuse masked_mse_loss().
# ============================================================

def train_one_epoch_v2_relaxed(
    model,
    loader,
    optimizer,
):

    model.train()

    total_loss = 0.0
    total_valid = 0.0


    for (
        x,
        target,
        mask,
        _,
    ) in loader:

        x = x.to(
            DEVICE
        )

        target = target.to(
            DEVICE
        )

        mask = mask.to(
            DEVICE
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        prediction = (
            model(
                x
            )
        )


        loss = (
            masked_mse_loss(
                prediction,
                target,
                mask,
            )
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            V2_RELAXED_GRADIENT_CLIP,
        )


        optimizer.step()


        valid = float(
            mask.sum().item()
        )


        total_loss += (
            float(
                loss.detach().cpu()
            )
            * valid
        )


        total_valid += (
            valid
        )


    return (
        total_loss
        / total_valid
    )


def evaluate_v2_relaxed(
    model,
    loader,
):

    model.eval()

    total_loss = 0.0
    total_valid = 0.0


    with torch.no_grad():

        for (
            x,
            target,
            mask,
            _,
        ) in loader:

            x = x.to(
                DEVICE
            )

            target = target.to(
                DEVICE
            )

            mask = mask.to(
                DEVICE
            )


            prediction = (
                model(
                    x
                )
            )


            loss = (
                masked_mse_loss(
                    prediction,
                    target,
                    mask,
                )
            )


            valid = float(
                mask.sum().item()
            )


            total_loss += (
                float(
                    loss.cpu()
                )
                * valid
            )


            total_valid += (
                valid
            )


    return (
        total_loss
        / total_valid
    )


# ============================================================
# 5. Train relaxed V2
# ============================================================

best_v2_relaxed_val_loss = float(
    "inf"
)

best_v2_relaxed_state = None

best_v2_relaxed_epoch = None

v2_relaxed_wait = 0

v2_relaxed_history = []


for epoch in range(
    1,
    V2_RELAXED_EPOCHS + 1,
):

    train_loss = (
        train_one_epoch_v2_relaxed(
            model_v2_relaxed,
            train_loader,
            optimizer_v2_relaxed,
        )
    )


    val_loss = (
        evaluate_v2_relaxed(
            model_v2_relaxed,
            validation_loader,
        )
    )


    scheduler_v2_relaxed.step(
        val_loss
    )


    current_lr = (
        optimizer_v2_relaxed
        .param_groups[0]["lr"]
    )


    improvement_vs_persistence = (
        1.0
        - (
            val_loss
            / v2_validation_persistence_loss
        )
    )


    v2_relaxed_history.append(
        {
            "epoch": (
                epoch
            ),

            "train_loss": (
                train_loss
            ),

            "validation_loss": (
                val_loss
            ),

            "improvement_vs_persistence": (
                improvement_vs_persistence
            ),

            "learning_rate": (
                current_lr
            ),
        }
    )


    print(
        f"Relaxed V2 epoch={epoch:03d} | "
        f"train={train_loss:.6f} | "
        f"val={val_loss:.6f} | "
        f"persistence={v2_validation_persistence_loss:.6f} | "
        f"improvement="
        f"{100 * improvement_vs_persistence:.2f}% | "
        f"lr={current_lr:.2e}"
    )


    if (
        val_loss
        < best_v2_relaxed_val_loss
        - 1e-6
    ):

        best_v2_relaxed_val_loss = (
            val_loss
        )

        best_v2_relaxed_epoch = (
            epoch
        )

        best_v2_relaxed_state = (
            copy.deepcopy(
                model_v2_relaxed.state_dict()
            )
        )

        v2_relaxed_wait = 0


    else:

        v2_relaxed_wait += 1


    if (
        v2_relaxed_wait
        >= V2_RELAXED_EARLY_STOPPING_PATIENCE
    ):

        print(
            "\nRelaxed V2 early stopping."
        )

        break


# ============================================================
# 6. Restore best relaxed V2
# ============================================================

if (
    best_v2_relaxed_state
    is None
):

    raise RuntimeError(
        "No relaxed V2 best state found."
    )


model_v2_relaxed.load_state_dict(
    best_v2_relaxed_state
)


print(
    "\nBest relaxed V2 epoch:"
)

print(
    best_v2_relaxed_epoch
)


print(
    "Best relaxed V2 validation loss:"
)

print(
    best_v2_relaxed_val_loss
)


print(
    "Best relaxed V2 validation improvement "
    "vs persistence:"
)

print(
    1.0
    - (
        best_v2_relaxed_val_loss
        / v2_validation_persistence_loss
    )
)


# ============================================================
# 7. Raw validation/test evaluation
# ============================================================

(
    v2_relaxed_validation_raw_summary,
    v2_relaxed_validation_sample_metrics,
) = evaluate_raw_split(
    split_name="validation",
    loader=validation_loader,
    model=model_v2_relaxed,
    samples=samples,
    normalizers=normalizers,
    device=DEVICE,
)


(
    v2_relaxed_test_raw_summary,
    v2_relaxed_test_sample_metrics,
) = evaluate_raw_split(
    split_name="test",
    loader=test_loader,
    model=model_v2_relaxed,
    samples=samples,
    normalizers=normalizers,
    device=DEVICE,
)


print(
    "\nRELAXED V2 VALIDATION RAW:"
)

print(
    json.dumps(
        v2_relaxed_validation_raw_summary,
        indent=2,
    )
)


print(
    "\nRELAXED V2 TEST RAW:"
)

print(
    json.dumps(
        v2_relaxed_test_raw_summary,
        indent=2,
    )
)


# ============================================================
# 8. Day-level test metrics
# ============================================================

v2_relaxed_daily_test = (
    v2_relaxed_test_sample_metrics
    .groupby(
        "trade_date",
        observed=True,
    )
    .agg(
        cnn_mse=(
            "cnn_mse",
            "mean",
        ),

        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        n_samples=(
            "sample_index",
            "size",
        ),
    )
    .reset_index()
)


v2_relaxed_daily_test[
    "mse_difference"
] = (
    v2_relaxed_daily_test[
        "cnn_mse"
    ]
    - v2_relaxed_daily_test[
        "persistence_mse"
    ]
)


v2_relaxed_day_win_rate = float(
    (
        v2_relaxed_daily_test[
            "mse_difference"
        ]
        < 0
    ).mean()
)


print(
    "\nRelaxed V2 test day win rate:"
)

print(
    v2_relaxed_day_win_rate
)


# ============================================================
# 9. Day-level bootstrap
# ============================================================

v2_relaxed_bootstrap = (
    bootstrap_daily_difference(
        v2_relaxed_daily_test[
            "mse_difference"
        ].to_numpy()
    )
)


print(
    "\nRELAXED V2 DAY-LEVEL BOOTSTRAP:"
)

print(
    json.dumps(
        v2_relaxed_bootstrap,
        indent=2,
    )
)


# ============================================================
# 10. Compare original V2 vs relaxed V2
# ============================================================

comparison_v2_training_schedule = pd.DataFrame(
    [
        {
            "model": (
                "V2 original"
            ),

            "best_epoch": (
                ORIGINAL_V2_BEST_EPOCH
            ),

            "best_standardized_val_loss": (
                ORIGINAL_V2_BEST_VAL_LOSS
            ),

            "validation_rmse": (
                v2_validation_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "validation_mae": (
                v2_validation_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_rmse": (
                v2_test_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "test_mae": (
                v2_test_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_mse_improvement": (
                v2_test_raw_summary[
                    "relative_mse_improvement"
                ]
            ),

            "test_mae_improvement": (
                1.0
                - (
                    v2_test_raw_summary[
                        "cnn_mae"
                    ]
                    / v2_test_raw_summary[
                        "persistence_mae"
                    ]
                )
            ),

            "test_day_win_rate": (
                v2_day_win_rate
            ),
        },

        {
            "model": (
                "V2 relaxed"
            ),

            "best_epoch": (
                best_v2_relaxed_epoch
            ),

            "best_standardized_val_loss": (
                best_v2_relaxed_val_loss
            ),

            "validation_rmse": (
                v2_relaxed_validation_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "validation_mae": (
                v2_relaxed_validation_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_rmse": (
                v2_relaxed_test_raw_summary[
                    "cnn_rmse"
                ]
            ),

            "test_mae": (
                v2_relaxed_test_raw_summary[
                    "cnn_mae"
                ]
            ),

            "test_mse_improvement": (
                v2_relaxed_test_raw_summary[
                    "relative_mse_improvement"
                ]
            ),

            "test_mae_improvement": (
                1.0
                - (
                    v2_relaxed_test_raw_summary[
                        "cnn_mae"
                    ]
                    / v2_relaxed_test_raw_summary[
                        "persistence_mae"
                    ]
                )
            ),

            "test_day_win_rate": (
                v2_relaxed_day_win_rate
            ),
        },
    ]
)


print(
    "\n================ V2 TRAINING-SCHEDULE COMPARISON ================"
)

display(
    comparison_v2_training_schedule
)


# ============================================================
# 11. Training curve
# ============================================================

v2_relaxed_history_df = pd.DataFrame(
    v2_relaxed_history
)


plt.figure(
    figsize=(8, 5)
)


plt.plot(
    v2_relaxed_history_df[
        "epoch"
    ],

    v2_relaxed_history_df[
        "train_loss"
    ],

    label="train",
)


plt.plot(
    v2_relaxed_history_df[
        "epoch"
    ],

    v2_relaxed_history_df[
        "validation_loss"
    ],

    label="validation",
)


plt.axhline(
    v2_validation_persistence_loss,
    linestyle="--",
    label="persistence",
)


plt.axvline(
    best_v2_relaxed_epoch,
    linestyle=":",
    label="best epoch",
)


plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Masked standardized MSE"
)

plt.title(
    "V2 relaxed training"
)

plt.legend()

plt.show()


print(
    "\nBLOCK 24 finished successfully."
)

In [ ]:
# ============================================================
# BLOCK 25
# V2 Correction-Strength Calibration
#
# Final deterministic mean:
#
#     Y_hat_calibrated
#         =
#     g(p_failure) * Y_hat_V2
#
# with
#
#     g(p) = clip(a * p + b, 0, 1)
#
# IMPORTANT:
#
#   1. CNN is NOT retrained.
#   2. Failure classifier is NOT retrained.
#   3. (a, b) are selected using VALIDATION only.
#   4. TEST is evaluated only after (a, b) are frozen.
# ============================================================


# ============================================================
# 0. Choose the mean model
#
# Use ORIGINAL V2, not relaxed V2 and not V3.
# ============================================================

mean_model_v2 = model_v2

mean_model_v2.eval()


print(
    "Mean model: ORIGINAL V2"
)

print(
    "Parameters:",
    sum(
        p.numel()
        for p in mean_model_v2.parameters()
    )
)


# ============================================================
# 1. Get raw V2 innovation predictions
#
# Reuse helper from Block 21:
#
#     get_raw_cnn_predictions()
#
# Output:
#     indices
#     raw innovation prediction [N, 41, 10]
# ============================================================

(
    val_indices_v2_cal,
    val_v2_innovation,
) = get_raw_cnn_predictions(
    model=mean_model_v2,
    loader=validation_loader,
    normalizers=normalizers,
    device=DEVICE,
)


(
    test_indices_v2_cal,
    test_v2_innovation,
) = get_raw_cnn_predictions(
    model=mean_model_v2,
    loader=test_loader,
    normalizers=normalizers,
    device=DEVICE,
)


print(
    "\nValidation prediction shape:",
    val_v2_innovation.shape
)

print(
    "Test prediction shape:",
    test_v2_innovation.shape
)


# ============================================================
# 2. Get persistence-failure scores
#
# Everything here uses CURRENT/PAST features only.
#
# feature_scaler
# failure_classifier
# X_all
#
# come from Block 20.
# ============================================================

val_features_cal = (
    feature_scaler.transform(
        X_all[
            val_indices_v2_cal
        ]
    )
)


test_features_cal = (
    feature_scaler.transform(
        X_all[
            test_indices_v2_cal
        ]
    )
)


val_failure_score = (
    failure_classifier.predict_proba(
        val_features_cal
    )[:, 1]
)


test_failure_score = (
    failure_classifier.predict_proba(
        test_features_cal
    )[:, 1]
)


print(
    "\nValidation failure-score range:",
    float(
        val_failure_score.min()
    ),
    "->",
    float(
        val_failure_score.max()
    )
)


print(
    "Test failure-score range:",
    float(
        test_failure_score.min()
    ),
    "->",
    float(
        test_failure_score.max()
    )
)


# ============================================================
# 3. Build raw target innovation and transition mask
# ============================================================

val_true_innovation = (
    samples[
        "future"
    ][
        val_indices_v2_cal
    ]
    - samples[
        "rolled"
    ][
        val_indices_v2_cal
    ]
).astype(
    np.float64
)


test_true_innovation = (
    samples[
        "future"
    ][
        test_indices_v2_cal
    ]
    - samples[
        "rolled"
    ][
        test_indices_v2_cal
    ]
).astype(
    np.float64
)


val_transition_mask_cal = (
    samples[
        "transition_mask"
    ][
        val_indices_v2_cal
    ]
).astype(
    np.float64
)


test_transition_mask_cal = (
    samples[
        "transition_mask"
    ][
        test_indices_v2_cal
    ]
).astype(
    np.float64
)


# ============================================================
# 4. Precompute quadratic MSE terms
#
# For sample i:
#
#   L_i(g)
#     =
#   sum_j mask_ij * (g_i*C_ij - Y_ij)^2
#
#     =
#   A_i*g_i^2
#   - 2*B_i*g_i
#   + D_i
#
# This makes searching (a,b) extremely cheap.
# ============================================================

val_A = (
    (
        val_v2_innovation ** 2
    )
    * val_transition_mask_cal
).sum(
    axis=(1, 2)
)


val_B = (
    (
        val_v2_innovation
        * val_true_innovation
    )
    * val_transition_mask_cal
).sum(
    axis=(1, 2)
)


val_D = (
    (
        val_true_innovation ** 2
    )
    * val_transition_mask_cal
).sum(
    axis=(1, 2)
)


val_valid_points = (
    val_transition_mask_cal.sum()
)


def fast_validation_mse_from_gate(
    gate,
):

    numerator = (
        (
            val_A
            * gate**2
        )
        - (
            2.0
            * val_B
            * gate
        )
        + val_D
    ).sum()


    return float(
        numerator
        / val_valid_points
    )


# ============================================================
# 5. Sanity checks:
# persistence gate=0
# raw V2 gate=1
# ============================================================

val_persistence_mse_cal = (
    fast_validation_mse_from_gate(
        np.zeros_like(
            val_failure_score
        )
    )
)


val_raw_v2_mse_cal = (
    fast_validation_mse_from_gate(
        np.ones_like(
            val_failure_score
        )
    )
)


print(
    "\nValidation persistence MSE:",
    val_persistence_mse_cal
)


print(
    "Validation raw V2 MSE:",
    val_raw_v2_mse_cal
)


print(
    "Raw V2 MSE improvement:"
)

print(
    1.0
    - (
        val_raw_v2_mse_cal
        / val_persistence_mse_cal
    )
)


# ============================================================
# 6. Coarse search
#
# g(p) = clip(a*p+b, 0, 1)
#
# Constraint:
#   a >= 0
#
# Therefore correction strength is monotone non-decreasing
# with predicted persistence-failure risk.
# ============================================================

A_COARSE = np.arange(
    0.0,
    3.01,
    0.05,
)


B_COARSE = np.arange(
    -1.00,
    0.51,
    0.05,
)


coarse_rows = []


for a in A_COARSE:

    for b in B_COARSE:

        gate = np.clip(
            a
            * val_failure_score
            + b,
            0.0,
            1.0,
        )


        mse = (
            fast_validation_mse_from_gate(
                gate
            )
        )


        coarse_rows.append(
            {
                "a": float(
                    a
                ),

                "b": float(
                    b
                ),

                "mse": (
                    mse
                ),

                "mse_improvement_vs_persistence": float(
                    1.0
                    - (
                        mse
                        / val_persistence_mse_cal
                    )
                ),

                "mean_gate": float(
                    gate.mean()
                ),

                "fraction_gate_zero": float(
                    (
                        gate
                        <= 1e-12
                    ).mean()
                ),

                "fraction_gate_one": float(
                    (
                        gate
                        >= 1.0 - 1e-12
                    ).mean()
                ),
            }
        )


coarse_table = (
    pd.DataFrame(
        coarse_rows
    )
    .sort_values(
        "mse"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n================ COARSE SEARCH TOP 15 ================"
)

display(
    coarse_table.head(
        15
    )
)


best_coarse_a = float(
    coarse_table.iloc[0][
        "a"
    ]
)


best_coarse_b = float(
    coarse_table.iloc[0][
        "b"
    ]
)


print(
    "\nBest coarse a:",
    best_coarse_a
)

print(
    "Best coarse b:",
    best_coarse_b
)


# ============================================================
# 7. Fine search around coarse optimum
# ============================================================

A_FINE = np.arange(
    max(
        0.0,
        best_coarse_a - 0.15,
    ),

    best_coarse_a + 0.1501,

    0.01,
)


B_FINE = np.arange(
    best_coarse_b - 0.15,
    best_coarse_b + 0.1501,
    0.01,
)


fine_rows = []


for a in A_FINE:

    for b in B_FINE:

        gate = np.clip(
            a
            * val_failure_score
            + b,
            0.0,
            1.0,
        )


        mse = (
            fast_validation_mse_from_gate(
                gate
            )
        )


        fine_rows.append(
            {
                "a": float(
                    a
                ),

                "b": float(
                    b
                ),

                "mse": (
                    mse
                ),

                "mse_improvement_vs_persistence": float(
                    1.0
                    - (
                        mse
                        / val_persistence_mse_cal
                    )
                ),

                "mean_gate": float(
                    gate.mean()
                ),

                "fraction_gate_zero": float(
                    (
                        gate
                        <= 1e-12
                    ).mean()
                ),

                "fraction_gate_one": float(
                    (
                        gate
                        >= 1.0 - 1e-12
                    ).mean()
                ),
            }
        )


fine_table = (
    pd.DataFrame(
        fine_rows
    )
    .sort_values(
        "mse"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n================ FINE SEARCH TOP 15 ================"
)

display(
    fine_table.head(
        15
    )
)


BEST_CAL_A = float(
    fine_table.iloc[0][
        "a"
    ]
)


BEST_CAL_B = float(
    fine_table.iloc[0][
        "b"
    ]
)


print(
    "\n================ FINAL CALIBRATION ================"
)

print(
    "BEST_CAL_A:",
    BEST_CAL_A
)

print(
    "BEST_CAL_B:",
    BEST_CAL_B
)


# ============================================================
# 8. Freeze validation-selected calibration
# ============================================================

val_calibrated_gate = np.clip(
    BEST_CAL_A
    * val_failure_score
    + BEST_CAL_B,
    0.0,
    1.0,
)


test_calibrated_gate = np.clip(
    BEST_CAL_A
    * test_failure_score
    + BEST_CAL_B,
    0.0,
    1.0,
)


print(
    "\nValidation mean calibrated gate:",
    float(
        val_calibrated_gate.mean()
    )
)


print(
    "Test mean calibrated gate:",
    float(
        test_calibrated_gate.mean()
    )
)


print(
    "Test gate=0 fraction:",
    float(
        (
            test_calibrated_gate
            <= 1e-12
        ).mean()
    )
)


print(
    "Test gate=1 fraction:",
    float(
        (
            test_calibrated_gate
            >= 1.0 - 1e-12
        ).mean()
    )
)


# ============================================================
# 9. Build calibrated predictions
# ============================================================

val_calibrated_innovation = (
    val_v2_innovation
    * val_calibrated_gate[
        :,
        None,
        None,
    ]
)


test_calibrated_innovation = (
    test_v2_innovation
    * test_calibrated_gate[
        :,
        None,
        None,
    ]
)


# ============================================================
# 10. Generic evaluation
#
# Reuse evaluate_innovation_prediction() from Block 21.
# ============================================================

val_zero_v2 = np.zeros_like(
    val_v2_innovation
)


test_zero_v2 = np.zeros_like(
    test_v2_innovation
)


val_persistence_cal_metrics = (
    evaluate_innovation_prediction(
        indices=val_indices_v2_cal,
        predicted_innovation=val_zero_v2,
        samples=samples,
    )
)


val_raw_v2_metrics = (
    evaluate_innovation_prediction(
        indices=val_indices_v2_cal,
        predicted_innovation=val_v2_innovation,
        samples=samples,
    )
)


val_calibrated_v2_metrics = (
    evaluate_innovation_prediction(
        indices=val_indices_v2_cal,
        predicted_innovation=val_calibrated_innovation,
        samples=samples,
    )
)


test_persistence_cal_metrics = (
    evaluate_innovation_prediction(
        indices=test_indices_v2_cal,
        predicted_innovation=test_zero_v2,
        samples=samples,
    )
)


test_raw_v2_metrics = (
    evaluate_innovation_prediction(
        indices=test_indices_v2_cal,
        predicted_innovation=test_v2_innovation,
        samples=samples,
    )
)


test_calibrated_v2_metrics = (
    evaluate_innovation_prediction(
        indices=test_indices_v2_cal,
        predicted_innovation=test_calibrated_innovation,
        samples=samples,
    )
)


# ============================================================
# 11. Validation comparison
# ============================================================

validation_calibration_comparison = pd.DataFrame(
    [
        {
            "model": "Persistence",

            "mse": (
                val_persistence_cal_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                val_persistence_cal_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                val_persistence_cal_metrics[
                    "mae"
                ]
            ),
        },

        {
            "model": "Raw V2",

            "mse": (
                val_raw_v2_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                val_raw_v2_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                val_raw_v2_metrics[
                    "mae"
                ]
            ),
        },

        {
            "model": "Calibrated V2",

            "mse": (
                val_calibrated_v2_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                val_calibrated_v2_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                val_calibrated_v2_metrics[
                    "mae"
                ]
            ),
        },
    ]
)


validation_calibration_comparison[
    "mse_improvement_vs_persistence"
] = (
    1.0
    - validation_calibration_comparison[
        "mse"
    ]
    / val_persistence_cal_metrics[
        "mse"
    ]
)


validation_calibration_comparison[
    "mae_improvement_vs_persistence"
] = (
    1.0
    - validation_calibration_comparison[
        "mae"
    ]
    / val_persistence_cal_metrics[
        "mae"
    ]
)


print(
    "\n================ VALIDATION CALIBRATION COMPARISON ================"
)

display(
    validation_calibration_comparison
)


# ============================================================
# 12. FINAL TEST comparison
# ============================================================

test_calibration_comparison = pd.DataFrame(
    [
        {
            "model": "Persistence",

            "mse": (
                test_persistence_cal_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                test_persistence_cal_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                test_persistence_cal_metrics[
                    "mae"
                ]
            ),
        },

        {
            "model": "Raw V2",

            "mse": (
                test_raw_v2_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                test_raw_v2_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                test_raw_v2_metrics[
                    "mae"
                ]
            ),
        },

        {
            "model": "Calibrated V2",

            "mse": (
                test_calibrated_v2_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                test_calibrated_v2_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                test_calibrated_v2_metrics[
                    "mae"
                ]
            ),
        },
    ]
)


test_calibration_comparison[
    "mse_improvement_vs_persistence"
] = (
    1.0
    - test_calibration_comparison[
        "mse"
    ]
    / test_persistence_cal_metrics[
        "mse"
    ]
)


test_calibration_comparison[
    "mae_improvement_vs_persistence"
] = (
    1.0
    - test_calibration_comparison[
        "mae"
    ]
    / test_persistence_cal_metrics[
        "mae"
    ]
)


print(
    "\n================ FINAL TEST CALIBRATION COMPARISON ================"
)

display(
    test_calibration_comparison
)


# ============================================================
# 13. Sample-level dataframe
# ============================================================

test_calibration_sample_df = pd.DataFrame(
    {
        "sample_index": (
            test_indices_v2_cal
        ),

        "trade_date": (
            samples[
                "date"
            ][
                test_indices_v2_cal
            ]
        ),

        "clock_time": (
            samples[
                "clock_time"
            ][
                test_indices_v2_cal
            ]
        ),

        "failure_score": (
            test_failure_score
        ),

        "calibrated_gate": (
            test_calibrated_gate
        ),

        "persistence_mse": (
            test_persistence_cal_metrics[
                "sample_mse"
            ]
        ),

        "raw_v2_mse": (
            test_raw_v2_metrics[
                "sample_mse"
            ]
        ),

        "calibrated_v2_mse": (
            test_calibrated_v2_metrics[
                "sample_mse"
            ]
        ),
    }
)


# ============================================================
# 14. Sample win rates
# ============================================================

raw_v2_sample_win_rate = float(
    (
        test_calibration_sample_df[
            "raw_v2_mse"
        ]
        < test_calibration_sample_df[
            "persistence_mse"
        ]
    ).mean()
)


calibrated_v2_sample_win_rate = float(
    (
        test_calibration_sample_df[
            "calibrated_v2_mse"
        ]
        < test_calibration_sample_df[
            "persistence_mse"
        ]
    ).mean()
)


print(
    "\nRaw V2 sample win rate:",
    raw_v2_sample_win_rate
)


print(
    "Calibrated V2 sample win rate:",
    calibrated_v2_sample_win_rate
)


# ============================================================
# 15. Day-level comparison
# ============================================================

daily_calibrated_v2 = (
    test_calibration_sample_df
    .groupby(
        "trade_date",
        observed=True,
    )
    .agg(
        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        raw_v2_mse=(
            "raw_v2_mse",
            "mean",
        ),

        calibrated_v2_mse=(
            "calibrated_v2_mse",
            "mean",
        ),

        samples=(
            "sample_index",
            "size",
        ),
    )
    .reset_index()
)


daily_calibrated_v2[
    "raw_difference"
] = (
    daily_calibrated_v2[
        "raw_v2_mse"
    ]
    - daily_calibrated_v2[
        "persistence_mse"
    ]
)


daily_calibrated_v2[
    "calibrated_difference"
] = (
    daily_calibrated_v2[
        "calibrated_v2_mse"
    ]
    - daily_calibrated_v2[
        "persistence_mse"
    ]
)


raw_v2_day_win_rate = float(
    (
        daily_calibrated_v2[
            "raw_difference"
        ]
        < 0
    ).mean()
)


calibrated_v2_day_win_rate = float(
    (
        daily_calibrated_v2[
            "calibrated_difference"
        ]
        < 0
    ).mean()
)


print(
    "\nRaw V2 day win rate:",
    raw_v2_day_win_rate
)


print(
    "Calibrated V2 day win rate:",
    calibrated_v2_day_win_rate
)


# ============================================================
# 16. Day-level bootstrap
# ============================================================

raw_v2_bootstrap_cal = (
    bootstrap_daily_difference(
        daily_calibrated_v2[
            "raw_difference"
        ].to_numpy()
    )
)


calibrated_v2_bootstrap_cal = (
    bootstrap_daily_difference(
        daily_calibrated_v2[
            "calibrated_difference"
        ].to_numpy()
    )
)


bootstrap_calibration_table = pd.DataFrame(
    [
        {
            "model": "Raw V2",
            **raw_v2_bootstrap_cal,
        },

        {
            "model": "Calibrated V2",
            **calibrated_v2_bootstrap_cal,
        },
    ]
)


print(
    "\n================ DAY-LEVEL BOOTSTRAP ================"
)

display(
    bootstrap_calibration_table
)


# ============================================================
# 17. Analyze by failure-score quintile
# ============================================================

test_calibration_sample_df[
    "failure_score_bucket"
] = pd.qcut(
    test_calibration_sample_df[
        "failure_score"
    ].rank(
        method="first"
    ),

    q=5,

    labels=[
        "P_Q1",
        "P_Q2",
        "P_Q3",
        "P_Q4",
        "P_Q5",
    ],
)


calibration_regime_table = (
    test_calibration_sample_df
    .groupby(
        "failure_score_bucket",
        observed=True,
    )
    .agg(
        samples=(
            "sample_index",
            "size",
        ),

        mean_failure_score=(
            "failure_score",
            "mean",
        ),

        mean_gate=(
            "calibrated_gate",
            "mean",
        ),

        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        raw_v2_mse=(
            "raw_v2_mse",
            "mean",
        ),

        calibrated_v2_mse=(
            "calibrated_v2_mse",
            "mean",
        ),
    )
    .reset_index()
)


calibration_regime_table[
    "raw_v2_improvement"
] = (
    1.0
    - calibration_regime_table[
        "raw_v2_mse"
    ]
    / calibration_regime_table[
        "persistence_mse"
    ]
)


calibration_regime_table[
    "calibrated_v2_improvement"
] = (
    1.0
    - calibration_regime_table[
        "calibrated_v2_mse"
    ]
    / calibration_regime_table[
        "persistence_mse"
    ]
)


print(
    "\n================ TEST BY FAILURE-SCORE BUCKET ================"
)

display(
    calibration_regime_table
)


# ============================================================
# 18. Gate shape
# ============================================================

p_grid = np.linspace(
    0.0,
    1.0,
    201,
)


g_grid = np.clip(
    BEST_CAL_A
    * p_grid
    + BEST_CAL_B,
    0.0,
    1.0,
)


plt.figure(
    figsize=(7, 5)
)


plt.plot(
    p_grid,
    g_grid,
)


plt.xlabel(
    "Predicted persistence-failure score"
)

plt.ylabel(
    "CNN correction strength g(p)"
)

plt.title(
    "Validation-calibrated V2 correction strength"
)

plt.ylim(
    -0.05,
    1.05,
)

plt.show()


# ============================================================
# 19. Save the final calibrated deterministic mean components
#
# These will be used by the next low-rank residual analysis.
# ============================================================

final_mean_model = (
    mean_model_v2
)


FINAL_CALIBRATION_A = (
    BEST_CAL_A
)


FINAL_CALIBRATION_B = (
    BEST_CAL_B
)


final_validation_indices = (
    val_indices_v2_cal
)


final_test_indices = (
    test_indices_v2_cal
)


final_validation_mean_innovation = (
    val_calibrated_innovation
)


final_test_mean_innovation = (
    test_calibrated_innovation
)


print(
    "\nFinal deterministic mean model:"
)

print(
    "V2 persistence-anchored CNN"
)


print(
    "g(p) = clip("
    f"{FINAL_CALIBRATION_A:.6f}"
    " * p + "
    f"{FINAL_CALIBRATION_B:.6f}"
    ", 0, 1)"
)


print(
    "\nBLOCK 25 finished successfully."
)

In [ ]:
# ============================================================
# BLOCK 26
# Constant shrinkage vs state-dependent shrinkage
#
# Compare:
#
#   1. Persistence
#   2. Raw V2
#   3. Constant-shrinkage V2
#   4. State-dependent calibrated V2
#
# Constant model:
#
#       Y_hat = lambda * Y_hat_V2
#
# State-dependent model:
#
#       Y_hat = g(p) * Y_hat_V2
#
#       g(p) = clip(a*p + b, 0, 1)
#
# IMPORTANT:
#   lambda selected using VALIDATION only.
#   (a,b) already selected on validation in Block 25.
# ============================================================


# ============================================================
# 0. Reuse Block 25 objects
# ============================================================

print(
    "Using ORIGINAL V2 predictions from Block 25."
)

print(
    "Validation samples:",
    len(
        val_indices_v2_cal
    )
)

print(
    "Test samples:",
    len(
        test_indices_v2_cal
    )
)


print(
    "\nState-dependent calibration from Block 25:"
)

print(
    "a =",
    BEST_CAL_A
)

print(
    "b =",
    BEST_CAL_B
)


# ============================================================
# 1. Search optimal CONSTANT shrinkage on validation
#
# Fast quadratic loss:
#
#   L(lambda)
#
#   =
#
#   sum (lambda*C - Y)^2
#
# Since Block 25 already created:
#
#   val_A = sum(C^2)
#   val_B = sum(CY)
#   val_D = sum(Y^2)
#
# across each sample,
# we can search lambda very cheaply.
# ============================================================

LAMBDA_GRID = np.arange(
    0.0,
    1.5001,
    0.005,
)


constant_rows = []


for lam in LAMBDA_GRID:

    gate = np.full_like(
        val_failure_score,
        fill_value=lam,
        dtype=np.float64,
    )


    mse = (
        fast_validation_mse_from_gate(
            gate
        )
    )


    constant_rows.append(
        {
            "lambda": float(
                lam
            ),

            "mse": float(
                mse
            ),

            "mse_improvement_vs_persistence": float(
                1.0
                - (
                    mse
                    / val_persistence_mse_cal
                )
            ),
        }
    )


constant_search_table = (
    pd.DataFrame(
        constant_rows
    )
    .sort_values(
        "mse"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n================ CONSTANT SHRINKAGE TOP 20 ================"
)

display(
    constant_search_table.head(
        20
    )
)


BEST_CONSTANT_LAMBDA = float(
    constant_search_table.iloc[0][
        "lambda"
    ]
)


print(
    "\nBEST CONSTANT LAMBDA:"
)

print(
    BEST_CONSTANT_LAMBDA
)


# ============================================================
# 2. Analytic lambda sanity check
#
# Without [0,1] constraints:
#
# lambda* =
#
#     sum(CY)
#     --------
#     sum(C^2)
#
# ============================================================

analytic_lambda = float(
    val_B.sum()
    / val_A.sum()
)


print(
    "\nAnalytic unconstrained validation lambda:"
)

print(
    analytic_lambda
)


print(
    "Difference grid vs analytic:"
)

print(
    BEST_CONSTANT_LAMBDA
    - analytic_lambda
)


# ============================================================
# 3. Build validation/test constant gates
# ============================================================

val_constant_gate = np.full(
    shape=len(
        val_indices_v2_cal
    ),
    fill_value=BEST_CONSTANT_LAMBDA,
    dtype=np.float64,
)


test_constant_gate = np.full(
    shape=len(
        test_indices_v2_cal
    ),
    fill_value=BEST_CONSTANT_LAMBDA,
    dtype=np.float64,
)


# ============================================================
# 4. Build constant predictions
# ============================================================

val_constant_innovation = (
    val_v2_innovation
    * BEST_CONSTANT_LAMBDA
)


test_constant_innovation = (
    test_v2_innovation
    * BEST_CONSTANT_LAMBDA
)


# ============================================================
# 5. State-dependent predictions
#
# Reuse Block 25 calibrated versions
# ============================================================

val_state_innovation = (
    val_calibrated_innovation
)


test_state_innovation = (
    test_calibrated_innovation
)


# ============================================================
# 6. Validation metrics
# ============================================================

val_constant_metrics = (
    evaluate_innovation_prediction(
        indices=val_indices_v2_cal,
        predicted_innovation=val_constant_innovation,
        samples=samples,
    )
)


val_state_metrics = (
    evaluate_innovation_prediction(
        indices=val_indices_v2_cal,
        predicted_innovation=val_state_innovation,
        samples=samples,
    )
)


validation_shrinkage_comparison = pd.DataFrame(
    [
        {
            "model": "Persistence",

            "mse": (
                val_persistence_cal_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                val_persistence_cal_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                val_persistence_cal_metrics[
                    "mae"
                ]
            ),
        },

        {
            "model": "Raw V2",

            "mse": (
                val_raw_v2_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                val_raw_v2_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                val_raw_v2_metrics[
                    "mae"
                ]
            ),
        },

        {
            "model": "Constant-shrinkage V2",

            "mse": (
                val_constant_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                val_constant_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                val_constant_metrics[
                    "mae"
                ]
            ),
        },

        {
            "model": "State-dependent V2",

            "mse": (
                val_state_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                val_state_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                val_state_metrics[
                    "mae"
                ]
            ),
        },
    ]
)


validation_shrinkage_comparison[
    "mse_improvement_vs_persistence"
] = (
    1.0
    - (
        validation_shrinkage_comparison[
            "mse"
        ]
        / val_persistence_cal_metrics[
            "mse"
        ]
    )
)


validation_shrinkage_comparison[
    "mae_improvement_vs_persistence"
] = (
    1.0
    - (
        validation_shrinkage_comparison[
            "mae"
        ]
        / val_persistence_cal_metrics[
            "mae"
        ]
    )
)


print(
    "\n================ VALIDATION SHRINKAGE COMPARISON ================"
)

display(
    validation_shrinkage_comparison
)


# ============================================================
# 7. Test metrics
# ============================================================

test_constant_metrics = (
    evaluate_innovation_prediction(
        indices=test_indices_v2_cal,
        predicted_innovation=test_constant_innovation,
        samples=samples,
    )
)


test_state_metrics = (
    evaluate_innovation_prediction(
        indices=test_indices_v2_cal,
        predicted_innovation=test_state_innovation,
        samples=samples,
    )
)


test_shrinkage_comparison = pd.DataFrame(
    [
        {
            "model": "Persistence",

            "mse": (
                test_persistence_cal_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                test_persistence_cal_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                test_persistence_cal_metrics[
                    "mae"
                ]
            ),
        },

        {
            "model": "Raw V2",

            "mse": (
                test_raw_v2_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                test_raw_v2_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                test_raw_v2_metrics[
                    "mae"
                ]
            ),
        },

        {
            "model": "Constant-shrinkage V2",

            "mse": (
                test_constant_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                test_constant_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                test_constant_metrics[
                    "mae"
                ]
            ),
        },

        {
            "model": "State-dependent V2",

            "mse": (
                test_state_metrics[
                    "mse"
                ]
            ),

            "rmse": (
                test_state_metrics[
                    "rmse"
                ]
            ),

            "mae": (
                test_state_metrics[
                    "mae"
                ]
            ),
        },
    ]
)


test_shrinkage_comparison[
    "mse_improvement_vs_persistence"
] = (
    1.0
    - (
        test_shrinkage_comparison[
            "mse"
        ]
        / test_persistence_cal_metrics[
            "mse"
        ]
    )
)


test_shrinkage_comparison[
    "mae_improvement_vs_persistence"
] = (
    1.0
    - (
        test_shrinkage_comparison[
            "mae"
        ]
        / test_persistence_cal_metrics[
            "mae"
        ]
    )
)


print(
    "\n================ FINAL TEST SHRINKAGE COMPARISON ================"
)

display(
    test_shrinkage_comparison
)


# ============================================================
# 8. Sample-level metrics
# ============================================================

test_shrinkage_sample_df = pd.DataFrame(
    {
        "sample_index": (
            test_indices_v2_cal
        ),

        "trade_date": (
            samples[
                "date"
            ][
                test_indices_v2_cal
            ]
        ),

        "clock_time": (
            samples[
                "clock_time"
            ][
                test_indices_v2_cal
            ]
        ),

        "failure_score": (
            test_failure_score
        ),

        "state_gate": (
            test_calibrated_gate
        ),

        "persistence_mse": (
            test_persistence_cal_metrics[
                "sample_mse"
            ]
        ),

        "raw_v2_mse": (
            test_raw_v2_metrics[
                "sample_mse"
            ]
        ),

        "constant_v2_mse": (
            test_constant_metrics[
                "sample_mse"
            ]
        ),

        "state_v2_mse": (
            test_state_metrics[
                "sample_mse"
            ]
        ),
    }
)


# ============================================================
# 9. Sample win rates
# ============================================================

for label, column in [

    (
        "Raw V2",
        "raw_v2_mse",
    ),

    (
        "Constant-shrinkage V2",
        "constant_v2_mse",
    ),

    (
        "State-dependent V2",
        "state_v2_mse",
    ),

]:

    win_rate = float(
        (
            test_shrinkage_sample_df[
                column
            ]
            < test_shrinkage_sample_df[
                "persistence_mse"
            ]
        ).mean()
    )


    print(
        f"{label} sample win rate:",
        win_rate
    )


# ============================================================
# 10. Day-level metrics
# ============================================================

daily_shrinkage = (
    test_shrinkage_sample_df
    .groupby(
        "trade_date",
        observed=True,
    )
    .agg(
        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        raw_v2_mse=(
            "raw_v2_mse",
            "mean",
        ),

        constant_v2_mse=(
            "constant_v2_mse",
            "mean",
        ),

        state_v2_mse=(
            "state_v2_mse",
            "mean",
        ),

        samples=(
            "sample_index",
            "size",
        ),
    )
    .reset_index()
)


daily_shrinkage[
    "raw_difference"
] = (
    daily_shrinkage[
        "raw_v2_mse"
    ]
    - daily_shrinkage[
        "persistence_mse"
    ]
)


daily_shrinkage[
    "constant_difference"
] = (
    daily_shrinkage[
        "constant_v2_mse"
    ]
    - daily_shrinkage[
        "persistence_mse"
    ]
)


daily_shrinkage[
    "state_difference"
] = (
    daily_shrinkage[
        "state_v2_mse"
    ]
    - daily_shrinkage[
        "persistence_mse"
    ]
)


for label, column in [

    (
        "Raw V2",
        "raw_difference",
    ),

    (
        "Constant-shrinkage V2",
        "constant_difference",
    ),

    (
        "State-dependent V2",
        "state_difference",
    ),

]:

    day_win_rate = float(
        (
            daily_shrinkage[
                column
            ]
            < 0
        ).mean()
    )


    print(
        f"{label} day win rate:",
        day_win_rate
    )


# ============================================================
# 11. Bootstrap
# ============================================================

bootstrap_rows_26 = []


for label, column in [

    (
        "Raw V2",
        "raw_difference",
    ),

    (
        "Constant-shrinkage V2",
        "constant_difference",
    ),

    (
        "State-dependent V2",
        "state_difference",
    ),

]:

    result = (
        bootstrap_daily_difference(
            daily_shrinkage[
                column
            ].to_numpy()
        )
    )


    bootstrap_rows_26.append(
        {
            "model": label,
            **result,
        }
    )


bootstrap_table_26 = (
    pd.DataFrame(
        bootstrap_rows_26
    )
)


print(
    "\n================ DAY-LEVEL BOOTSTRAP ================"
)

display(
    bootstrap_table_26
)


# ============================================================
# 12. Direct constant-vs-state comparison
#
# Negative difference means state-dependent is better.
# ============================================================

daily_shrinkage[
    "state_minus_constant"
] = (
    daily_shrinkage[
        "state_v2_mse"
    ]
    - daily_shrinkage[
        "constant_v2_mse"
    ]
)


state_vs_constant_bootstrap = (
    bootstrap_daily_difference(
        daily_shrinkage[
            "state_minus_constant"
        ].to_numpy()
    )
)


print(
    "\n================ STATE VS CONSTANT DIRECT BOOTSTRAP ================"
)

print(
    state_vs_constant_bootstrap
)


state_beats_constant_day_fraction = float(
    (
        daily_shrinkage[
            "state_minus_constant"
        ]
        < 0
    ).mean()
)


print(
    "\nState-dependent beats constant on fraction of days:"
)

print(
    state_beats_constant_day_fraction
)


# ============================================================
# 13. Failure-score buckets
# ============================================================

test_shrinkage_sample_df[
    "failure_bucket"
] = pd.qcut(
    test_shrinkage_sample_df[
        "failure_score"
    ].rank(
        method="first"
    ),

    q=5,

    labels=[
        "P_Q1",
        "P_Q2",
        "P_Q3",
        "P_Q4",
        "P_Q5",
    ],
)


shrinkage_bucket_table = (
    test_shrinkage_sample_df
    .groupby(
        "failure_bucket",
        observed=True,
    )
    .agg(
        samples=(
            "sample_index",
            "size",
        ),

        mean_failure_score=(
            "failure_score",
            "mean",
        ),

        mean_state_gate=(
            "state_gate",
            "mean",
        ),

        persistence_mse=(
            "persistence_mse",
            "mean",
        ),

        raw_v2_mse=(
            "raw_v2_mse",
            "mean",
        ),

        constant_v2_mse=(
            "constant_v2_mse",
            "mean",
        ),

        state_v2_mse=(
            "state_v2_mse",
            "mean",
        ),
    )
    .reset_index()
)


for prefix in [
    "raw_v2",
    "constant_v2",
    "state_v2",
]:

    shrinkage_bucket_table[
        f"{prefix}_improvement"
    ] = (
        1.0
        - (
            shrinkage_bucket_table[
                f"{prefix}_mse"
            ]
            / shrinkage_bucket_table[
                "persistence_mse"
            ]
        )
    )


print(
    "\n================ TEST BY FAILURE-SCORE BUCKET ================"
)

display(
    shrinkage_bucket_table
)


# ============================================================
# 14. Plot validation MSE vs constant lambda
# ============================================================

constant_plot_table = (
    pd.DataFrame(
        constant_rows
    )
    .sort_values(
        "lambda"
    )
)


plt.figure(
    figsize=(8, 5)
)


plt.plot(
    constant_plot_table[
        "lambda"
    ],

    constant_plot_table[
        "mse"
    ],
)


plt.axvline(
    BEST_CONSTANT_LAMBDA,
    linestyle="--",
    label=(
        f"best lambda="
        f"{BEST_CONSTANT_LAMBDA:.3f}"
    ),
)


plt.axhline(
    val_persistence_mse_cal,
    linestyle=":",
    label="persistence",
)


plt.xlabel(
    "Constant correction strength lambda"
)

plt.ylabel(
    "Validation MSE"
)

plt.title(
    "Validation constant-shrinkage search"
)

plt.legend()

plt.show()


# ============================================================
# 15. Decision diagnostics
# ============================================================

constant_test_mse = (
    test_constant_metrics[
        "mse"
    ]
)


state_test_mse = (
    test_state_metrics[
        "mse"
    ]
)


relative_state_gain_over_constant = (
    1.0
    - (
        state_test_mse
        / constant_test_mse
    )
)


print(
    "\n================ DECISION DIAGNOSTICS ================"
)


print(
    "Best constant lambda:",
    BEST_CONSTANT_LAMBDA
)


print(
    "State-dependent test MSE gain "
    "over constant:"
)

print(
    relative_state_gain_over_constant
)


print(
    "State-dependent beats constant "
    "day fraction:"
)

print(
    state_beats_constant_day_fraction
)


print(
    "State-vs-constant bootstrap:"
)

print(
    state_vs_constant_bootstrap
)


# ============================================================
# 16. Store both candidates for next block
# ============================================================

final_constant_lambda = (
    BEST_CONSTANT_LAMBDA
)


final_constant_validation_mean_innovation = (
    val_constant_innovation
)


final_constant_test_mean_innovation = (
    test_constant_innovation
)


final_state_validation_mean_innovation = (
    val_state_innovation
)


final_state_test_mean_innovation = (
    test_state_innovation
)


print(
    "\nBLOCK 26 finished successfully."
)

## Appendix: data-quality diagnostics

One-off destructive repair cells from the research notebook were removed from
this public version.  Known unusable dates are excluded non-destructively via
`EXCLUDED_DATES`.  This keeps `Run All` reproducible and prevents the notebook
from deleting local files.
